<a href="https://colab.research.google.com/github/amzad-786githumb/SPP_GAN_Research/blob/main/10_SPP_GAN_Differential_Privacy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==================================================================================================
# 1. HEADER & SCOPE
# ==================================================================================================

print("\n" + "=" * 100)
print("1. HEADER & SCOPE")
print("=" * 100)

from pathlib import Path
from google.colab import drive

# --------------------------------------------------------------------------------------------------
# Notebook identity
# --------------------------------------------------------------------------------------------------

NOTEBOOK_ID = "10"
NOTEBOOK_NAME = "SPP-GAN Differential Privacy"
FRAMEWORK_NAME = "SPP-GAN"

# --------------------------------------------------------------------------------------------------
# Canonical Google Drive
# --------------------------------------------------------------------------------------------------

DRIVE_ROOT = Path(
    "/content/drive"
)

MYDRIVE = (
    DRIVE_ROOT /
    "MyDrive"
)

if not MYDRIVE.exists():

    print(
        "Google Drive not mounted. Mounting..."
    )

    drive.mount(
        "/content/drive"
    )

if not MYDRIVE.exists():

    raise RuntimeError(
        "Google Drive mount failed.\n"
        f"Expected MyDrive at:\n{MYDRIVE}"
    )

# --------------------------------------------------------------------------------------------------
# Canonical project root
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = (
    MYDRIVE /
    "SPP_GAN_Research"
)

if not PROJECT_ROOT.exists():

    raise FileNotFoundError(
        "Canonical SPP-GAN project root not found:\n"
        f"{PROJECT_ROOT}\n\n"
        "Verify that the project exists in Google Drive."
    )

# --------------------------------------------------------------------------------------------------
# Notebook scope
# --------------------------------------------------------------------------------------------------

NOTEBOOK_SCOPE = {

    "purpose": (
        "Define and validate the differential privacy mechanism "
        "for the SPP-GAN discriminator."
    ),

    "protected_component":
        "SPP-GAN discriminator",

    "privacy_mechanism":
        "DP-SGD",

    "gradient_mechanism":
        "per-example discriminator gradients",

    "clipping_mechanism":
        "flat L2 clipping",

    "noise_mechanism":
        "Gaussian",

    "sampling_mechanism":
        "Poisson",

    "accountant":
        "RDP",

    "training_performed":
        False,

    "accounting_performed":
        False,

    "synthetic_generation_performed":
        False,

    "end_to_end_privacy_claim":
        False,
}

# --------------------------------------------------------------------------------------------------
# Privacy boundary
# --------------------------------------------------------------------------------------------------

PRIVACY_BOUNDARY = {

    "discriminator_update_private":
        True,

    "generator_update_private_in_notebook_10":
        False,

    "statistical_guidance_private_in_notebook_10":
        False,

    "preprocessing_private_in_notebook_10":
        False,

    "conditioning_private_in_notebook_10":
        False,

    "formal_accounting_notebook":
        "11",

    "training_notebook":
        "12",

    "generation_notebook":
        "13",
}

# --------------------------------------------------------------------------------------------------
# Final scope display
# --------------------------------------------------------------------------------------------------

print(
    f"Notebook ID                    : {NOTEBOOK_ID}"
)

print(
    f"Notebook                       : {NOTEBOOK_NAME}"
)

print(
    f"Framework                      : {FRAMEWORK_NAME}"
)

print(
    f"MyDrive                        : {MYDRIVE}"
)

print(
    f"Project root                   : {PROJECT_ROOT}"
)

print(
    f"Privacy mechanism              : "
    f"{NOTEBOOK_SCOPE['privacy_mechanism']}"
)

print(
    f"Protected component            : "
    f"{NOTEBOOK_SCOPE['protected_component']}"
)

print(
    f"Gradient mechanism             : "
    f"{NOTEBOOK_SCOPE['gradient_mechanism']}"
)

print(
    f"Clipping mechanism             : "
    f"{NOTEBOOK_SCOPE['clipping_mechanism']}"
)

print(
    f"Noise mechanism                : "
    f"{NOTEBOOK_SCOPE['noise_mechanism']}"
)

print(
    f"Sampling mechanism             : "
    f"{NOTEBOOK_SCOPE['sampling_mechanism']}"
)

print(
    f"Privacy accountant             : "
    f"{NOTEBOOK_SCOPE['accountant']}"
)

print()
print(
    "Privacy boundary:"
)

print(
    "  ✓ Discriminator update       : PRIVATE"
)

print(
    "  ✓ Statistical guidance       : "
    "NOT PRIVATIZED IN NB10"
)

print(
    "  ✓ Preprocessing              : "
    "NOT PRIVATIZED IN NB10"
)

print(
    "  ✓ Generator update           : "
    "NOT PRIVATIZED IN NB10"
)

print(
    "  ✓ Formal accounting          : "
    "DEFERRED TO NB11"
)

print(
    "  ✓ SPP-GAN training           : "
    "DEFERRED TO NB12"
)

print(
    "  ✓ Synthetic generation       : "
    "DEFERRED TO NB13"
)

print()
print(
    "✓ Google Drive verified."
)

print(
    "✓ Canonical project root verified."
)

print(
    "✓ Notebook 10 scope established."
)

print(
    "SECTION 1 STATUS: PASS"
)


1. HEADER & SCOPE
Google Drive not mounted. Mounting...
Mounted at /content/drive
Notebook ID                    : 10
Notebook                       : SPP-GAN Differential Privacy
Framework                      : SPP-GAN
MyDrive                        : /content/drive/MyDrive
Project root                   : /content/drive/MyDrive/SPP_GAN_Research
Privacy mechanism              : DP-SGD
Protected component            : SPP-GAN discriminator
Gradient mechanism             : per-example discriminator gradients
Clipping mechanism             : flat L2 clipping
Noise mechanism                : Gaussian
Sampling mechanism             : Poisson
Privacy accountant             : RDP

Privacy boundary:
  ✓ Discriminator update       : PRIVATE
  ✓ Statistical guidance       : NOT PRIVATIZED IN NB10
  ✓ Preprocessing              : NOT PRIVATIZED IN NB10
  ✓ Generator update           : NOT PRIVATIZED IN NB10
  ✓ Formal accounting          : DEFERRED TO NB11
  ✓ SPP-GAN training           : DEFE

In [2]:
# ==================================================================================================
# 2. LOAD CONFIGURATION
# ==================================================================================================

print("\n" + "=" * 100)
print("2. LOAD CONFIGURATION")
print("=" * 100)

import json
import hashlib
import math
import sys
import subprocess
import importlib.util
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# --------------------------------------------------------------------------------------------------
# Google Drive
# --------------------------------------------------------------------------------------------------

from google.colab import drive

MYDRIVE = Path(
    "/content/drive/MyDrive"
)

if not MYDRIVE.exists():
    drive.mount("/content/drive")

if not MYDRIVE.exists():
    raise RuntimeError(
        "Google Drive is not available."
    )

print(f"✓ MyDrive        : {MYDRIVE}")

# --------------------------------------------------------------------------------------------------
# Canonical roots
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = (
    MYDRIVE /
    "SPP_GAN_Research"
)

NB08_ROOT = (
    PROJECT_ROOT /
    "results" /
    "notebooks" /
    "notebook_08"
)

NB09_ROOT = (
    PROJECT_ROOT /
    "results" /
    "notebooks" /
    "notebook_09"
)

NB10_ROOT = (
    PROJECT_ROOT /
    "results" /
    "notebooks" /
    "notebook_10"
)

for name, path in {
    "Project root": PROJECT_ROOT,
    "Notebook 08": NB08_ROOT,
    "Notebook 09": NB09_ROOT,
}.items():

    if not path.exists():
        raise FileNotFoundError(
            f"{name} not found:\n{path}"
        )

# --------------------------------------------------------------------------------------------------
# Notebook 10 directories
# --------------------------------------------------------------------------------------------------

DIRS = {
    "root": NB10_ROOT,
    "models": NB10_ROOT / "models",
    "configuration": NB10_ROOT / "configuration",
    "metadata": NB10_ROOT / "metadata",
    "validation": NB10_ROOT / "validation",
    "manifests": NB10_ROOT / "manifests",
}

for directory in DIRS.values():
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

# --------------------------------------------------------------------------------------------------
# Dataset registry
# --------------------------------------------------------------------------------------------------

DATASET_IDS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

if len(DATASET_IDS) != 3:
    raise RuntimeError(
        "Expected exactly three datasets."
    )

# --------------------------------------------------------------------------------------------------
# Canonical Notebook 02 training sizes
# --------------------------------------------------------------------------------------------------

TRAIN_ROWS = {
    "adult_income": 34189,
    "bank_marketing": 31647,
    "diabetes_130us": 71236,
}

# --------------------------------------------------------------------------------------------------
# Reproducibility
# --------------------------------------------------------------------------------------------------

MASTER_SEED = 2025

np.random.seed(
    MASTER_SEED
)

torch.manual_seed(
    MASTER_SEED
)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        MASTER_SEED
    )

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(f"✓ Project root   : {PROJECT_ROOT}")
print(f"✓ Notebook 08    : {NB08_ROOT}")
print(f"✓ Notebook 09    : {NB09_ROOT}")
print(f"✓ Notebook 10    : {NB10_ROOT}")
print(f"✓ Master seed    : {MASTER_SEED}")
print(f"✓ Device         : {DEVICE}")

# --------------------------------------------------------------------------------------------------
# Notebook 09 configuration
# --------------------------------------------------------------------------------------------------

NB09_CONFIGURATION_PATH = (
    NB09_ROOT /
    "configuration" /
    "sppgan_statistical_guidance_configuration.json"
)

if not NB09_CONFIGURATION_PATH.exists():
    raise FileNotFoundError(
        f"Notebook 09 configuration not found:\n"
        f"{NB09_CONFIGURATION_PATH}"
    )

with open(
    NB09_CONFIGURATION_PATH,
    "r",
    encoding="utf-8",
) as f:
    NB09_GUIDANCE_CONFIGURATION = json.load(f)

REQUIRED_NB09_KEYS = {
    "configuration_version",
    "objective",
    "weights",
    "distribution_guidance",
    "moment_guidance",
    "categorical_guidance",
    "dependency_guidance",
    "privacy",
}

missing_nb09_keys = (
    REQUIRED_NB09_KEYS
    -
    set(
        NB09_GUIDANCE_CONFIGURATION.keys()
    )
)

if missing_nb09_keys:
    raise RuntimeError(
        "Notebook 09 configuration missing keys:\n"
        f"{sorted(missing_nb09_keys)}"
    )

LAMBDA_STAT = float(
    NB09_GUIDANCE_CONFIGURATION[
        "objective"
    ][
        "lambda_stat"
    ]
)

if LAMBDA_STAT < 0:
    raise ValueError(
        "lambda_stat must be non-negative."
    )

print()
print(
    f"✓ Notebook 09 configuration loaded:\n"
    f"  {NB09_CONFIGURATION_PATH}"
)

print(
    f"✓ λ_stat = {LAMBDA_STAT:.6f}"
)

print("SECTION 2 STATUS: PASS")


2. LOAD CONFIGURATION
✓ MyDrive        : /content/drive/MyDrive
✓ Project root   : /content/drive/MyDrive/SPP_GAN_Research
✓ Notebook 08    : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08
✓ Notebook 09    : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_09
✓ Notebook 10    : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10
✓ Master seed    : 2025
✓ Device         : cpu

✓ Notebook 09 configuration loaded:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_09/configuration/sppgan_statistical_guidance_configuration.json
✓ λ_stat = 1.000000
SECTION 2 STATUS: PASS


In [3]:
# ==================================================================================================
# 3. LOAD SPP-GAN ARCHITECTURE
# ==================================================================================================

print("\n" + "=" * 100)
print("3. LOAD SPP-GAN ARCHITECTURE")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# Locate architecture summary
# --------------------------------------------------------------------------------------------------

ARCHITECTURE_CANDIDATES = sorted(
    set(
        list(
            NB08_ROOT.rglob(
                "*architecture*summary*.csv"
            )
        )
        +
        list(
            NB08_ROOT.rglob(
                "*architecture_summary*.csv"
            )
        )
    ),
    key=lambda p: len(str(p))
)

if not ARCHITECTURE_CANDIDATES:
    raise FileNotFoundError(
        "Notebook 08 architecture summary not found."
    )

ARCHITECTURE_SUMMARY_PATH = (
    ARCHITECTURE_CANDIDATES[0]
)

ARCHITECTURE_SUMMARY_DF = pd.read_csv(
    ARCHITECTURE_SUMMARY_PATH
)

REQUIRED_ARCH_COLUMNS = {
    "dataset",
    "generative_dimension",
    "transformed_dimension",
    "numerical_features",
    "categorical_features",
}

missing_columns = (
    REQUIRED_ARCH_COLUMNS
    -
    set(
        ARCHITECTURE_SUMMARY_DF.columns
    )
)

if missing_columns:
    raise RuntimeError(
        "Architecture summary missing columns:\n"
        f"{sorted(missing_columns)}"
    )

ARCHITECTURE_SUMMARY_DF["dataset"] = (
    ARCHITECTURE_SUMMARY_DF[
        "dataset"
    ]
    .astype(str)
)

ARCHITECTURE_SUMMARY_DF = (
    ARCHITECTURE_SUMMARY_DF[
        ARCHITECTURE_SUMMARY_DF[
            "dataset"
        ].isin(
            DATASET_IDS
        )
    ]
    .copy()
)

if len(ARCHITECTURE_SUMMARY_DF) != 3:
    raise RuntimeError(
        "Architecture summary does not contain all three datasets."
    )

ARCHITECTURE_SUMMARY_DF = (
    ARCHITECTURE_SUMMARY_DF
    .set_index("dataset")
    .loc[DATASET_IDS]
    .reset_index()
)

for column in [
    "generative_dimension",
    "transformed_dimension",
    "numerical_features",
    "categorical_features",
]:

    ARCHITECTURE_SUMMARY_DF[column] = pd.to_numeric(
        ARCHITECTURE_SUMMARY_DF[column],
        errors="coerce",
    )

    if ARCHITECTURE_SUMMARY_DF[column].isna().any():
        raise RuntimeError(
            f"Invalid values in architecture column: {column}"
        )

    ARCHITECTURE_SUMMARY_DF[column] = (
        ARCHITECTURE_SUMMARY_DF[column]
        .astype(int)
    )

# --------------------------------------------------------------------------------------------------
# Validate architecture dimensions
# --------------------------------------------------------------------------------------------------

for row in ARCHITECTURE_SUMMARY_DF.itertuples():

    expected_generative_dimension = (
        row.numerical_features
        +
        row.categorical_features
        +
        1
    )

    if row.generative_dimension != (
        expected_generative_dimension
    ):
        raise RuntimeError(
            f"Generative dimension mismatch: {row.dataset}"
        )

CRITIC_INPUT_DIMENSIONS = {
    row.dataset: int(
        row.transformed_dimension
    )
    for row in ARCHITECTURE_SUMMARY_DF.itertuples()
}

LATENT_DIMENSION = 128

print(
    f"✓ Architecture summary:\n"
    f"  {ARCHITECTURE_SUMMARY_PATH}"
)

display(
    ARCHITECTURE_SUMMARY_DF[
        [
            "dataset",
            "generative_dimension",
            "transformed_dimension",
            "numerical_features",
            "categorical_features",
        ]
    ]
)

print(
    f"✓ Latent dimension : {LATENT_DIMENSION}"
)

print(
    "✓ Critic dimensions:"
)

for dataset_id in DATASET_IDS:
    print(
        f"  {dataset_id:18s}: "
        f"{CRITIC_INPUT_DIMENSIONS[dataset_id]}"
    )

print("SECTION 3 STATUS: PASS")


3. LOAD SPP-GAN ARCHITECTURE
✓ Architecture summary:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/architecture/sppgan_architecture_summary.csv


,dataset,generative_dimension,transformed_dimension,numerical_features,categorical_features
0,adult_income,15,105,6,8
1,bank_marketing,17,51,7,9
2,diabetes_130us,48,2329,11,36


✓ Latent dimension : 128
✓ Critic dimensions:
  adult_income      : 105
  bank_marketing    : 51
  diabetes_130us    : 2329
SECTION 3 STATUS: PASS


In [4]:
# ==================================================================================================
# 4. LOAD PRIVACY CONFIGURATION
# ==================================================================================================

print("\n" + "=" * 100)
print("4. LOAD PRIVACY CONFIGURATION")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# Opacus
# --------------------------------------------------------------------------------------------------

if importlib.util.find_spec(
    "opacus"
) is None:

    print(
        "Opacus not installed. Installing..."
    )

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "opacus",
        ]
    )

import opacus

from opacus import PrivacyEngine
from opacus.grad_sample import GradSampleModule
from opacus.validators import ModuleValidator
from opacus.accountants.utils import get_noise_multiplier

print(
    f"✓ Opacus version : {opacus.__version__}"
)

# --------------------------------------------------------------------------------------------------
# Methodology-aligned privacy parameters
# --------------------------------------------------------------------------------------------------

TARGET_EPSILON = 5.0

MAX_GRAD_NORM = 1.0

DP_BATCH_SIZE = 128

DP_EPOCHS = 300

ACCOUNTANT = "rdp"

SAMPLING_MECHANISM = "poisson"

CLIPPING_MECHANISM = "flat"

LOSS_REDUCTION = "mean"

GRAD_SAMPLE_MODE = "hooks"

# The generator is not independently privatized in this notebook.
GENERATOR_PRIVATE = False

# Statistical guidance remains outside the DP mechanism here.
STATISTICAL_GUIDANCE_PRIVATE = False

# Preprocessing remains outside this DP mechanism here.
PREPROCESSING_PRIVATE = False

# --------------------------------------------------------------------------------------------------
# Dataset-specific delta
# --------------------------------------------------------------------------------------------------

DELTA_BY_DATASET = {
    dataset_id: min(
        1e-5,
        1.0 / TRAIN_ROWS[dataset_id]
    )
    for dataset_id in DATASET_IDS
}

PRIVACY_CONFIGURATION = {
    "mechanism": "DP-SGD",
    "protected_component": "SPP-GAN discriminator",
    "target_epsilon": TARGET_EPSILON,
    "delta_rule": "min(1e-5, 1/N_train)",
    "max_grad_norm": MAX_GRAD_NORM,
    "batch_size": DP_BATCH_SIZE,
    "epochs": DP_EPOCHS,
    "accountant": ACCOUNTANT,
    "sampling": SAMPLING_MECHANISM,
    "clipping": CLIPPING_MECHANISM,
    "loss_reduction": LOSS_REDUCTION,
    "grad_sample_mode": GRAD_SAMPLE_MODE,
    "generator_private": GENERATOR_PRIVATE,
    "statistical_guidance_private": STATISTICAL_GUIDANCE_PRIVATE,
    "preprocessing_private": PREPROCESSING_PRIVATE,
    "end_to_end_privacy_claim": False,
}

print()
print("Privacy configuration:")

for key, value in PRIVACY_CONFIGURATION.items():
    print(
        f"  {key:40s}: {value}"
    )

print()
print("Dataset-specific δ:")

for dataset_id in DATASET_IDS:
    print(
        f"  {dataset_id:18s}: "
        f"{DELTA_BY_DATASET[dataset_id]:.12g}"
    )

print("SECTION 4 STATUS: PASS")


4. LOAD PRIVACY CONFIGURATION
Opacus not installed. Installing...
✓ Opacus version : 1.6.0

Privacy configuration:
  mechanism                               : DP-SGD
  protected_component                     : SPP-GAN discriminator
  target_epsilon                          : 5.0
  delta_rule                              : min(1e-5, 1/N_train)
  max_grad_norm                           : 1.0
  batch_size                              : 128
  epochs                                  : 300
  accountant                              : rdp
  sampling                                : poisson
  clipping                                : flat
  loss_reduction                          : mean
  grad_sample_mode                        : hooks
  generator_private                       : False
  statistical_guidance_private            : False
  preprocessing_private                   : False
  end_to_end_privacy_claim                : False

Dataset-specific δ:
  adult_income      : 1e-05
  bank_market

In [5]:
# ==================================================================================================
# 5. VALIDATE PRIVACY PARAMETERS
# ==================================================================================================

print("\n" + "=" * 100)
print("5. VALIDATE PRIVACY PARAMETERS")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Core privacy-parameter validation
# --------------------------------------------------------------------------------------------------

if not np.isfinite(TARGET_EPSILON) or TARGET_EPSILON <= 0:
    raise ValueError(
        "Target epsilon must be finite and strictly positive."
    )

if not np.isfinite(MAX_GRAD_NORM) or MAX_GRAD_NORM <= 0:
    raise ValueError(
        "Maximum gradient norm must be finite and strictly positive."
    )

if not isinstance(DP_BATCH_SIZE, int) or DP_BATCH_SIZE <= 0:
    raise ValueError(
        "DP batch size must be a positive integer."
    )

if not isinstance(DP_EPOCHS, int) or DP_EPOCHS <= 0:
    raise ValueError(
        "DP epochs must be a positive integer."
    )

if ACCOUNTANT != "rdp":
    raise ValueError(
        "This study requires the RDP accountant."
    )

if SAMPLING_MECHANISM != "poisson":
    raise ValueError(
        "This study requires Poisson sampling."
    )

if CLIPPING_MECHANISM != "flat":
    raise ValueError(
        "This study requires flat L2 clipping."
    )

if LOSS_REDUCTION != "mean":
    raise ValueError(
        "This study requires mean loss reduction."
    )

# --------------------------------------------------------------------------------------------------
# 2. Explicit RDP alpha-order grid
#
# The grid is explicitly recorded for reproducibility.
# The Opacus accountant remains authoritative for epsilon conversion.
# --------------------------------------------------------------------------------------------------

RDP_ALPHAS = (
    [round(1.01 + 0.01 * i, 2) for i in range(0, 900)]
    + [float(alpha) for alpha in range(10, 1001)]
)

RDP_ALPHAS = sorted(
    set(
        float(alpha)
        for alpha in RDP_ALPHAS
        if np.isfinite(alpha)
        and alpha > 1.0
    )
)

RDP_ALPHA_ARRAY = np.asarray(
    RDP_ALPHAS,
    dtype=float
)

if len(RDP_ALPHAS) < 100:
    raise RuntimeError(
        "Insufficient RDP alpha-order coverage."
    )

if RDP_ALPHA_ARRAY[0] <= 1.0:
    raise RuntimeError(
        "RDP alpha orders must be strictly greater than 1."
    )

if not np.all(
    np.diff(RDP_ALPHA_ARRAY) > 0
):
    raise RuntimeError(
        "RDP alpha orders must be strictly increasing."
    )

RDP_ALPHA_MIN = float(
    RDP_ALPHA_ARRAY.min()
)

RDP_ALPHA_MAX = float(
    RDP_ALPHA_ARRAY.max()
)

RDP_ALPHA_COUNT = int(
    len(RDP_ALPHA_ARRAY)
)

print(
    f"✓ RDP alpha orders : {RDP_ALPHA_COUNT}"
)

print(
    f"✓ RDP alpha range  : "
    f"{RDP_ALPHA_MIN:.2f} → {RDP_ALPHA_MAX:.0f}"
)

# --------------------------------------------------------------------------------------------------
# 3. Import Opacus RDP accounting utilities
# --------------------------------------------------------------------------------------------------

from opacus.accountants.analysis.rdp import (
    compute_rdp,
    get_privacy_spent,
)

# --------------------------------------------------------------------------------------------------
# 4. RDP evaluation helper
#
# Opacus' get_privacy_spent() is deliberately retained as the authoritative
# epsilon conversion mechanism.
#
# We do NOT reconstruct the epsilon conversion independently because different
# implementations can differ numerically, especially for fractional RDP orders.
# --------------------------------------------------------------------------------------------------

def evaluate_rdp_privacy(
    *,
    sample_rate,
    noise_multiplier,
    epochs,
    delta,
    alphas,
):
    """
    Evaluate RDP privacy using Opacus' RDP implementation.

    Returns
    -------
    epsilon : float
        Privacy loss converted from RDP by Opacus.

    optimal_alpha : float
        Alpha order selected by Opacus.

    rdp_values : np.ndarray
        RDP values for the supplied alpha orders.
    """

    if not (
        np.isfinite(sample_rate)
        and 0 < sample_rate < 1
    ):
        raise ValueError(
            f"Invalid sample rate: {sample_rate}"
        )

    if not (
        np.isfinite(noise_multiplier)
        and noise_multiplier > 0
    ):
        raise ValueError(
            f"Invalid noise multiplier: {noise_multiplier}"
        )

    if not (
        np.isfinite(epochs)
        and epochs > 0
    ):
        raise ValueError(
            f"Invalid epoch count: {epochs}"
        )

    if not (
        np.isfinite(delta)
        and 0 < delta < 1
    ):
        raise ValueError(
            f"Invalid delta: {delta}"
        )

    # ----------------------------------------------------------------------------------------------
    # Expected number of Poisson-sampled composition steps.
    # ----------------------------------------------------------------------------------------------

    steps = float(
        epochs / sample_rate
    )

    rdp_values = compute_rdp(
        q=float(sample_rate),
        noise_multiplier=float(noise_multiplier),
        steps=steps,
        orders=alphas,
    )

    rdp_values = np.asarray(
        rdp_values,
        dtype=float
    )

    if len(rdp_values) != len(alphas):
        raise RuntimeError(
            "RDP value/order length mismatch."
        )

    if not np.all(
        np.isfinite(rdp_values)
    ):
        raise RuntimeError(
            "RDP computation produced non-finite values."
        )

    epsilon, optimal_alpha = get_privacy_spent(
        orders=alphas,
        rdp=rdp_values,
        delta=float(delta),
    )

    epsilon = float(
        epsilon
    )

    optimal_alpha = float(
        optimal_alpha
    )

    if not (
        np.isfinite(epsilon)
        and epsilon > 0
    ):
        raise RuntimeError(
            f"Invalid RDP epsilon: {epsilon}"
        )

    if not np.isfinite(
        optimal_alpha
    ):
        raise RuntimeError(
            "Invalid optimal RDP alpha."
        )

    return (
        epsilon,
        optimal_alpha,
        rdp_values,
    )

# --------------------------------------------------------------------------------------------------
# 5. Explicit noise-multiplier calibration
#
# Calibration is performed directly against the Opacus RDP accountant.
#
# This avoids relying on Opacus' get_noise_multiplier() internal alpha grid.
# --------------------------------------------------------------------------------------------------

def calibrate_noise_multiplier_rdp(
    *,
    target_epsilon,
    target_delta,
    sample_rate,
    epochs,
    alphas,
    tolerance=1e-4,
    max_iterations=100,
):
    """
    Calibrate Gaussian noise multiplier by RDP binary search.

    Opacus is the authoritative privacy accountant.
    """

    if target_epsilon <= 0:
        raise ValueError(
            "Target epsilon must be positive."
        )

    # ----------------------------------------------------------------------------------------------
    # Initial bracket
    # ----------------------------------------------------------------------------------------------

    lower_sigma = 0.01
    upper_sigma = 10.0

    lower_epsilon, _, _ = evaluate_rdp_privacy(
        sample_rate=sample_rate,
        noise_multiplier=lower_sigma,
        epochs=epochs,
        delta=target_delta,
        alphas=alphas,
    )

    upper_epsilon, _, _ = evaluate_rdp_privacy(
        sample_rate=sample_rate,
        noise_multiplier=upper_sigma,
        epochs=epochs,
        delta=target_delta,
        alphas=alphas,
    )

    # ----------------------------------------------------------------------------------------------
    # Expand upper bracket if necessary.
    # ----------------------------------------------------------------------------------------------

    expansion_count = 0

    while (
        upper_epsilon > target_epsilon
        and expansion_count < 50
    ):

        upper_sigma *= 2.0

        upper_epsilon, _, _ = evaluate_rdp_privacy(
            sample_rate=sample_rate,
            noise_multiplier=upper_sigma,
            epochs=epochs,
            delta=target_delta,
            alphas=alphas,
        )

        expansion_count += 1

    if upper_epsilon > target_epsilon:
        raise RuntimeError(
            "Unable to bracket target epsilon."
        )

    if lower_epsilon < target_epsilon:
        raise RuntimeError(
            "Lower noise multiplier already provides epsilon "
            "below the requested target."
        )

    # ----------------------------------------------------------------------------------------------
    # Binary search
    # ----------------------------------------------------------------------------------------------

    best_sigma = None
    best_epsilon = None
    best_alpha = None
    best_rdp = None

    for _ in range(
        max_iterations
    ):

        midpoint_sigma = (
            lower_sigma
            +
            upper_sigma
        ) / 2.0

        midpoint_epsilon, midpoint_alpha, midpoint_rdp = (
            evaluate_rdp_privacy(
                sample_rate=sample_rate,
                noise_multiplier=midpoint_sigma,
                epochs=epochs,
                delta=target_delta,
                alphas=alphas,
            )
        )

        best_sigma = float(
            midpoint_sigma
        )

        best_epsilon = float(
            midpoint_epsilon
        )

        best_alpha = float(
            midpoint_alpha
        )

        best_rdp = midpoint_rdp

        epsilon_error = abs(
            midpoint_epsilon
            -
            target_epsilon
        )

        if epsilon_error <= tolerance:
            break

        if midpoint_epsilon > target_epsilon:
            lower_sigma = midpoint_sigma
        else:
            upper_sigma = midpoint_sigma

    if best_sigma is None:
        raise RuntimeError(
            "Noise-multiplier calibration failed."
        )

    return (
        best_sigma,
        best_epsilon,
        best_alpha,
        best_rdp,
    )

# --------------------------------------------------------------------------------------------------
# 6. Dataset-specific privacy calibration
# --------------------------------------------------------------------------------------------------

PRIVACY_PARAMETER_ROWS = []

for dataset_id in DATASET_IDS:

    n_train = int(
        TRAIN_ROWS[dataset_id]
    )

    if n_train <= DP_BATCH_SIZE:
        raise ValueError(
            f"{dataset_id}: training population must exceed DP batch size."
        )

    delta = float(
        DELTA_BY_DATASET[dataset_id]
    )

    if not (
        np.isfinite(delta)
        and 0 < delta < 1
    ):
        raise ValueError(
            f"{dataset_id}: invalid delta={delta}."
        )

    # ----------------------------------------------------------------------------------------------
    # Nominal Poisson sampling probability
    # ----------------------------------------------------------------------------------------------

    sample_rate = (
        float(DP_BATCH_SIZE)
        /
        float(n_train)
    )

    if not (
        0 < sample_rate < 1
    ):
        raise ValueError(
            f"{dataset_id}: invalid Poisson sample rate={sample_rate}."
        )

    # ----------------------------------------------------------------------------------------------
    # Explicit RDP calibration
    # ----------------------------------------------------------------------------------------------

    (
        noise_multiplier,
        calibration_epsilon,
        optimal_alpha,
        calibration_rdp,
    ) = calibrate_noise_multiplier_rdp(
        target_epsilon=TARGET_EPSILON,
        target_delta=delta,
        sample_rate=sample_rate,
        epochs=DP_EPOCHS,
        alphas=RDP_ALPHAS,
        tolerance=1e-4,
        max_iterations=100,
    )

    # ----------------------------------------------------------------------------------------------
    # Independent re-evaluation through the SAME Opacus accountant
    # ----------------------------------------------------------------------------------------------

    (
        verified_epsilon,
        verified_alpha,
        verified_rdp,
    ) = evaluate_rdp_privacy(
        sample_rate=sample_rate,
        noise_multiplier=noise_multiplier,
        epochs=DP_EPOCHS,
        delta=delta,
        alphas=RDP_ALPHAS,
    )

    # ----------------------------------------------------------------------------------------------
    # Calibration consistency
    # ----------------------------------------------------------------------------------------------

    if not np.isclose(
        calibration_epsilon,
        verified_epsilon,
        rtol=1e-10,
        atol=1e-10,
    ):
        raise RuntimeError(
            f"{dataset_id}: calibration and verification epsilon differ."
        )

    if not np.isclose(
        optimal_alpha,
        verified_alpha,
        rtol=0,
        atol=1e-12,
    ):
        raise RuntimeError(
            f"{dataset_id}: calibration and verification alpha differ."
        )

    epsilon_error = abs(
        verified_epsilon
        -
        TARGET_EPSILON
    )

    if epsilon_error > 1e-3:
        raise RuntimeError(
            f"{dataset_id}: calibrated epsilon "
            f"{verified_epsilon:.8f} differs from target "
            f"{TARGET_EPSILON:.8f} by "
            f"{epsilon_error:.8f}."
        )

    # ----------------------------------------------------------------------------------------------
    # Exact endpoint validation
    #
    # An RDP optimum is considered boundary-selected only when the accountant
    # explicitly returns the first or last configured alpha.
    # ----------------------------------------------------------------------------------------------

    alpha_at_lower_boundary = np.isclose(
        verified_alpha,
        RDP_ALPHA_MIN,
        rtol=0,
        atol=1e-12,
    )

    alpha_at_upper_boundary = np.isclose(
        verified_alpha,
        RDP_ALPHA_MAX,
        rtol=0,
        atol=1e-12,
    )

    alpha_at_boundary = (
        alpha_at_lower_boundary
        or
        alpha_at_upper_boundary
    )

    # Do not fail solely because an external warning is emitted.
    # The authoritative test is the actual alpha returned by the accountant.

    # ----------------------------------------------------------------------------------------------
    # Verify RDP output integrity
    # ----------------------------------------------------------------------------------------------

    if calibration_rdp is None:
        raise RuntimeError(
            f"{dataset_id}: calibration RDP values are missing."
        )

    if len(
        calibration_rdp
    ) != RDP_ALPHA_COUNT:
        raise RuntimeError(
            f"{dataset_id}: calibration RDP order coverage mismatch."
        )

    if not np.all(
        np.isfinite(
            np.asarray(
                calibration_rdp,
                dtype=float
            )
        )
    ):
        raise RuntimeError(
            f"{dataset_id}: non-finite calibration RDP values."
        )

    # ----------------------------------------------------------------------------------------------
    # Composition-step metadata
    # ----------------------------------------------------------------------------------------------

    nominal_steps_per_epoch = (
        1.0
        /
        sample_rate
    )

    nominal_total_steps = (
        float(DP_EPOCHS)
        *
        nominal_steps_per_epoch
    )

    # ----------------------------------------------------------------------------------------------
    # Record privacy calibration
    # ----------------------------------------------------------------------------------------------

    PRIVACY_PARAMETER_ROWS.append({
        "dataset": dataset_id,
        "n_train": n_train,
        "target_epsilon": float(
            TARGET_EPSILON
        ),
        "delta": delta,
        "batch_size_nominal": int(
            DP_BATCH_SIZE
        ),
        "poisson_sample_rate": float(
            sample_rate
        ),
        "epochs": int(
            DP_EPOCHS
        ),
        "nominal_steps_per_epoch": float(
            nominal_steps_per_epoch
        ),
        "nominal_total_steps": float(
            nominal_total_steps
        ),
        "max_grad_norm": float(
            MAX_GRAD_NORM
        ),
        "noise_multiplier": float(
            noise_multiplier
        ),
        "calibration_epsilon": float(
            verified_epsilon
        ),
        "epsilon_calibration_error": float(
            epsilon_error
        ),
        "rdp_alpha_min": float(
            RDP_ALPHA_MIN
        ),
        "rdp_alpha_max": float(
            RDP_ALPHA_MAX
        ),
        "rdp_alpha_count": int(
            RDP_ALPHA_COUNT
        ),
        "optimal_alpha": float(
            verified_alpha
        ),
        "optimal_alpha_at_boundary": bool(
            alpha_at_boundary
        ),
        "calibration_method": (
            "explicit_rdp_binary_search"
        ),
        "accountant": ACCOUNTANT,
        "sampling": SAMPLING_MECHANISM,
        "clipping": CLIPPING_MECHANISM,
        "loss_reduction": LOSS_REDUCTION,
        "calibration_status": "PASS",
        "achieved_epsilon_status": (
            "DEFERRED_TO_NOTEBOOK_11"
        ),
    })

# --------------------------------------------------------------------------------------------------
# 7. Assemble validation table
# --------------------------------------------------------------------------------------------------

PRIVACY_PARAMETER_DF = pd.DataFrame(
    PRIVACY_PARAMETER_ROWS
)

if PRIVACY_PARAMETER_DF.empty:
    raise RuntimeError(
        "No privacy-parameter records were generated."
    )

if len(
    PRIVACY_PARAMETER_DF
) != len(
    DATASET_IDS
):
    raise RuntimeError(
        "Privacy calibration does not contain exactly one row per dataset."
    )

if set(
    PRIVACY_PARAMETER_DF["dataset"]
) != set(
    DATASET_IDS
):
    raise RuntimeError(
        "Privacy calibration dataset coverage mismatch."
    )

required_columns = [
    "dataset",
    "n_train",
    "target_epsilon",
    "delta",
    "batch_size_nominal",
    "poisson_sample_rate",
    "epochs",
    "nominal_steps_per_epoch",
    "nominal_total_steps",
    "max_grad_norm",
    "noise_multiplier",
    "calibration_epsilon",
    "epsilon_calibration_error",
    "rdp_alpha_min",
    "rdp_alpha_max",
    "rdp_alpha_count",
    "optimal_alpha",
    "optimal_alpha_at_boundary",
    "calibration_method",
    "accountant",
    "sampling",
    "clipping",
    "loss_reduction",
    "calibration_status",
    "achieved_epsilon_status",
]

missing_columns = [
    column
    for column in required_columns
    if column not in PRIVACY_PARAMETER_DF.columns
]

if missing_columns:
    raise RuntimeError(
        f"Missing privacy-parameter columns: {missing_columns}"
    )

if not (
    PRIVACY_PARAMETER_DF[
        "calibration_status"
    ]
    .eq("PASS")
    .all()
):
    raise RuntimeError(
        "One or more privacy calibrations failed."
    )

if not (
    PRIVACY_PARAMETER_DF[
        "achieved_epsilon_status"
    ]
    .eq(
        "DEFERRED_TO_NOTEBOOK_11"
    )
    .all()
):
    raise RuntimeError(
        "Achieved-epsilon provenance is inconsistent."
    )

# --------------------------------------------------------------------------------------------------
# 8. Numerical validation
# --------------------------------------------------------------------------------------------------

numeric_columns = [
    "target_epsilon",
    "delta",
    "poisson_sample_rate",
    "epochs",
    "max_grad_norm",
    "noise_multiplier",
    "calibration_epsilon",
    "epsilon_calibration_error",
    "rdp_alpha_min",
    "rdp_alpha_max",
    "optimal_alpha",
]

for column in numeric_columns:

    values = pd.to_numeric(
        PRIVACY_PARAMETER_DF[column],
        errors="coerce",
    )

    if not np.all(
        np.isfinite(
            values.to_numpy()
        )
    ):
        raise RuntimeError(
            f"Non-finite values detected in '{column}'."
        )

# --------------------------------------------------------------------------------------------------
# 9. Final validation summary
# --------------------------------------------------------------------------------------------------

print()
print("✓ Privacy parameters validated.")

print()
print("Calibration policy:")
print(
    f"  Target ε                    : {TARGET_EPSILON:.6f}"
)
print(
    "  δ rule                      : min(1e-5, 1/N_train)"
)
print(
    f"  Max gradient norm           : {MAX_GRAD_NORM:.6f}"
)
print(
    f"  DP batch size               : {DP_BATCH_SIZE}"
)
print(
    f"  Epochs                      : {DP_EPOCHS}"
)
print(
    f"  Accountant                  : {ACCOUNTANT}"
)
print(
    f"  Sampling                    : {SAMPLING_MECHANISM}"
)
print(
    f"  RDP alpha count             : {RDP_ALPHA_COUNT}"
)
print(
    f"  RDP alpha range             : "
    f"{RDP_ALPHA_MIN:.2f} → {RDP_ALPHA_MAX:.0f}"
)
print(
    "  Calibration method          : explicit RDP binary search"
)
print(
    "  Epsilon conversion          : Opacus RDP accountant"
)

print()
print("Privacy calibration results:")

display(
    PRIVACY_PARAMETER_DF
)

print()
print("RDP validation:")

for _, row in PRIVACY_PARAMETER_DF.iterrows():

    print(
        f"  {row['dataset']:18s} : "
        f"ε_cal={row['calibration_epsilon']:.6f}, "
        f"α*={row['optimal_alpha']:.2f}, "
        f"σ={row['noise_multiplier']:.6f}, "
        f"boundary={row['optimal_alpha_at_boundary']}"
    )

print()
print("Privacy-accounting boundary:")

print(
    "  ✓ Target ε is a calibration target."
)

print(
    "  ✓ Gaussian noise multipliers are calibrated explicitly."
)

print(
    "  ✓ RDP orders are explicitly defined."
)

print(
    "  ✓ Opacus RDP accounting is the authoritative epsilon conversion."
)

print(
    "  ✓ RDP optimum is checked against the configured order endpoints."
)

print(
    "  ✓ Calibration epsilon is independently re-evaluated."
)

print(
    "  ✓ Achieved ε is NOT claimed in Notebook 10."
)

print(
    "  ✓ Final achieved ε is deferred to Notebook 11."
)

print(
    "  ✓ End-to-end privacy claim remains disabled."
)

print()
print("SECTION 5 STATUS: PASS")


5. VALIDATE PRIVACY PARAMETERS
✓ RDP alpha orders : 1890
✓ RDP alpha range  : 1.01 → 1000


/usr/local/lib/python3.13/dist-packages/opacus/accountants/analysis/rdp.py:332: UserWarning: Optimal order is the smallest alpha. Please consider expanding the range of alphas to get a tighter privacy bound.
  warnings.warn(



✓ Privacy parameters validated.

Calibration policy:
  Target ε                    : 5.000000
  δ rule                      : min(1e-5, 1/N_train)
  Max gradient norm           : 1.000000
  DP batch size               : 128
  Epochs                      : 300
  Accountant                  : rdp
  Sampling                    : poisson
  RDP alpha count             : 1890
  RDP alpha range             : 1.01 → 1000
  Calibration method          : explicit RDP binary search
  Epsilon conversion          : Opacus RDP accountant

Privacy calibration results:


,dataset,n_train,target_epsilon,delta,batch_size_nominal,poisson_sample_rate,epochs,nominal_steps_per_epoch,nominal_total_steps,max_grad_norm,...,rdp_alpha_count,optimal_alpha,optimal_alpha_at_boundary,calibration_method,accountant,sampling,clipping,loss_reduction,calibration_status,achieved_epsilon_status
0,adult_income,34189,5.0,0.00001,128,0.003744,300,267.101562,80130.46875,1.0,...,1890,5.18,False,explicit_rdp_binary_search,rdp,poisson,flat,mean,PASS,DEFERRED_TO_NOTEBOOK_11
1,bank_marketing,31647,5.0,0.00001,128,0.004045,300,247.242188,74172.65625,1.0,...,1890,5.18,False,explicit_rdp_binary_search,rdp,poisson,flat,mean,PASS,DEFERRED_TO_NOTEBOOK_11
2,diabetes_130us,71236,5.0,0.00001,128,0.001797,300,556.531250,166959.37500,1.0,...,1890,5.17,False,explicit_rdp_binary_search,rdp,poisson,flat,mean,PASS,DEFERRED_TO_NOTEBOOK_11



RDP validation:
  adult_income       : ε_cal=5.000035, α*=5.18, σ=1.216221, boundary=False
  bank_marketing     : ε_cal=5.000048, α*=5.18, σ=1.251014, boundary=False
  diabetes_130us     : ε_cal=5.000071, α*=5.17, σ=0.953403, boundary=False

Privacy-accounting boundary:
  ✓ Target ε is a calibration target.
  ✓ Gaussian noise multipliers are calibrated explicitly.
  ✓ RDP orders are explicitly defined.
  ✓ Opacus RDP accounting is the authoritative epsilon conversion.
  ✓ RDP optimum is checked against the configured order endpoints.
  ✓ Calibration epsilon is independently re-evaluated.
  ✓ Achieved ε is NOT claimed in Notebook 10.
  ✓ Final achieved ε is deferred to Notebook 11.
  ✓ End-to-end privacy claim remains disabled.

SECTION 5 STATUS: PASS


In [6]:
# ==================================================================================================
# 6. DEFINE PER-EXAMPLE DISCRIMINATOR GRADIENTS
# ==================================================================================================

print("\n" + "=" * 100)
print("6. DEFINE PER-EXAMPLE DISCRIMINATOR GRADIENTS")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Validate required Notebook 08 architecture objects
# --------------------------------------------------------------------------------------------------

required_objects = [
    "ARCHITECTURE_SUMMARY_DF",
    "CRITIC_INPUT_DIMENSIONS",
    "LATENT_DIMENSION",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Required Notebook 08 architecture objects are missing: "
        f"{missing_objects}"
    )

# --------------------------------------------------------------------------------------------------
# 2. Validate Notebook 08 architecture schema
# --------------------------------------------------------------------------------------------------

required_architecture_columns = [
    "dataset",
    "generative_dimension",
    "transformed_dimension",
    "numerical_features",
    "categorical_features",
]

missing_architecture_columns = [
    column
    for column in required_architecture_columns
    if column not in ARCHITECTURE_SUMMARY_DF.columns
]

if missing_architecture_columns:
    raise RuntimeError(
        "Notebook 08 architecture summary is missing required columns: "
        f"{missing_architecture_columns}"
    )

architecture_df = (
    ARCHITECTURE_SUMMARY_DF[
        required_architecture_columns
    ]
    .copy()
)

architecture_df = (
    architecture_df[
        architecture_df["dataset"].isin(
            DATASET_IDS
        )
    ]
    .copy()
)

if len(
    architecture_df
) != len(
    DATASET_IDS
):
    raise RuntimeError(
        "Notebook 08 architecture summary does not contain "
        "exactly one canonical row per dataset."
    )

architecture_df = (
    architecture_df
    .set_index("dataset")
    .loc[DATASET_IDS]
    .reset_index()
)

# --------------------------------------------------------------------------------------------------
# 3. Notebook 08-compatible SPP-GAN critic
#
# Architecture contract:
#
#     Input
#       ↓
#     Linear(input_dim, 256)
#       ↓
#     LeakyReLU(0.2)
#       ↓
#     Linear(256, 256)
#       ↓
#     LeakyReLU(0.2)
#       ↓
#     Linear(256, 1)
#
# No sigmoid, BCE output, dropout, or batch normalization is introduced.
# --------------------------------------------------------------------------------------------------

class SPPGANCritic(
    nn.Module
):
    """
    Notebook 08-compatible SPP-GAN critic.

    Input
    -----
    Transformed tabular representation.

    Output
    ------
    One scalar critic score per record.
    """

    def __init__(
        self,
        input_dim,
    ):
        super().__init__()

        self.input_dim = int(
            input_dim
        )

        if self.input_dim <= 0:
            raise ValueError(
                "Critic input dimension must be positive."
            )

        self.network = nn.Sequential(
            nn.Linear(
                self.input_dim,
                256,
            ),
            nn.LeakyReLU(
                negative_slope=0.2
            ),
            nn.Linear(
                256,
                256,
            ),
            nn.LeakyReLU(
                negative_slope=0.2
            ),
            nn.Linear(
                256,
                1,
            ),
        )

    def forward(
        self,
        x,
    ):
        if x.ndim != 2:
            raise ValueError(
                "Critic input must be a two-dimensional tensor."
            )

        if x.shape[1] != self.input_dim:
            raise ValueError(
                "Critic input dimension mismatch: "
                f"expected {self.input_dim}, "
                f"received {x.shape[1]}."
            )

        return self.network(x)

# --------------------------------------------------------------------------------------------------
# 4. Opacus compatibility validation
# --------------------------------------------------------------------------------------------------

def validate_critic_for_dp(
    critic,
):
    """
    Validate critic compatibility with Opacus.
    """

    validation_errors = ModuleValidator.validate(
        critic,
        strict=False,
    )

    if validation_errors:
        raise RuntimeError(
            "Critic is not compatible with Opacus:\n"
            +
            "\n".join(
                str(error)
                for error in validation_errors
            )
        )

    return True

# --------------------------------------------------------------------------------------------------
# 5. Wrap critic for per-example gradients
# --------------------------------------------------------------------------------------------------

def wrap_critic_for_per_sample_gradients(
    critic,
):
    """
    Wrap the critic with Opacus GradSampleModule.

    The wrapped module exposes parameter.grad_sample after
    a loss.backward() call.
    """

    validate_critic_for_dp(
        critic
    )

    dp_critic = GradSampleModule(
        critic,
        batch_first=True,
        loss_reduction=LOSS_REDUCTION,
        strict=True,
        force_functorch=False,
    )

    return dp_critic

# --------------------------------------------------------------------------------------------------
# 6. Extract per-example gradients
# --------------------------------------------------------------------------------------------------

def get_per_example_gradients(
    dp_critic,
):
    """
    Extract per-example gradients from GradSampleModule.

    Returns
    -------
    dict
        Parameter name → grad_sample tensor.

    Expected shape
    --------------
    [batch_size, *parameter.shape]
    """

    gradients = {}

    for name, parameter in (
        dp_critic.named_parameters()
    ):

        grad_sample = getattr(
            parameter,
            "grad_sample",
            None,
        )

        if grad_sample is None:
            raise RuntimeError(
                f"Missing grad_sample for parameter: {name}"
            )

        # ------------------------------------------------------------------------------------------
        # Opacus may retain accumulated grad_sample tensors as a list.
        # A single backward pass must produce exactly one tensor.
        # ------------------------------------------------------------------------------------------

        if isinstance(
            grad_sample,
            list,
        ):

            if len(
                grad_sample
            ) != 1:
                raise RuntimeError(
                    f"Unexpected accumulated grad_sample for {name}: "
                    f"{len(grad_sample)} entries."
                )

            grad_sample = grad_sample[0]

        if not torch.is_tensor(
            grad_sample
        ):
            raise RuntimeError(
                f"grad_sample for {name} is not a tensor."
            )

        gradients[name] = grad_sample

    if not gradients:
        raise RuntimeError(
            "No per-example gradients were extracted."
        )

    return gradients

# --------------------------------------------------------------------------------------------------
# 7. Validate per-example gradient tensors
# --------------------------------------------------------------------------------------------------

def validate_per_example_gradient_tensors(
    dp_critic,
    gradients,
    batch_size,
):
    """
    Validate per-example gradient tensor shape, finiteness,
    and sample dimension.
    """

    rows = []

    for name, parameter in (
        dp_critic.named_parameters()
    ):

        if name not in gradients:
            raise RuntimeError(
                f"Missing gradient record for {name}."
            )

        grad_sample = gradients[name]

        expected_shape = (
            int(batch_size),
            *tuple(
                parameter.shape
            ),
        )

        actual_shape = tuple(
            grad_sample.shape
        )

        if actual_shape != expected_shape:
            raise RuntimeError(
                f"Invalid grad_sample shape for {name}: "
                f"expected {expected_shape}, "
                f"received {actual_shape}."
            )

        if not torch.isfinite(
            grad_sample
        ).all():
            raise RuntimeError(
                f"Non-finite per-example gradient detected for {name}."
            )

        rows.append({
            "parameter": name,
            "batch_size": int(
                batch_size
            ),
            "parameter_shape": str(
                tuple(
                    parameter.shape
                )
            ),
            "grad_sample_shape": str(
                actual_shape
            ),
            "finite": True,
            "status": "PASS",
        })

    return pd.DataFrame(
        rows
    )

# --------------------------------------------------------------------------------------------------
# 8. Verify genuinely sample-wise gradient information
#
# A per-example gradient implementation should preserve the leading
# sample dimension. For a non-degenerate test batch, at least one
# trainable parameter should exhibit different gradient values across
# examples.
# --------------------------------------------------------------------------------------------------

def validate_samplewise_variation(
    gradients,
    batch_size,
):
    """
    Confirm that per-example gradients retain sample-wise information.

    This is a diagnostic smoke test, not a mathematical proof of privacy.
    """

    if batch_size < 2:
        raise ValueError(
            "Sample-wise validation requires at least two examples."
        )

    variation_records = []

    for name, grad_sample in gradients.items():

        flat = grad_sample.reshape(
            batch_size,
            -1,
        )

        sample_difference = (
            flat[1:]
            -
            flat[:-1]
        )

        variation_norm = torch.linalg.vector_norm(
            sample_difference,
            dim=1,
        )

        max_variation = float(
            variation_norm.max().detach().cpu()
        )

        variation_records.append({
            "parameter": name,
            "max_adjacent_sample_gradient_difference": (
                max_variation
            ),
            "samplewise_variation": (
                max_variation > 0.0
            ),
        })

    variation_df = pd.DataFrame(
        variation_records
    )

    if not variation_df[
        "samplewise_variation"
    ].any():
        raise RuntimeError(
            "No sample-wise gradient variation was detected. "
            "Per-example gradient behavior could not be validated."
        )

    return variation_df

# --------------------------------------------------------------------------------------------------
# 9. Run actual Opacus per-example-gradient smoke tests
# --------------------------------------------------------------------------------------------------

PER_EXAMPLE_GRADIENT_VALIDATION_ROWS = []

PER_EXAMPLE_GRADIENT_TENSOR_ROWS = []

PER_EXAMPLE_GRADIENT_VARIATION_ROWS = []

DP_GRADIENT_TEST_BATCH_SIZE = min(
    8,
    DP_BATCH_SIZE,
)

if DP_GRADIENT_TEST_BATCH_SIZE < 2:
    raise RuntimeError(
        "DP gradient test batch size must be at least 2."
    )

for dataset_id in DATASET_IDS:

    input_dim = int(
        architecture_df.loc[
            architecture_df["dataset"] == dataset_id,
            "transformed_dimension",
        ].iloc[0]
    )

    if input_dim <= 0:
        raise RuntimeError(
            f"{dataset_id}: invalid transformed dimension {input_dim}."
        )

    # ----------------------------------------------------------------------------------------------
    # Deterministic test seed
    # ----------------------------------------------------------------------------------------------

    dataset_seed = (
        MASTER_SEED
        +
        DATASET_IDS.index(
            dataset_id
        )
        +
        1000
    )

    torch.manual_seed(
        dataset_seed
    )

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            dataset_seed
        )

    # ----------------------------------------------------------------------------------------------
    # Construct fresh critic
    # ----------------------------------------------------------------------------------------------

    critic = SPPGANCritic(
        input_dim=input_dim
    ).to(
        DEVICE
    )

    critic.train()

    # ----------------------------------------------------------------------------------------------
    # Opacus module validation
    # ----------------------------------------------------------------------------------------------

    validate_critic_for_dp(
        critic
    )

    # ----------------------------------------------------------------------------------------------
    # Wrap with GradSampleModule
    # ----------------------------------------------------------------------------------------------

    dp_critic = wrap_critic_for_per_sample_gradients(
        critic
    )

    dp_critic = dp_critic.to(
        DEVICE
    )

    dp_critic.train()

    # ----------------------------------------------------------------------------------------------
    # Synthetic transformed input for gradient-interface testing.
    #
    # This is an interface test only. No real dataset rows are used here.
    # ----------------------------------------------------------------------------------------------

    x = torch.randn(
        DP_GRADIENT_TEST_BATCH_SIZE,
        input_dim,
        device=DEVICE,
        dtype=torch.float32,
        requires_grad=False,
    )

    if not torch.isfinite(
        x
    ).all():
        raise RuntimeError(
            f"{dataset_id}: test input contains non-finite values."
        )

    # ----------------------------------------------------------------------------------------------
    # Forward pass
    # ----------------------------------------------------------------------------------------------

    dp_critic.zero_grad(
        set_to_none=True
    )

    scores = dp_critic(
        x
    )

    if scores.shape != (
        DP_GRADIENT_TEST_BATCH_SIZE,
        1,
    ):
        raise RuntimeError(
            f"{dataset_id}: invalid critic output shape "
            f"{tuple(scores.shape)}."
        )

    if not torch.isfinite(
        scores
    ).all():
        raise RuntimeError(
            f"{dataset_id}: critic output contains non-finite values."
        )

    # ----------------------------------------------------------------------------------------------
    # Per-example scalar loss.
    #
    # For the gradient-interface test, the mean of the individual
    # critic scores is used. Opacus converts this backward pass into
    # parameter-wise per-example gradients.
    # ----------------------------------------------------------------------------------------------

    loss = (
        -scores
    ).mean()

    if not (
        torch.is_tensor(loss)
        and loss.ndim == 0
    ):
        raise RuntimeError(
            f"{dataset_id}: critic loss is not a scalar."
        )

    if not torch.isfinite(
        loss
    ):
        raise RuntimeError(
            f"{dataset_id}: critic loss is non-finite."
        )

    # ----------------------------------------------------------------------------------------------
    # Backward pass
    # ----------------------------------------------------------------------------------------------

    loss.backward()

    # ----------------------------------------------------------------------------------------------
    # Extract grad_sample
    # ----------------------------------------------------------------------------------------------

    gradients = get_per_example_gradients(
        dp_critic
    )

    # ----------------------------------------------------------------------------------------------
    # Validate gradient tensors
    # ----------------------------------------------------------------------------------------------

    tensor_validation_df = (
        validate_per_example_gradient_tensors(
            dp_critic=dp_critic,
            gradients=gradients,
            batch_size=DP_GRADIENT_TEST_BATCH_SIZE,
        )
    )

    tensor_validation_df.insert(
        0,
        "dataset",
        dataset_id,
    )

    PER_EXAMPLE_GRADIENT_TENSOR_ROWS.extend(
        tensor_validation_df.to_dict(
            orient="records"
        )
    )

    # ----------------------------------------------------------------------------------------------
    # Validate sample-wise variation
    # ----------------------------------------------------------------------------------------------

    variation_df = (
        validate_samplewise_variation(
            gradients=gradients,
            batch_size=DP_GRADIENT_TEST_BATCH_SIZE,
        )
    )

    variation_df.insert(
        0,
        "dataset",
        dataset_id,
    )

    PER_EXAMPLE_GRADIENT_VARIATION_ROWS.extend(
        variation_df.to_dict(
            orient="records"
        )
    )

    # ----------------------------------------------------------------------------------------------
    # Gradient norm diagnostics
    # ----------------------------------------------------------------------------------------------

    total_parameter_count = 0
    total_gradient_elements = 0
    finite_gradient_parameters = 0

    for name, parameter in (
        dp_critic.named_parameters()
    ):

        total_parameter_count += 1

        total_gradient_elements += int(
            gradients[name].numel()
        )

        if torch.isfinite(
            gradients[name]
        ).all():
            finite_gradient_parameters += 1

    # ----------------------------------------------------------------------------------------------
    # Final dataset-level result
    # ----------------------------------------------------------------------------------------------

    PER_EXAMPLE_GRADIENT_VALIDATION_ROWS.append({
        "dataset": dataset_id,
        "critic_input_dimension": int(
            input_dim
        ),
        "test_batch_size": int(
            DP_GRADIENT_TEST_BATCH_SIZE
        ),
        "critic_output_shape": str(
            tuple(
                scores.shape
            )
        ),
        "loss_scalar": bool(
            loss.ndim == 0
        ),
        "parameter_count": int(
            total_parameter_count
        ),
        "gradient_parameter_count": int(
            len(
                gradients
            )
        ),
        "total_gradient_elements": int(
            total_gradient_elements
        ),
        "finite_gradient_parameters": int(
            finite_gradient_parameters
        ),
        "all_gradient_tensors_finite": bool(
            finite_gradient_parameters
            ==
            total_parameter_count
        ),
        "samplewise_gradient_variation": bool(
            variation_df[
                "samplewise_variation"
            ].any()
        ),
        "opacus_grad_sample_available": True,
        "status": "PASS",
    })

    # ----------------------------------------------------------------------------------------------
    # Cleanup
    # ----------------------------------------------------------------------------------------------

    del (
        dp_critic,
        critic,
        x,
        scores,
        loss,
        gradients,
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# --------------------------------------------------------------------------------------------------
# 10. Assemble validation DataFrames
# --------------------------------------------------------------------------------------------------

PER_EXAMPLE_GRADIENT_VALIDATION_DF = pd.DataFrame(
    PER_EXAMPLE_GRADIENT_VALIDATION_ROWS
)

PER_EXAMPLE_GRADIENT_TENSOR_DF = pd.DataFrame(
    PER_EXAMPLE_GRADIENT_TENSOR_ROWS
)

PER_EXAMPLE_GRADIENT_VARIATION_DF = pd.DataFrame(
    PER_EXAMPLE_GRADIENT_VARIATION_ROWS
)

# --------------------------------------------------------------------------------------------------
# 11. Final structural validation
# --------------------------------------------------------------------------------------------------

if len(
    PER_EXAMPLE_GRADIENT_VALIDATION_DF
) != len(
    DATASET_IDS
):
    raise RuntimeError(
        "Per-example gradient validation does not contain one row per dataset."
    )

if set(
    PER_EXAMPLE_GRADIENT_VALIDATION_DF["dataset"]
) != set(
    DATASET_IDS
):
    raise RuntimeError(
        "Per-example gradient dataset coverage mismatch."
    )

if not (
    PER_EXAMPLE_GRADIENT_VALIDATION_DF[
        "status"
    ]
    .eq("PASS")
    .all()
):
    raise RuntimeError(
        "One or more per-example gradient tests failed."
    )

if not (
    PER_EXAMPLE_GRADIENT_VALIDATION_DF[
        "opacus_grad_sample_available"
    ]
    .all()
):
    raise RuntimeError(
        "Opacus grad_sample was not available for every dataset."
    )

if not (
    PER_EXAMPLE_GRADIENT_VALIDATION_DF[
        "all_gradient_tensors_finite"
    ]
    .all()
):
    raise RuntimeError(
        "Non-finite per-example gradients detected."
    )

if not (
    PER_EXAMPLE_GRADIENT_VALIDATION_DF[
        "samplewise_gradient_variation"
    ]
    .all()
):
    raise RuntimeError(
        "Sample-wise gradient variation was not validated for every dataset."
    )

if PER_EXAMPLE_GRADIENT_TENSOR_DF.empty:
    raise RuntimeError(
        "Per-example gradient tensor validation is empty."
    )

if PER_EXAMPLE_GRADIENT_VARIATION_DF.empty:
    raise RuntimeError(
        "Per-example gradient variation validation is empty."
    )

# --------------------------------------------------------------------------------------------------
# 12. Save Section 6 validation artifacts
# --------------------------------------------------------------------------------------------------

NB10_METADATA_ROOT = (
    NB10_ROOT
    /
    "metadata"
)

NB10_VALIDATION_ROOT = (
    NB10_ROOT
    /
    "validation"
)

NB10_METADATA_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

NB10_VALIDATION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

PER_EXAMPLE_GRADIENT_VALIDATION_PATH = (
    NB10_VALIDATION_ROOT
    /
    "per_example_gradient_validation.csv"
)

PER_EXAMPLE_GRADIENT_TENSOR_PATH = (
    NB10_VALIDATION_ROOT
    /
    "per_example_gradient_tensor_validation.csv"
)

PER_EXAMPLE_GRADIENT_VARIATION_PATH = (
    NB10_VALIDATION_ROOT
    /
    "per_example_gradient_variation.csv"
)

PER_EXAMPLE_GRADIENT_VALIDATION_DF.to_csv(
    PER_EXAMPLE_GRADIENT_VALIDATION_PATH,
    index=False,
)

PER_EXAMPLE_GRADIENT_TENSOR_DF.to_csv(
    PER_EXAMPLE_GRADIENT_TENSOR_PATH,
    index=False,
)

PER_EXAMPLE_GRADIENT_VARIATION_DF.to_csv(
    PER_EXAMPLE_GRADIENT_VARIATION_PATH,
    index=False,
)

# --------------------------------------------------------------------------------------------------
# 13. Save Section 6 validation manifest
# --------------------------------------------------------------------------------------------------

SECTION_06_MANIFEST = {
    "notebook": "10",
    "section": "6",
    "title": "Define Per-Example Discriminator Gradients",
    "framework": "SPP-GAN",
    "protected_component": "SPP-GAN discriminator",
    "mechanism": "Opacus GradSampleModule",
    "opacus_version": str(
        opacus.__version__
    ),
    "grad_sample_mode": GRAD_SAMPLE_MODE,
    "loss_reduction": LOSS_REDUCTION,
    "critic_architecture_source": (
        "Notebook 08 validated architecture contract"
    ),
    "critic_architecture": {
        "hidden_dimension_1": 256,
        "hidden_dimension_2": 256,
        "activation": "LeakyReLU",
        "negative_slope": 0.2,
        "output_dimension": 1,
        "dropout": False,
        "batch_normalization": False,
        "sigmoid_output": False,
    },
    "datasets_validated": DATASET_IDS,
    "test_batch_size": int(
        DP_GRADIENT_TEST_BATCH_SIZE
    ),
    "real_dataset_rows_used": False,
    "samplewise_gradient_shape": (
        "[batch_size, *parameter.shape]"
    ),
    "finite_gradient_validation": True,
    "samplewise_variation_validation": True,
    "training_performed": False,
    "privacy_accounting_performed": False,
    "achieved_epsilon": "DEFERRED_TO_NOTEBOOK_11",
    "created_utc": datetime.utcnow().isoformat()
    + "Z",
    "artifacts": {
        "dataset_validation": str(
            PER_EXAMPLE_GRADIENT_VALIDATION_PATH
        ),
        "tensor_validation": str(
            PER_EXAMPLE_GRADIENT_TENSOR_PATH
        ),
        "variation_validation": str(
            PER_EXAMPLE_GRADIENT_VARIATION_PATH
        ),
    },
}

SECTION_06_MANIFEST_PATH = (
    NB10_METADATA_ROOT
    /
    "section_06_per_example_gradient_manifest.json"
)

with open(
    SECTION_06_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        SECTION_06_MANIFEST,
        f,
        indent=2,
    )

# --------------------------------------------------------------------------------------------------
# 14. Final output
# --------------------------------------------------------------------------------------------------

print()
print(
    "✓ SPP-GAN critic defined from Notebook 08 architecture contract."
)

print(
    "✓ Opacus ModuleValidator compatibility : PASS"
)

print(
    "✓ GradSampleModule wrapping             : PASS"
)

print(
    "✓ Per-example grad_sample extraction    : PASS"
)

print(
    "✓ Gradient shape validation             : PASS"
)

print(
    "✓ Gradient finiteness validation        : PASS"
)

print(
    "✓ Sample-wise gradient variation        : PASS"
)

print()
print("Per-example gradient validation:")

display(
    PER_EXAMPLE_GRADIENT_VALIDATION_DF
)

print()
print(
    f"✓ Dataset validation artifact : "
    f"{PER_EXAMPLE_GRADIENT_VALIDATION_PATH}"
)

print(
    f"✓ Tensor validation artifact  : "
    f"{PER_EXAMPLE_GRADIENT_TENSOR_PATH}"
)

print(
    f"✓ Variation artifact          : "
    f"{PER_EXAMPLE_GRADIENT_VARIATION_PATH}"
)

print(
    f"✓ Section manifest            : "
    f"{SECTION_06_MANIFEST_PATH}"
)

print()
print(
    "Privacy boundary:"
)

print(
    "  ✓ Per-example discriminator gradients defined."
)

print(
    "  ✓ Privacy noise NOT applied in Section 6."
)

print(
    "  ✓ Gradient clipping deferred to Section 7."
)

print(
    "  ✓ Gaussian noise deferred to Section 8."
)

print(
    "  ✓ DP-SGD update deferred to Section 9."
)

print(
    "  ✓ Formal privacy accounting deferred to Notebook 11."
)

print(
    "  ✓ End-to-end privacy claim remains disabled."
)

print()
print("SECTION 6 STATUS: PASS")


6. DEFINE PER-EXAMPLE DISCRIMINATOR GRADIENTS


/tmp/ipykernel_5574/3596670857.py:622: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  loss.backward()
/tmp/ipykernel_5574/3596670857.py:622: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  loss.backward()
/tmp/ipykernel_5574/3596670857.py:622: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  loss.backward()
/tmp/ipykernel_5574/35966


✓ SPP-GAN critic defined from Notebook 08 architecture contract.
✓ Opacus ModuleValidator compatibility : PASS
✓ GradSampleModule wrapping             : PASS
✓ Per-example grad_sample extraction    : PASS
✓ Gradient shape validation             : PASS
✓ Gradient finiteness validation        : PASS
✓ Sample-wise gradient variation        : PASS

Per-example gradient validation:


,dataset,critic_input_dimension,test_batch_size,critic_output_shape,loss_scalar,parameter_count,gradient_parameter_count,total_gradient_elements,finite_gradient_parameters,all_gradient_tensors_finite,samplewise_gradient_variation,opacus_grad_sample_available,status
0,adult_income,105,8,"(8, 1)",True,6,6,745480,6,True,True,True,PASS
1,bank_marketing,51,8,"(8, 1)",True,6,6,634888,6,True,True,True,PASS
2,diabetes_130us,2329,8,"(8, 1)",True,6,6,5300232,6,True,True,True,PASS



✓ Dataset validation artifact : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10/validation/per_example_gradient_validation.csv
✓ Tensor validation artifact  : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10/validation/per_example_gradient_tensor_validation.csv
✓ Variation artifact          : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10/validation/per_example_gradient_variation.csv
✓ Section manifest            : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10/metadata/section_06_per_example_gradient_manifest.json

Privacy boundary:
  ✓ Per-example discriminator gradients defined.
  ✓ Privacy noise NOT applied in Section 6.
  ✓ Gradient clipping deferred to Section 7.
  ✓ Gaussian noise deferred to Section 8.
  ✓ DP-SGD update deferred to Section 9.
  ✓ Formal privacy accounting deferred to Notebook 11.
  ✓ End-to-end privacy claim remains disabled.

SECTION 6 STATUS: PASS


In [7]:
# ==================================================================================================
# 7. DEFINE GRADIENT CLIPPING
# ==================================================================================================

print("\n" + "=" * 100)
print("7. DEFINE GRADIENT CLIPPING")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 0. Imports and required objects
# --------------------------------------------------------------------------------------------------

from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd
import torch

# --------------------------------------------------------------------------------------------------
# 1. Validate required configuration and Section 6 dependencies
# --------------------------------------------------------------------------------------------------

if "MAX_GRAD_NORM" not in globals():
    raise RuntimeError(
        "MAX_GRAD_NORM is not defined. Run Notebook 10 Section 4 first."
    )

if not np.isfinite(float(MAX_GRAD_NORM)):
    raise ValueError(
        f"MAX_GRAD_NORM must be finite. Received: {MAX_GRAD_NORM}"
    )

if float(MAX_GRAD_NORM) <= 0:
    raise ValueError(
        f"MAX_GRAD_NORM must be positive. Received: {MAX_GRAD_NORM}"
    )

if "DATASET_IDS" not in globals():
    raise RuntimeError(
        "DATASET_IDS is not defined. Run Notebook 10 Section 2 first."
    )

if "CRITIC_INPUT_DIMENSIONS" not in globals():
    raise RuntimeError(
        "CRITIC_INPUT_DIMENSIONS is not defined. Run Notebook 10 Section 3 first."
    )

if "SPPGANCritic" not in globals():
    raise RuntimeError(
        "SPPGANCritic is not defined. Run Notebook 10 Section 6 first."
    )

if "get_per_example_gradients" not in globals():
    raise RuntimeError(
        "get_per_example_gradients is not defined. Run Notebook 10 Section 6 first."
    )

print("✓ Required configuration validated.")
print(f"✓ Flat L2 clipping threshold C = {float(MAX_GRAD_NORM):.6f}")


# --------------------------------------------------------------------------------------------------
# 2. Per-example flat L2 gradient norms
# --------------------------------------------------------------------------------------------------

def per_example_gradient_norms(gradients):
    """
    Compute the flat L2 norm across all discriminator parameters
    independently for every example.

    For example i:

        ||g_i||_2 =
            sqrt(
                sum_p sum_j g_{i,p,j}^2
            )

    Parameters
    ----------
    gradients : dict[str, torch.Tensor]
        Per-example gradients keyed by parameter name.
        Each tensor must have shape:

            [batch_size, *parameter_shape]

    Returns
    -------
    torch.Tensor
        Per-example L2 norms with shape [batch_size].
    """

    if not isinstance(gradients, dict) or not gradients:
        raise ValueError(
            "gradients must be a non-empty dictionary."
        )

    first_name, first_gradient = next(iter(gradients.items()))

    if not isinstance(first_gradient, torch.Tensor):
        raise TypeError(
            f"Gradient for '{first_name}' must be a torch.Tensor."
        )

    if first_gradient.ndim < 1:
        raise ValueError(
            f"Gradient for '{first_name}' must have at least one dimension."
        )

    batch_size = int(first_gradient.shape[0])

    if batch_size <= 0:
        raise ValueError(
            "Gradient batch size must be positive."
        )

    if not torch.isfinite(first_gradient).all():
        raise ValueError(
            f"Non-finite values detected in gradient '{first_name}'."
        )

    squared_norms = torch.zeros(
        batch_size,
        device=first_gradient.device,
        dtype=first_gradient.dtype,
    )

    for name, gradient in gradients.items():

        if not isinstance(gradient, torch.Tensor):
            raise TypeError(
                f"Gradient for '{name}' must be a torch.Tensor."
            )

        if gradient.ndim < 1:
            raise ValueError(
                f"Gradient for '{name}' must have at least one dimension."
            )

        if int(gradient.shape[0]) != batch_size:
            raise ValueError(
                "All per-example gradients must have the same batch dimension. "
                f"Expected {batch_size}, got {gradient.shape[0]} for '{name}'."
            )

        if gradient.device != first_gradient.device:
            raise ValueError(
                f"Gradient '{name}' is on {gradient.device}, "
                f"expected {first_gradient.device}."
            )

        if gradient.dtype != first_gradient.dtype:
            raise ValueError(
                f"Gradient '{name}' has dtype {gradient.dtype}, "
                f"expected {first_gradient.dtype}."
            )

        if not torch.isfinite(gradient).all():
            raise ValueError(
                f"Non-finite values detected in gradient '{name}'."
            )

        flat_gradient = gradient.reshape(
            batch_size,
            -1,
        )

        squared_norms = squared_norms + (
            flat_gradient
            .pow(2)
            .sum(dim=1)
        )

    norms = torch.sqrt(
        squared_norms.clamp_min(0.0)
    )

    if not torch.isfinite(norms).all():
        raise ValueError(
            "Non-finite per-example gradient norms detected."
        )

    return norms


# --------------------------------------------------------------------------------------------------
# 3. Flat clipping factors
# --------------------------------------------------------------------------------------------------

def flat_clipping_factors(
    gradients,
    max_grad_norm,
):
    """
    Compute flat per-example L2 clipping factors.

    For example i:

        alpha_i = min(
            1,
            C / (||g_i||_2 + epsilon)
        )

    where C is the maximum allowed gradient norm.
    """

    max_grad_norm = float(max_grad_norm)

    if not np.isfinite(max_grad_norm):
        raise ValueError(
            "max_grad_norm must be finite."
        )

    if max_grad_norm <= 0:
        raise ValueError(
            "max_grad_norm must be positive."
        )

    norms = per_example_gradient_norms(
        gradients
    )

    epsilon = torch.tensor(
        1e-12,
        device=norms.device,
        dtype=norms.dtype,
    )

    factors = (
        max_grad_norm
        /
        (norms + epsilon)
    ).clamp(
        min=0.0,
        max=1.0,
    )

    if not torch.isfinite(factors).all():
        raise ValueError(
            "Non-finite clipping factors detected."
        )

    return factors


# --------------------------------------------------------------------------------------------------
# 4. Apply flat per-example L2 clipping
# --------------------------------------------------------------------------------------------------

def clip_per_example_gradients(
    gradients,
    max_grad_norm,
):
    """
    Apply flat per-example L2 clipping independently to every example.

    The same clipping factor for a given example is applied across
    every discriminator parameter tensor.
    """

    factors = flat_clipping_factors(
        gradients,
        max_grad_norm,
    )

    clipped_gradients = {}

    for name, gradient in gradients.items():

        view_shape = (
            [int(factors.shape[0])]
            +
            [1] * (gradient.ndim - 1)
        )

        clipped_gradient = (
            gradient
            *
            factors.reshape(view_shape)
        )

        if not torch.isfinite(clipped_gradient).all():
            raise ValueError(
                f"Non-finite clipped gradient detected for '{name}'."
            )

        if clipped_gradient.shape != gradient.shape:
            raise RuntimeError(
                f"Gradient shape changed during clipping for '{name}'. "
                f"Original={tuple(gradient.shape)}, "
                f"Clipped={tuple(clipped_gradient.shape)}."
            )

        clipped_gradients[name] = clipped_gradient

    return clipped_gradients


# --------------------------------------------------------------------------------------------------
# 5. Validate clipping mathematically
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("CLIPPING MATHEMATICAL VALIDATION")
print("-" * 100)

VALIDATION_DEVICE = (
    DEVICE
    if "DEVICE" in globals()
    else torch.device("cuda" if torch.cuda.is_available() else "cpu")
)

VALIDATION_DTYPE = torch.float32
VALIDATION_BATCH_SIZE = 8
VALIDATION_SEED = (
    int(MASTER_SEED) + 7000
    if "MASTER_SEED" in globals()
    else 9025
)

torch.manual_seed(VALIDATION_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(VALIDATION_SEED)

# -----------------------------------------------------------------------------------------------
# Construct deterministic synthetic per-example gradients.
#
# The values intentionally include:
#   - zero gradients
#   - gradients below C
#   - gradients exactly at C
#   - gradients above C
#
# This validates both clipping and non-clipping behavior.
# -----------------------------------------------------------------------------------------------

C = float(MAX_GRAD_NORM)

test_gradients = {
    "weight": torch.tensor(
        [
            [0.0, 0.0],
            [0.3, 0.4],
            [C, 0.0],
            [2.0 * C, 0.0],
            [3.0 * C, 0.0],
            [0.6, 0.8],
            [0.0, 0.75 * C],
            [4.0 * C, 0.0],
        ],
        dtype=VALIDATION_DTYPE,
        device=VALIDATION_DEVICE,
    ),
    "bias": torch.tensor(
        [
            [0.0],
            [0.0],
            [0.0],
            [0.0],
            [0.0],
            [0.0],
            [0.0],
            [0.0],
        ],
        dtype=VALIDATION_DTYPE,
        device=VALIDATION_DEVICE,
    ),
}

raw_norms = per_example_gradient_norms(
    test_gradients
)

clipping_factors = flat_clipping_factors(
    test_gradients,
    C,
)

clipped_test_gradients = clip_per_example_gradients(
    test_gradients,
    C,
)

clipped_norms = per_example_gradient_norms(
    clipped_test_gradients
)

# -----------------------------------------------------------------------------------------------
# Validation 1 — raw norms finite
# -----------------------------------------------------------------------------------------------

raw_norms_finite = bool(
    torch.isfinite(raw_norms).all().item()
)

if not raw_norms_finite:
    raise RuntimeError(
        "Raw gradient norm validation failed."
    )

print("✓ Raw per-example gradient norms : PASS")


# -----------------------------------------------------------------------------------------------
# Validation 2 — clipping factors finite and bounded
# -----------------------------------------------------------------------------------------------

factors_finite = bool(
    torch.isfinite(clipping_factors).all().item()
)

factors_bounded = bool(
    (
        (clipping_factors >= 0.0)
        &
        (clipping_factors <= 1.0)
    ).all().item()
)

if not factors_finite:
    raise RuntimeError(
        "Clipping factors contain non-finite values."
    )

if not factors_bounded:
    raise RuntimeError(
        "Clipping factors are outside [0, 1]."
    )

print("✓ Clipping factors finite and bounded [0, 1] : PASS")


# -----------------------------------------------------------------------------------------------
# Validation 3 — zero gradients remain zero
# -----------------------------------------------------------------------------------------------

zero_gradient_preserved = bool(
    torch.allclose(
        clipped_test_gradients["weight"][0],
        test_gradients["weight"][0],
        atol=0.0,
        rtol=0.0,
    )
    and
    torch.allclose(
        clipped_test_gradients["bias"][0],
        test_gradients["bias"][0],
        atol=0.0,
        rtol=0.0,
    )
)

if not zero_gradient_preserved:
    raise RuntimeError(
        "Zero-gradient preservation validation failed."
    )

print("✓ Zero-gradient stability : PASS")


# -----------------------------------------------------------------------------------------------
# Validation 4 — gradients at or below C remain unchanged
# -----------------------------------------------------------------------------------------------

non_clipped_mask = raw_norms <= C + 1e-7

unchanged = True

for name in test_gradients:

    if not torch.allclose(
        clipped_test_gradients[name][non_clipped_mask],
        test_gradients[name][non_clipped_mask],
        atol=1e-7,
        rtol=1e-6,
    ):
        unchanged = False
        break

if not unchanged:
    raise RuntimeError(
        "Gradients at or below C were unexpectedly modified."
    )

print("✓ Gradients at or below C preserved : PASS")


# -----------------------------------------------------------------------------------------------
# Validation 5 — gradients above C are clipped
# -----------------------------------------------------------------------------------------------

above_threshold_mask = raw_norms > C

if not bool(above_threshold_mask.any().item()):
    raise RuntimeError(
        "Clipping test did not contain any above-threshold examples."
    )

above_threshold_reduced = bool(
    (
        clipped_norms[above_threshold_mask]
        <
        raw_norms[above_threshold_mask]
    ).all().item()
)

if not above_threshold_reduced:
    raise RuntimeError(
        "Above-threshold gradients were not reduced."
    )

print("✓ Above-threshold gradients clipped : PASS")


# -----------------------------------------------------------------------------------------------
# Validation 6 — post-clipping norm is bounded by C
# -----------------------------------------------------------------------------------------------

post_clip_bound = bool(
    (
        clipped_norms
        <=
        C + 1e-6
    ).all().item()
)

if not post_clip_bound:
    raise RuntimeError(
        "Post-clipping gradient norm exceeds C."
    )

print("✓ Post-clipping norm ≤ C : PASS")


# -----------------------------------------------------------------------------------------------
# Validation 7 — clipping factors agree with the mathematical definition
# -----------------------------------------------------------------------------------------------

expected_factors = torch.minimum(
    torch.ones_like(raw_norms),
    C / (raw_norms + 1e-12),
)

factor_formula_match = bool(
    torch.allclose(
        clipping_factors,
        expected_factors,
        atol=1e-7,
        rtol=1e-6,
    )
)

if not factor_formula_match:
    raise RuntimeError(
        "Clipping factor formula validation failed."
    )

print("✓ Clipping factor mathematical definition : PASS")


# -----------------------------------------------------------------------------------------------
# Validation 8 — output shape preservation
# -----------------------------------------------------------------------------------------------

shape_preserved = all(
    clipped_test_gradients[name].shape
    ==
    test_gradients[name].shape
    for name in test_gradients
)

if not shape_preserved:
    raise RuntimeError(
        "Gradient tensor shapes changed during clipping."
    )

print("✓ Per-example gradient tensor shapes preserved : PASS")


# -----------------------------------------------------------------------------------------------
# Validation 9 — finite clipped gradients
# -----------------------------------------------------------------------------------------------

all_clipped_finite = all(
    bool(torch.isfinite(gradient).all().item())
    for gradient in clipped_test_gradients.values()
)

if not all_clipped_finite:
    raise RuntimeError(
        "Non-finite clipped gradients detected."
    )

print("✓ Clipped gradients finite : PASS")


# --------------------------------------------------------------------------------------------------
# 6. Validate clipping against all Notebook 08 critic dimensions
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("DATASET-SPECIFIC CLIPPING VALIDATION")
print("-" * 100)

DATASET_CLIPPING_RESULTS = []

for dataset_index, dataset_id in enumerate(DATASET_IDS):

    dataset_seed = (
        int(MASTER_SEED)
        + 7000
        + dataset_index
        if "MASTER_SEED" in globals()
        else 9025 + dataset_index
    )

    torch.manual_seed(dataset_seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(dataset_seed)

    if dataset_id not in CRITIC_INPUT_DIMENSIONS:
        raise KeyError(
            f"Missing critic input dimension for dataset '{dataset_id}'."
        )

    input_dim = int(
        CRITIC_INPUT_DIMENSIONS[dataset_id]
    )

    critic = SPPGANCritic(
        input_dim=input_dim
    ).to(VALIDATION_DEVICE)

    critic.eval()

    # -------------------------------------------------------------------------------------------
    # Build deterministic per-example gradients matching the actual critic parameter structure.
    # -------------------------------------------------------------------------------------------

    gradients = {}

    for parameter_name, parameter in critic.named_parameters():

        if not parameter.requires_grad:
            continue

        parameter_shape = tuple(
            parameter.shape
        )

        generator = torch.Generator(
            device=VALIDATION_DEVICE
        )

        generator.manual_seed(
            dataset_seed
            + len(gradients)
            + 1
        )

        gradient = torch.randn(
            (
                VALIDATION_BATCH_SIZE,
                *parameter_shape,
            ),
            generator=generator,
            device=VALIDATION_DEVICE,
            dtype=VALIDATION_DTYPE,
        )

        gradients[parameter_name] = gradient

    if not gradients:
        raise RuntimeError(
            f"No trainable critic parameters found for '{dataset_id}'."
        )

    raw_dataset_norms = per_example_gradient_norms(
        gradients
    )

    factors = flat_clipping_factors(
        gradients,
        C,
    )

    clipped_gradients = clip_per_example_gradients(
        gradients,
        C,
    )

    clipped_dataset_norms = per_example_gradient_norms(
        clipped_gradients
    )

    raw_finite = bool(
        torch.isfinite(raw_dataset_norms).all().item()
    )

    factor_finite = bool(
        torch.isfinite(factors).all().item()
    )

    factor_bounded = bool(
        (
            (factors >= 0.0)
            &
            (factors <= 1.0)
        ).all().item()
    )

    clipped_finite = bool(
        all(
            torch.isfinite(g).all().item()
            for g in clipped_gradients.values()
        )
    )

    bounded_after_clipping = bool(
        (
            clipped_dataset_norms
            <=
            C + 1e-6
        ).all().item()
    )

    shape_preserved_dataset = all(
        clipped_gradients[name].shape
        ==
        gradients[name].shape
        for name in gradients
    )

    sample_count_preserved = all(
        int(clipped_gradients[name].shape[0])
        ==
        VALIDATION_BATCH_SIZE
        for name in clipped_gradients
    )

    dataset_status = all(
        [
            raw_finite,
            factor_finite,
            factor_bounded,
            clipped_finite,
            bounded_after_clipping,
            shape_preserved_dataset,
            sample_count_preserved,
        ]
    )

    if not dataset_status:
        raise RuntimeError(
            f"Dataset-specific clipping validation failed for '{dataset_id}'."
        )

    DATASET_CLIPPING_RESULTS.append(
        {
            "dataset_id": dataset_id,
            "critic_input_dimension": input_dim,
            "validation_batch_size": VALIDATION_BATCH_SIZE,
            "trainable_parameter_count": len(gradients),
            "raw_max_norm": float(
                raw_dataset_norms.max().detach().cpu().item()
            ),
            "raw_mean_norm": float(
                raw_dataset_norms.mean().detach().cpu().item()
            ),
            "clipped_max_norm": float(
                clipped_dataset_norms.max().detach().cpu().item()
            ),
            "clipped_mean_norm": float(
                clipped_dataset_norms.mean().detach().cpu().item()
            ),
            "minimum_clipping_factor": float(
                factors.min().detach().cpu().item()
            ),
            "maximum_clipping_factor": float(
                factors.max().detach().cpu().item()
            ),
            "raw_norms_finite": raw_finite,
            "clipping_factors_finite": factor_finite,
            "clipping_factors_bounded": factor_bounded,
            "clipped_gradients_finite": clipped_finite,
            "post_clipping_norm_bounded": bounded_after_clipping,
            "gradient_shapes_preserved": shape_preserved_dataset,
            "sample_count_preserved": sample_count_preserved,
            "status": "PASS",
        }
    )

    print(
        f"✓ {dataset_id:<20} : PASS | "
        f"input={input_dim:<4} | "
        f"parameters={len(gradients):<2} | "
        f"raw_max={raw_dataset_norms.max().item():.6f} | "
        f"clipped_max={clipped_dataset_norms.max().item():.6f}"
    )


# --------------------------------------------------------------------------------------------------
# 7. Build clipping validation dataframe
# --------------------------------------------------------------------------------------------------

GRADIENT_CLIPPING_VALIDATION_DF = pd.DataFrame(
    [
        {
            "validation": "raw_norm_finite",
            "status": "PASS" if raw_norms_finite else "FAIL",
        },
        {
            "validation": "clipping_factors_finite",
            "status": "PASS" if factors_finite else "FAIL",
        },
        {
            "validation": "clipping_factors_bounded",
            "status": "PASS" if factors_bounded else "FAIL",
        },
        {
            "validation": "zero_gradient_stability",
            "status": "PASS" if zero_gradient_preserved else "FAIL",
        },
        {
            "validation": "below_threshold_gradients_preserved",
            "status": "PASS" if unchanged else "FAIL",
        },
        {
            "validation": "above_threshold_gradients_clipped",
            "status": "PASS" if above_threshold_reduced else "FAIL",
        },
        {
            "validation": "post_clipping_norm_leq_C",
            "status": "PASS" if post_clip_bound else "FAIL",
        },
        {
            "validation": "mathematical_factor_definition",
            "status": "PASS" if factor_formula_match else "FAIL",
        },
        {
            "validation": "gradient_shape_preservation",
            "status": "PASS" if shape_preserved else "FAIL",
        },
        {
            "validation": "clipped_gradient_finiteness",
            "status": "PASS" if all_clipped_finite else "FAIL",
        },
    ]
)

# --------------------------------------------------------------------------------------------------
# 8. Persist dataset-specific validation
# --------------------------------------------------------------------------------------------------

NB10_ROOT = (
    Path("/content/drive/MyDrive/SPP_GAN_Research")
    / "results"
    / "notebooks"
    / "notebook_10"
)

VALIDATION_ROOT = NB10_ROOT / "validation"
METADATA_ROOT = NB10_ROOT / "metadata"

VALIDATION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

METADATA_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

CLIPPING_VALIDATION_PATH = (
    VALIDATION_ROOT
    / "gradient_clipping_validation.csv"
)

DATASET_CLIPPING_PATH = (
    VALIDATION_ROOT
    / "gradient_clipping_dataset_validation.csv"
)

GRADIENT_CLIPPING_VALIDATION_DF.to_csv(
    CLIPPING_VALIDATION_PATH,
    index=False,
)

DATASET_CLIPPING_VALIDATION_DF = pd.DataFrame(
    DATASET_CLIPPING_RESULTS
)

DATASET_CLIPPING_VALIDATION_DF.to_csv(
    DATASET_CLIPPING_PATH,
    index=False,
)

print("\n✓ Clipping validation artifacts persisted.")
print(
    f"  General validation : {CLIPPING_VALIDATION_PATH}"
)
print(
    f"  Dataset validation: {DATASET_CLIPPING_PATH}"
)


# --------------------------------------------------------------------------------------------------
# 9. Persist Section 7 manifest
# --------------------------------------------------------------------------------------------------

SECTION_07_MANIFEST = {
    "notebook": "10",
    "section": "07",
    "title": "Define Gradient Clipping",
    "status": "PASS",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "clipping": {
        "mechanism": "flat_per_example_l2",
        "maximum_gradient_norm": float(MAX_GRAD_NORM),
        "epsilon_stability": 1e-12,
        "same_factor_across_parameter_tensors": True,
        "noise_injection": False,
        "aggregation": False,
    },
    "validation": {
        "raw_norms_finite": raw_norms_finite,
        "clipping_factors_finite": factors_finite,
        "clipping_factors_bounded": factors_bounded,
        "zero_gradient_stability": zero_gradient_preserved,
        "below_threshold_preserved": unchanged,
        "above_threshold_clipped": above_threshold_reduced,
        "post_clipping_norm_bounded": post_clip_bound,
        "mathematical_factor_definition": factor_formula_match,
        "gradient_shape_preserved": shape_preserved,
        "clipped_gradient_finite": all_clipped_finite,
        "dataset_count": len(DATASET_CLIPPING_RESULTS),
        "all_dataset_validations_passed": all(
            row["status"] == "PASS"
            for row in DATASET_CLIPPING_RESULTS
        ),
    },
    "datasets": [
        row["dataset_id"]
        for row in DATASET_CLIPPING_RESULTS
    ],
    "artifacts": {
        "general_validation": str(
            CLIPPING_VALIDATION_PATH
        ),
        "dataset_validation": str(
            DATASET_CLIPPING_PATH
        ),
    },
    "privacy_boundary": {
        "per_example_gradients": "defined_in_section_06",
        "gradient_clipping": "defined_and_validated_in_section_07",
        "gaussian_noise": "deferred_to_section_08",
        "dp_sgd_update": "deferred_to_section_09",
        "privacy_accounting": "deferred_to_notebook_11",
        "end_to_end_privacy_claim": False,
    },
}

SECTION_07_MANIFEST_PATH = (
    METADATA_ROOT
    / "section_07_gradient_clipping_manifest.json"
)

with open(
    SECTION_07_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        SECTION_07_MANIFEST,
        f,
        indent=2,
        sort_keys=True,
    )

print(
    f"✓ Section 7 manifest persisted: "
    f"{SECTION_07_MANIFEST_PATH}"
)


# --------------------------------------------------------------------------------------------------
# 10. Final Section 7 verification
# --------------------------------------------------------------------------------------------------

GENERAL_VALIDATION_PASS = bool(
    (
        GRADIENT_CLIPPING_VALIDATION_DF["status"]
        == "PASS"
    ).all()
)

DATASET_VALIDATION_PASS = bool(
    (
        DATASET_CLIPPING_VALIDATION_DF["status"]
        == "PASS"
    ).all()
)

ARTIFACTS_EXIST = (
    CLIPPING_VALIDATION_PATH.exists()
    and DATASET_CLIPPING_PATH.exists()
    and SECTION_07_MANIFEST_PATH.exists()
)

SECTION_07_PASS = all(
    [
        GENERAL_VALIDATION_PASS,
        DATASET_VALIDATION_PASS,
        ARTIFACTS_EXIST,
    ]
)

if not SECTION_07_PASS:
    raise RuntimeError(
        "SECTION 7 FINAL VERIFICATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 11. Final output
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 7 FINAL VERIFICATION")
print("=" * 100)

print(
    "✓ Flat L2 norm computation                 : PASS"
)
print(
    "✓ Clipping factor computation              : PASS"
)
print(
    "✓ Above-threshold gradients clipped       : PASS"
)
print(
    "✓ Below-threshold gradients preserved     : PASS"
)
print(
    "✓ Zero-gradient stability                 : PASS"
)
print(
    "✓ Post-clipping norm ≤ C                  : PASS"
)
print(
    "✓ Finite clipping factors                 : PASS"
)
print(
    "✓ Per-example tensor shapes preserved     : PASS"
)
print(
    "✓ Clipped gradients finite                : PASS"
)
print(
    "✓ Adult Income                             : PASS"
)
print(
    "✓ Bank Marketing                           : PASS"
)
print(
    "✓ Diabetes 130US                           : PASS"
)

print("\n" + "-" * 100)
print("PRIVACY BOUNDARY")
print("-" * 100)

print(
    "Per-example gradients : defined in Section 6"
)
print(
    "Gradient clipping     : implemented in Section 7"
)
print(
    "Gaussian noise        : deferred to Section 8"
)
print(
    "DP-SGD update         : deferred to Section 9"
)
print(
    "Privacy accounting    : deferred to Notebook 11"
)
print(
    "End-to-end DP claim   : DISABLED"
)

print("\n" + "=" * 100)
print("SECTION 7 STATUS: PASS")
print("=" * 100)


7. DEFINE GRADIENT CLIPPING
✓ Required configuration validated.
✓ Flat L2 clipping threshold C = 1.000000

----------------------------------------------------------------------------------------------------
CLIPPING MATHEMATICAL VALIDATION
----------------------------------------------------------------------------------------------------
✓ Raw per-example gradient norms : PASS
✓ Clipping factors finite and bounded [0, 1] : PASS
✓ Zero-gradient stability : PASS
✓ Gradients at or below C preserved : PASS
✓ Above-threshold gradients clipped : PASS
✓ Post-clipping norm ≤ C : PASS
✓ Clipping factor mathematical definition : PASS
✓ Per-example gradient tensor shapes preserved : PASS
✓ Clipped gradients finite : PASS

----------------------------------------------------------------------------------------------------
DATASET-SPECIFIC CLIPPING VALIDATION
----------------------------------------------------------------------------------------------------
✓ adult_income         : PASS | input

In [8]:
# ==================================================================================================
# 8. DEFINE GAUSSIAN NOISE MECHANISM
# ==================================================================================================

print("\n" + "=" * 100)
print("8. DEFINE GAUSSIAN NOISE MECHANISM")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 0. Imports
# --------------------------------------------------------------------------------------------------

from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd
import torch


# --------------------------------------------------------------------------------------------------
# 1. Validate Section 5 dependency
# --------------------------------------------------------------------------------------------------

if "PRIVACY_PARAMETER_DF" not in globals():
    raise RuntimeError(
        "PRIVACY_PARAMETER_DF is not defined. "
        "Run Notebook 10 Section 5 first."
    )

if not isinstance(
    PRIVACY_PARAMETER_DF,
    pd.DataFrame,
):
    raise TypeError(
        "PRIVACY_PARAMETER_DF must be a pandas DataFrame."
    )

if "DATASET_IDS" not in globals():
    raise RuntimeError(
        "DATASET_IDS is not defined. "
        "Run Notebook 10 Section 2 first."
    )

if "MAX_GRAD_NORM" not in globals():
    raise RuntimeError(
        "MAX_GRAD_NORM is not defined. "
        "Run Notebook 10 Section 4 first."
    )


# --------------------------------------------------------------------------------------------------
# 2. Validate Section 5 schema
# --------------------------------------------------------------------------------------------------

REQUIRED_SECTION_05_COLUMNS = {
    "dataset",
    "n_train",
    "target_epsilon",
    "delta",
    "batch_size_nominal",
    "poisson_sample_rate",
    "epochs",
    "max_grad_norm",
    "noise_multiplier",
    "calibration_epsilon",
    "rdp_alpha_min",
    "rdp_alpha_max",
    "rdp_alpha_count",
    "optimal_alpha",
    "calibration_method",
    "accountant",
    "sampling",
    "clipping",
    "loss_reduction",
    "calibration_status",
    "achieved_epsilon_status",
}

missing_section_05_columns = (
    REQUIRED_SECTION_05_COLUMNS
    -
    set(PRIVACY_PARAMETER_DF.columns)
)

if missing_section_05_columns:
    raise RuntimeError(
        "Section 5 output is missing required columns: "
        f"{sorted(missing_section_05_columns)}"
    )


# --------------------------------------------------------------------------------------------------
# 3. Validate dataset coverage
# --------------------------------------------------------------------------------------------------

EXPECTED_DATASETS = {
    str(dataset_id)
    for dataset_id in DATASET_IDS
}

SECTION_05_DATASETS = {
    str(dataset_id)
    for dataset_id in PRIVACY_PARAMETER_DF["dataset"]
}

if SECTION_05_DATASETS != EXPECTED_DATASETS:
    raise RuntimeError(
        "Section 5 dataset coverage mismatch. "
        f"Expected={sorted(EXPECTED_DATASETS)}, "
        f"Received={sorted(SECTION_05_DATASETS)}"
    )

if len(PRIVACY_PARAMETER_DF) != len(
    DATASET_IDS
):
    raise RuntimeError(
        "Section 5 must contain exactly one "
        "privacy-calibration row per dataset."
    )


# --------------------------------------------------------------------------------------------------
# 4. Validate calibration status
# --------------------------------------------------------------------------------------------------

if not (
    PRIVACY_PARAMETER_DF[
        "calibration_status"
    ]
    .astype(str)
    .eq("PASS")
    .all()
):
    raise RuntimeError(
        "One or more Section 5 privacy calibrations did not pass."
    )

if not (
    PRIVACY_PARAMETER_DF[
        "achieved_epsilon_status"
    ]
    .astype(str)
    .eq("DEFERRED_TO_NOTEBOOK_11")
    .all()
):
    raise RuntimeError(
        "Section 5 achieved-epsilon provenance is inconsistent."
    )

print(
    "✓ Section 5 privacy-calibration table loaded."
)

print(
    f"✓ Dataset coverage : {len(DATASET_IDS)} datasets"
)

print(
    "✓ Calibration status : PASS"
)

print(
    "✓ Achieved epsilon remains deferred to Notebook 11."
)


# --------------------------------------------------------------------------------------------------
# 5. Validate calibrated noise multipliers
# --------------------------------------------------------------------------------------------------

CALIBRATED_NOISE_MULTIPLIERS = {}

for _, row in PRIVACY_PARAMETER_DF.iterrows():

    dataset_id = str(
        row["dataset"]
    )

    noise_multiplier = float(
        row["noise_multiplier"]
    )

    row_clip_norm = float(
        row["max_grad_norm"]
    )

    if not np.isfinite(
        noise_multiplier
    ):
        raise ValueError(
            f"{dataset_id}: noise multiplier is not finite."
        )

    if noise_multiplier <= 0:
        raise ValueError(
            f"{dataset_id}: noise multiplier must be positive."
        )

    if not np.isfinite(
        row_clip_norm
    ):
        raise ValueError(
            f"{dataset_id}: max_grad_norm is not finite."
        )

    if row_clip_norm <= 0:
        raise ValueError(
            f"{dataset_id}: max_grad_norm must be positive."
        )

    CALIBRATED_NOISE_MULTIPLIERS[
        dataset_id
    ] = noise_multiplier


print(
    "✓ Calibrated Gaussian noise multipliers validated."
)


# --------------------------------------------------------------------------------------------------
# 6. Validate global clipping threshold consistency
# --------------------------------------------------------------------------------------------------

GLOBAL_MAX_GRAD_NORM = float(
    MAX_GRAD_NORM
)

if not np.isfinite(
    GLOBAL_MAX_GRAD_NORM
):
    raise ValueError(
        "MAX_GRAD_NORM must be finite."
    )

if GLOBAL_MAX_GRAD_NORM <= 0:
    raise ValueError(
        "MAX_GRAD_NORM must be positive."
    )

SECTION_05_CLIP_VALUES = pd.to_numeric(
    PRIVACY_PARAMETER_DF[
        "max_grad_norm"
    ],
    errors="coerce",
).to_numpy(
    dtype=float
)

if not np.all(
    np.isfinite(
        SECTION_05_CLIP_VALUES
    )
):
    raise RuntimeError(
        "Section 5 contains non-finite clipping thresholds."
    )

if not np.allclose(
    SECTION_05_CLIP_VALUES,
    GLOBAL_MAX_GRAD_NORM,
    rtol=0.0,
    atol=1e-12,
):
    raise RuntimeError(
        "Section 5 clipping thresholds do not match "
        "Notebook 10 MAX_GRAD_NORM."
    )

print(
    f"✓ Maximum gradient norm C = "
    f"{GLOBAL_MAX_GRAD_NORM:.6f}"
)

print(
    "✓ Section 5 clipping threshold consistency : PASS"
)


# --------------------------------------------------------------------------------------------------
# 7. Define Gaussian noise mechanism
# --------------------------------------------------------------------------------------------------

def gaussian_noise(
    reference_tensor,
    noise_multiplier,
    max_grad_norm,
    generator=None,
):
    """
    Generate Gaussian noise:

        noise ~ N(0, (sigma * C)^2)

    where:

        sigma = noise_multiplier
        C     = max_grad_norm

    The reference tensor defines the output shape,
    device, and floating-point dtype.
    """

    if not isinstance(
        reference_tensor,
        torch.Tensor,
    ):
        raise TypeError(
            "reference_tensor must be a torch.Tensor."
        )

    if not torch.is_floating_point(
        reference_tensor
    ):
        raise TypeError(
            "Reference tensor must be floating point."
        )

    noise_multiplier = float(
        noise_multiplier
    )

    max_grad_norm = float(
        max_grad_norm
    )

    if not np.isfinite(
        noise_multiplier
    ):
        raise ValueError(
            "noise_multiplier must be finite."
        )

    if noise_multiplier <= 0:
        raise ValueError(
            "noise_multiplier must be positive."
        )

    if not np.isfinite(
        max_grad_norm
    ):
        raise ValueError(
            "max_grad_norm must be finite."
        )

    if max_grad_norm <= 0:
        raise ValueError(
            "max_grad_norm must be positive."
        )

    noise_std = (
        noise_multiplier
        *
        max_grad_norm
    )

    if not np.isfinite(
        noise_std
    ):
        raise ValueError(
            "Gaussian noise standard deviation is not finite."
        )

    noise = torch.normal(
        mean=0.0,
        std=noise_std,
        size=reference_tensor.shape,
        device=reference_tensor.device,
        dtype=reference_tensor.dtype,
        generator=generator,
    )

    if not torch.isfinite(
        noise
    ).all():
        raise RuntimeError(
            "Generated Gaussian noise contains non-finite values."
        )

    return noise


# --------------------------------------------------------------------------------------------------
# 8. Define additive Gaussian noise helper
# --------------------------------------------------------------------------------------------------

def add_gaussian_noise(
    gradient,
    noise_multiplier,
    max_grad_norm,
    generator=None,
):
    """
    Add Gaussian noise to a gradient tensor.

    This function defines the Gaussian noise primitive only.

    Production DP-SGD optimizer behavior is delegated
    to Opacus in Section 9.
    """

    if not isinstance(
        gradient,
        torch.Tensor,
    ):
        raise TypeError(
            "gradient must be a torch.Tensor."
        )

    if not torch.is_floating_point(
        gradient
    ):
        raise TypeError(
            "gradient must be floating point."
        )

    noise = gaussian_noise(
        reference_tensor=gradient,
        noise_multiplier=noise_multiplier,
        max_grad_norm=max_grad_norm,
        generator=generator,
    )

    noisy_gradient = (
        gradient
        +
        noise
    )

    if noisy_gradient.shape != gradient.shape:
        raise RuntimeError(
            "Gradient shape changed during Gaussian noise addition."
        )

    if not torch.isfinite(
        noisy_gradient
    ).all():
        raise RuntimeError(
            "Noisy gradient contains non-finite values."
        )

    return noisy_gradient


print(
    "✓ Gaussian noise mechanism defined."
)

print(
    "✓ Noise standard deviation = σ × C"
)

print(
    "✓ Production DP update delegated to Opacus DPOptimizer."
)


# --------------------------------------------------------------------------------------------------
# 9. General Gaussian mechanism validation
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("GAUSSIAN MECHANISM MATHEMATICAL VALIDATION")
print("-" * 100)


VALIDATION_DEVICE = (
    DEVICE
    if "DEVICE" in globals()
    else torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )
)

VALIDATION_DTYPE = torch.float32

VALIDATION_SEED = (
    int(MASTER_SEED) + 8000
    if "MASTER_SEED" in globals()
    else 10025
)

VALIDATION_SAMPLE_COUNT = 20000

VALIDATION_NOISE_MULTIPLIER = 1.25

VALIDATION_GENERATOR = torch.Generator(
    device=VALIDATION_DEVICE
)

VALIDATION_GENERATOR.manual_seed(
    VALIDATION_SEED
)

REFERENCE_TENSOR = torch.zeros(
    VALIDATION_SAMPLE_COUNT,
    dtype=VALIDATION_DTYPE,
    device=VALIDATION_DEVICE,
)

EXPECTED_NOISE_STD = (
    VALIDATION_NOISE_MULTIPLIER
    *
    GLOBAL_MAX_GRAD_NORM
)

NOISE = gaussian_noise(
    reference_tensor=REFERENCE_TENSOR,
    noise_multiplier=VALIDATION_NOISE_MULTIPLIER,
    max_grad_norm=GLOBAL_MAX_GRAD_NORM,
    generator=VALIDATION_GENERATOR,
)


# --------------------------------------------------------------------------------------------------
# 9.1 Shape validation
# --------------------------------------------------------------------------------------------------

NOISE_SHAPE_VALID = bool(
    NOISE.shape
    ==
    REFERENCE_TENSOR.shape
)

if not NOISE_SHAPE_VALID:
    raise RuntimeError(
        "Gaussian noise shape does not match reference tensor."
    )

print(
    "✓ Noise shape preservation : PASS"
)


# --------------------------------------------------------------------------------------------------
# 9.2 Device validation
# --------------------------------------------------------------------------------------------------

NOISE_DEVICE_VALID = bool(
    NOISE.device
    ==
    REFERENCE_TENSOR.device
)

if not NOISE_DEVICE_VALID:
    raise RuntimeError(
        "Gaussian noise device does not match reference tensor."
    )

print(
    "✓ Noise device preservation : PASS"
)


# --------------------------------------------------------------------------------------------------
# 9.3 Dtype validation
# --------------------------------------------------------------------------------------------------

NOISE_DTYPE_VALID = bool(
    NOISE.dtype
    ==
    REFERENCE_TENSOR.dtype
)

if not NOISE_DTYPE_VALID:
    raise RuntimeError(
        "Gaussian noise dtype does not match reference tensor."
    )

print(
    "✓ Noise dtype preservation : PASS"
)


# --------------------------------------------------------------------------------------------------
# 9.4 Finiteness validation
# --------------------------------------------------------------------------------------------------

NOISE_FINITE = bool(
    torch.isfinite(
        NOISE
    ).all().item()
)

if not NOISE_FINITE:
    raise RuntimeError(
        "Gaussian noise contains non-finite values."
    )

print(
    "✓ Gaussian noise finiteness : PASS"
)


# --------------------------------------------------------------------------------------------------
# 9.5 Empirical mean validation
# --------------------------------------------------------------------------------------------------

OBSERVED_MEAN = float(
    NOISE.mean()
    .detach()
    .cpu()
    .item()
)

MEAN_TOLERANCE = float(
    max(
        0.05,
        6.0
        *
        EXPECTED_NOISE_STD
        /
        np.sqrt(
            VALIDATION_SAMPLE_COUNT
        ),
    )
)

NOISE_MEAN_VALID = bool(
    abs(OBSERVED_MEAN)
    <=
    MEAN_TOLERANCE
)

if not NOISE_MEAN_VALID:
    raise RuntimeError(
        "Gaussian noise empirical mean is outside tolerance. "
        f"Observed={OBSERVED_MEAN:.6f}, "
        f"Tolerance={MEAN_TOLERANCE:.6f}."
    )

print(
    f"✓ Gaussian noise empirical mean ≈ 0 : PASS "
    f"(mean={OBSERVED_MEAN:.6f})"
)


# --------------------------------------------------------------------------------------------------
# 9.6 Empirical standard-deviation validation
# --------------------------------------------------------------------------------------------------

OBSERVED_STD = float(
    NOISE.std(
        unbiased=True
    )
    .detach()
    .cpu()
    .item()
)

STD_RELATIVE_TOLERANCE = 0.05

STD_LOWER = float(
    EXPECTED_NOISE_STD
    *
    0.95
)

STD_UPPER = float(
    EXPECTED_NOISE_STD
    *
    1.05
)

NOISE_STD_VALID = bool(
    STD_LOWER
    <=
    OBSERVED_STD
    <=
    STD_UPPER
)

if not NOISE_STD_VALID:
    raise RuntimeError(
        "Gaussian noise empirical standard deviation is outside "
        "the validation interval. "
        f"Expected={EXPECTED_NOISE_STD:.6f}, "
        f"Observed={OBSERVED_STD:.6f}, "
        f"Interval=[{STD_LOWER:.6f}, {STD_UPPER:.6f}]."
    )

print(
    f"✓ Gaussian noise standard deviation : PASS "
    f"(expected={EXPECTED_NOISE_STD:.6f}, "
    f"observed={OBSERVED_STD:.6f})"
)


# --------------------------------------------------------------------------------------------------
# 9.7 Independent random-draw validation
# --------------------------------------------------------------------------------------------------

GENERATOR_A = torch.Generator(
    device=VALIDATION_DEVICE
)

GENERATOR_B = torch.Generator(
    device=VALIDATION_DEVICE
)

GENERATOR_A.manual_seed(
    VALIDATION_SEED + 1
)

GENERATOR_B.manual_seed(
    VALIDATION_SEED + 2
)

NOISE_A = gaussian_noise(
    reference_tensor=REFERENCE_TENSOR,
    noise_multiplier=VALIDATION_NOISE_MULTIPLIER,
    max_grad_norm=GLOBAL_MAX_GRAD_NORM,
    generator=GENERATOR_A,
)

NOISE_B = gaussian_noise(
    reference_tensor=REFERENCE_TENSOR,
    noise_multiplier=VALIDATION_NOISE_MULTIPLIER,
    max_grad_norm=GLOBAL_MAX_GRAD_NORM,
    generator=GENERATOR_B,
)

INDEPENDENT_DRAWS_VALID = bool(
    not torch.equal(
        NOISE_A,
        NOISE_B,
    )
)

if not INDEPENDENT_DRAWS_VALID:
    raise RuntimeError(
        "Independent Gaussian noise draws were identical."
    )

print(
    "✓ Independent random draws differ : PASS"
)


# --------------------------------------------------------------------------------------------------
# 9.8 Zero-gradient validation
# --------------------------------------------------------------------------------------------------

ZERO_GRADIENT = torch.zeros(
    1000,
    dtype=VALIDATION_DTYPE,
    device=VALIDATION_DEVICE,
)

ZERO_GENERATOR = torch.Generator(
    device=VALIDATION_DEVICE
)

ZERO_GENERATOR.manual_seed(
    VALIDATION_SEED + 3
)

NOISY_ZERO_GRADIENT = add_gaussian_noise(
    gradient=ZERO_GRADIENT,
    noise_multiplier=VALIDATION_NOISE_MULTIPLIER,
    max_grad_norm=GLOBAL_MAX_GRAD_NORM,
    generator=ZERO_GENERATOR,
)

ZERO_NOISE_FINITE = bool(
    torch.isfinite(
        NOISY_ZERO_GRADIENT
    ).all().item()
)

ZERO_NOISE_NONZERO = bool(
    torch.any(
        NOISY_ZERO_GRADIENT != 0
    ).item()
)

if not ZERO_NOISE_FINITE:
    raise RuntimeError(
        "Noisy zero gradient contains non-finite values."
    )

if not ZERO_NOISE_NONZERO:
    raise RuntimeError(
        "Zero gradient did not receive Gaussian noise."
    )

print(
    "✓ Zero gradient receives finite Gaussian noise : PASS"
)


# --------------------------------------------------------------------------------------------------
# 9.9 Additive consistency validation
# --------------------------------------------------------------------------------------------------

BASE_GRADIENT = torch.ones(
    1000,
    dtype=VALIDATION_DTYPE,
    device=VALIDATION_DEVICE,
)

ADDITIVE_SEED = (
    VALIDATION_SEED + 4
)

EXPECTED_GENERATOR = torch.Generator(
    device=VALIDATION_DEVICE
)

EXPECTED_GENERATOR.manual_seed(
    ADDITIVE_SEED
)

EXPECTED_NOISE = gaussian_noise(
    reference_tensor=BASE_GRADIENT,
    noise_multiplier=VALIDATION_NOISE_MULTIPLIER,
    max_grad_norm=GLOBAL_MAX_GRAD_NORM,
    generator=EXPECTED_GENERATOR,
)

ACTUAL_GENERATOR = torch.Generator(
    device=VALIDATION_DEVICE
)

ACTUAL_GENERATOR.manual_seed(
    ADDITIVE_SEED
)

NOISY_GRADIENT = add_gaussian_noise(
    gradient=BASE_GRADIENT,
    noise_multiplier=VALIDATION_NOISE_MULTIPLIER,
    max_grad_norm=GLOBAL_MAX_GRAD_NORM,
    generator=ACTUAL_GENERATOR,
)

ADDITIVE_CONSISTENCY_VALID = bool(
    torch.allclose(
        NOISY_GRADIENT,
        BASE_GRADIENT + EXPECTED_NOISE,
        atol=1e-6,
        rtol=1e-6,
    )
)

if not ADDITIVE_CONSISTENCY_VALID:
    raise RuntimeError(
        "Additive Gaussian-noise consistency validation failed."
    )

print(
    "✓ Additive Gaussian noise consistency : PASS"
)


# --------------------------------------------------------------------------------------------------
# 10. Dataset-specific calibrated noise validation
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("DATASET-SPECIFIC CALIBRATED NOISE VALIDATION")
print("-" * 100)


GAUSSIAN_NOISE_DATASET_RESULTS = []

DATASET_VALIDATION_SAMPLE_COUNT = 20000


for dataset_index, dataset_id in enumerate(
    DATASET_IDS
):

    dataset_id = str(
        dataset_id
    )

    noise_multiplier = float(
        CALIBRATED_NOISE_MULTIPLIERS[
            dataset_id
        ]
    )

    expected_std = float(
        noise_multiplier
        *
        GLOBAL_MAX_GRAD_NORM
    )

    dataset_seed = (
        int(MASTER_SEED)
        + 8000
        + dataset_index
        if "MASTER_SEED" in globals()
        else 10025 + dataset_index
    )

    dataset_generator = torch.Generator(
        device=VALIDATION_DEVICE
    )

    dataset_generator.manual_seed(
        dataset_seed
    )

    dataset_reference = torch.zeros(
        DATASET_VALIDATION_SAMPLE_COUNT,
        dtype=VALIDATION_DTYPE,
        device=VALIDATION_DEVICE,
    )

    dataset_noise = gaussian_noise(
        reference_tensor=dataset_reference,
        noise_multiplier=noise_multiplier,
        max_grad_norm=GLOBAL_MAX_GRAD_NORM,
        generator=dataset_generator,
    )

    dataset_observed_mean = float(
        dataset_noise.mean()
        .detach()
        .cpu()
        .item()
    )

    dataset_observed_std = float(
        dataset_noise.std(
            unbiased=True
        )
        .detach()
        .cpu()
        .item()
    )

    dataset_mean_tolerance = float(
        max(
            0.05,
            6.0
            *
            expected_std
            /
            np.sqrt(
                DATASET_VALIDATION_SAMPLE_COUNT
            ),
        )
    )

    dataset_std_lower = float(
        expected_std
        *
        0.95
    )

    dataset_std_upper = float(
        expected_std
        *
        1.05
    )

    dataset_shape_valid = bool(
        dataset_noise.shape
        ==
        dataset_reference.shape
    )

    dataset_finite = bool(
        torch.isfinite(
            dataset_noise
        ).all().item()
    )

    dataset_mean_valid = bool(
        abs(dataset_observed_mean)
        <=
        dataset_mean_tolerance
    )

    dataset_std_valid = bool(
        dataset_std_lower
        <=
        dataset_observed_std
        <=
        dataset_std_upper
    )

    dataset_status = bool(
        all(
            [
                dataset_shape_valid,
                dataset_finite,
                dataset_mean_valid,
                dataset_std_valid,
            ]
        )
    )

    if not dataset_status:
        raise RuntimeError(
            f"Gaussian noise validation failed for '{dataset_id}'. "
            f"Expected std={expected_std:.6f}, "
            f"observed std={dataset_observed_std:.6f}."
        )

    GAUSSIAN_NOISE_DATASET_RESULTS.append(
        {
            "dataset_id": dataset_id,
            "noise_multiplier": float(
                noise_multiplier
            ),
            "max_grad_norm": float(
                GLOBAL_MAX_GRAD_NORM
            ),
            "expected_noise_std": float(
                expected_std
            ),
            "observed_noise_mean": float(
                dataset_observed_mean
            ),
            "observed_noise_std": float(
                dataset_observed_std
            ),
            "mean_tolerance": float(
                dataset_mean_tolerance
            ),
            "std_lower_bound": float(
                dataset_std_lower
            ),
            "std_upper_bound": float(
                dataset_std_upper
            ),
            "validation_sample_count": int(
                DATASET_VALIDATION_SAMPLE_COUNT
            ),
            "shape": str(
                tuple(dataset_noise.shape)
            ),
            "shape_valid": bool(
                dataset_shape_valid
            ),
            "finite": bool(
                dataset_finite
            ),
            "mean_valid": bool(
                dataset_mean_valid
            ),
            "std_valid": bool(
                dataset_std_valid
            ),
            "status": "PASS",
        }
    )

    print(
        f"✓ {dataset_id:<20} : PASS | "
        f"σ={noise_multiplier:.6f} | "
        f"expected_std={expected_std:.6f} | "
        f"observed_std={dataset_observed_std:.6f}"
    )


# --------------------------------------------------------------------------------------------------
# 11. Build validation tables
# --------------------------------------------------------------------------------------------------

GENERAL_GAUSSIAN_VALIDATION_DF = pd.DataFrame(
    [
        {
            "validation": "noise_shape",
            "status": (
                "PASS"
                if NOISE_SHAPE_VALID
                else "FAIL"
            ),
        },
        {
            "validation": "device_preservation",
            "status": (
                "PASS"
                if NOISE_DEVICE_VALID
                else "FAIL"
            ),
        },
        {
            "validation": "dtype_preservation",
            "status": (
                "PASS"
                if NOISE_DTYPE_VALID
                else "FAIL"
            ),
        },
        {
            "validation": "noise_finiteness",
            "status": (
                "PASS"
                if NOISE_FINITE
                else "FAIL"
            ),
        },
        {
            "validation": "empirical_mean_near_zero",
            "status": (
                "PASS"
                if NOISE_MEAN_VALID
                else "FAIL"
            ),
        },
        {
            "validation": "empirical_std_matches_expected",
            "status": (
                "PASS"
                if NOISE_STD_VALID
                else "FAIL"
            ),
        },
        {
            "validation": "independent_random_draws",
            "status": (
                "PASS"
                if INDEPENDENT_DRAWS_VALID
                else "FAIL"
            ),
        },
        {
            "validation": "zero_gradient_noise",
            "status": (
                "PASS"
                if (
                    ZERO_NOISE_FINITE
                    and ZERO_NOISE_NONZERO
                )
                else "FAIL"
            ),
        },
        {
            "validation": "additive_noise_consistency",
            "status": (
                "PASS"
                if ADDITIVE_CONSISTENCY_VALID
                else "FAIL"
            ),
        },
    ]
)


DATASET_GAUSSIAN_VALIDATION_DF = pd.DataFrame(
    GAUSSIAN_NOISE_DATASET_RESULTS
)


# --------------------------------------------------------------------------------------------------
# 12. Prepare canonical Notebook 10 directories
# --------------------------------------------------------------------------------------------------

NB10_ROOT = (
    Path("/content/drive/MyDrive/SPP_GAN_Research")
    /
    "results"
    /
    "notebooks"
    /
    "notebook_10"
)

VALIDATION_ROOT = (
    NB10_ROOT
    /
    "validation"
)

METADATA_ROOT = (
    NB10_ROOT
    /
    "metadata"
)

VALIDATION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

METADATA_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------------------------------------------------------
# 13. Persist validation artifacts
# --------------------------------------------------------------------------------------------------

GENERAL_GAUSSIAN_VALIDATION_PATH = (
    VALIDATION_ROOT
    /
    "gaussian_noise_validation.csv"
)

DATASET_GAUSSIAN_VALIDATION_PATH = (
    VALIDATION_ROOT
    /
    "gaussian_noise_dataset_validation.csv"
)

GENERAL_GAUSSIAN_VALIDATION_DF.to_csv(
    GENERAL_GAUSSIAN_VALIDATION_PATH,
    index=False,
)

DATASET_GAUSSIAN_VALIDATION_DF.to_csv(
    DATASET_GAUSSIAN_VALIDATION_PATH,
    index=False,
)

print(
    "\n✓ Gaussian noise validation artifacts persisted."
)

print(
    f"  General validation : "
    f"{GENERAL_GAUSSIAN_VALIDATION_PATH}"
)

print(
    f"  Dataset validation : "
    f"{DATASET_GAUSSIAN_VALIDATION_PATH}"
)


# --------------------------------------------------------------------------------------------------
# 14. Build Section 8 manifest
# --------------------------------------------------------------------------------------------------

SECTION_08_MANIFEST = {
    "notebook": "10",
    "section": "08",
    "title": "Define Gaussian Noise Mechanism",
    "status": "PASS",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source_section": {
        "notebook": "10",
        "section": "05",
        "source_dataframe": "PRIVACY_PARAMETER_DF",
        "noise_multiplier_column": "noise_multiplier",
    },

    "mechanism": {
        "type": "Gaussian",
        "distribution": "Normal(0, (sigma*C)^2)",
        "noise_multiplier_symbol": "sigma",
        "clipping_threshold_symbol": "C",
        "maximum_gradient_norm": float(
            GLOBAL_MAX_GRAD_NORM
        ),
        "noise_standard_deviation": (
            "noise_multiplier * max_grad_norm"
        ),
        "additive": True,
    },

    "calibrated_noise_multipliers": {
        str(dataset_id): float(
            noise_multiplier
        )
        for dataset_id, noise_multiplier
        in CALIBRATED_NOISE_MULTIPLIERS.items()
    },

    "validation": {
        "noise_shape": bool(
            NOISE_SHAPE_VALID
        ),
        "device_preservation": bool(
            NOISE_DEVICE_VALID
        ),
        "dtype_preservation": bool(
            NOISE_DTYPE_VALID
        ),
        "noise_finiteness": bool(
            NOISE_FINITE
        ),
        "empirical_mean_near_zero": bool(
            NOISE_MEAN_VALID
        ),
        "empirical_std_matches_expected": bool(
            NOISE_STD_VALID
        ),
        "independent_random_draws": bool(
            INDEPENDENT_DRAWS_VALID
        ),
        "zero_gradient_noise": bool(
            ZERO_NOISE_FINITE
            and ZERO_NOISE_NONZERO
        ),
        "additive_noise_consistency": bool(
            ADDITIVE_CONSISTENCY_VALID
        ),
        "dataset_count": int(
            len(
                GAUSSIAN_NOISE_DATASET_RESULTS
            )
        ),
        "all_dataset_validations_passed": bool(
            all(
                row["status"] == "PASS"
                for row
                in GAUSSIAN_NOISE_DATASET_RESULTS
            )
        ),
    },

    "artifacts": {
        "general_validation": str(
            GENERAL_GAUSSIAN_VALIDATION_PATH
        ),
        "dataset_validation": str(
            DATASET_GAUSSIAN_VALIDATION_PATH
        ),
    },

    "production_boundary": {
        "section_role": (
            "Gaussian noise primitive definition and validation"
        ),
        "production_optimizer": (
            "Opacus DPOptimizer"
        ),
        "production_dp_update": (
            "deferred_to_section_09"
        ),
        "privacy_accounting": (
            "deferred_to_notebook_11"
        ),
        "achieved_epsilon": (
            "deferred_to_notebook_11"
        ),
        "end_to_end_privacy_claim": False,
    },
}


# --------------------------------------------------------------------------------------------------
# 15. Explicit JSON serializability validation
# --------------------------------------------------------------------------------------------------

try:

    json.dumps(
        SECTION_08_MANIFEST,
        indent=2,
        sort_keys=True,
    )

except (
    TypeError,
    ValueError,
) as exc:

    raise RuntimeError(
        "Section 8 manifest failed JSON serializability validation."
    ) from exc

print(
    "✓ Section 8 manifest JSON serializability : PASS"
)


# --------------------------------------------------------------------------------------------------
# 16. Persist Section 8 manifest
# --------------------------------------------------------------------------------------------------

SECTION_08_MANIFEST_PATH = (
    METADATA_ROOT
    /
    "section_08_gaussian_noise_manifest.json"
)

with open(
    SECTION_08_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        SECTION_08_MANIFEST,
        f,
        indent=2,
        sort_keys=True,
    )

print(
    f"✓ Section 8 manifest persisted: "
    f"{SECTION_08_MANIFEST_PATH}"
)


# --------------------------------------------------------------------------------------------------
# 17. Final verification
# --------------------------------------------------------------------------------------------------

GENERAL_VALIDATION_PASS = bool(
    (
        GENERAL_GAUSSIAN_VALIDATION_DF[
            "status"
        ]
        ==
        "PASS"
    ).all()
)

DATASET_VALIDATION_PASS = bool(
    (
        DATASET_GAUSSIAN_VALIDATION_DF[
            "status"
        ]
        ==
        "PASS"
    ).all()
)

ARTIFACTS_EXIST = bool(
    GENERAL_GAUSSIAN_VALIDATION_PATH.exists()
    and
    DATASET_GAUSSIAN_VALIDATION_PATH.exists()
    and
    SECTION_08_MANIFEST_PATH.exists()
)

SECTION_08_PASS = bool(
    GENERAL_VALIDATION_PASS
    and
    DATASET_VALIDATION_PASS
    and
    ARTIFACTS_EXIST
)

if not SECTION_08_PASS:
    raise RuntimeError(
        "SECTION 8 FINAL VERIFICATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 18. Final output
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 8 FINAL VERIFICATION")
print("=" * 100)

print(
    "✓ Section 5 calibration dependency        : PASS"
)

print(
    "✓ Gaussian noise generation               : PASS"
)

print(
    "✓ Noise standard deviation σ × C         : PASS"
)

print(
    "✓ Noise shape preservation                : PASS"
)

print(
    "✓ Noise device preservation               : PASS"
)

print(
    "✓ Noise dtype preservation                : PASS"
)

print(
    "✓ Gaussian noise finiteness               : PASS"
)

print(
    "✓ Empirical mean approximately zero       : PASS"
)

print(
    "✓ Empirical standard deviation validated  : PASS"
)

print(
    "✓ Independent random draws                : PASS"
)

print(
    "✓ Zero-gradient noise injection           : PASS"
)

print(
    "✓ Additive noise consistency              : PASS"
)

print(
    "✓ Adult Income calibrated noise           : PASS"
)

print(
    "✓ Bank Marketing calibrated noise         : PASS"
)

print(
    "✓ Diabetes 130US calibrated noise         : PASS"
)

print(
    "✓ Validation artifacts persisted           : PASS"
)

print(
    "✓ Section 8 manifest JSON serializability  : PASS"
)

print(
    "✓ Section 8 manifest persisted             : PASS"
)


# --------------------------------------------------------------------------------------------------
# 19. Privacy boundary
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("PRIVACY BOUNDARY")
print("-" * 100)

print(
    "Per-example gradients : defined in Section 6"
)

print(
    "Gradient clipping     : implemented in Section 7"
)

print(
    "Gaussian noise        : implemented and validated in Section 8"
)

print(
    "DP-SGD update         : delegated to Opacus / Section 9"
)

print(
    "Privacy accounting    : deferred to Notebook 11"
)

print(
    "Achieved epsilon      : deferred to Notebook 11"
)

print(
    "End-to-end DP claim   : DISABLED"
)

print("\n" + "=" * 100)
print("SECTION 8 STATUS: PASS")
print("=" * 100)


8. DEFINE GAUSSIAN NOISE MECHANISM
✓ Section 5 privacy-calibration table loaded.
✓ Dataset coverage : 3 datasets
✓ Calibration status : PASS
✓ Achieved epsilon remains deferred to Notebook 11.
✓ Calibrated Gaussian noise multipliers validated.
✓ Maximum gradient norm C = 1.000000
✓ Section 5 clipping threshold consistency : PASS
✓ Gaussian noise mechanism defined.
✓ Noise standard deviation = σ × C
✓ Production DP update delegated to Opacus DPOptimizer.

----------------------------------------------------------------------------------------------------
GAUSSIAN MECHANISM MATHEMATICAL VALIDATION
----------------------------------------------------------------------------------------------------
✓ Noise shape preservation : PASS
✓ Noise device preservation : PASS
✓ Noise dtype preservation : PASS
✓ Gaussian noise finiteness : PASS
✓ Gaussian noise empirical mean ≈ 0 : PASS (mean=0.022756)
✓ Gaussian noise standard deviation : PASS (expected=1.250000, observed=1.253565)
✓ Independent ra

In [11]:
# ==================================================================================================
# 9. DEFINE DP-SGD DISCRIMINATOR UPDATE
# ==================================================================================================

print("\n" + "=" * 100)
print("9. DEFINE DP-SGD DISCRIMINATOR UPDATE")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Validate required dependencies
# --------------------------------------------------------------------------------------------------

REQUIRED_SECTION_09_OBJECTS = [
    "PrivacyEngine",
    "SPPGANCritic",
    "CRITIC_INPUT_DIMENSIONS",
    "DATASET_IDS",
    "MAX_GRAD_NORM",
    "DP_BATCH_SIZE",
    "DP_EPOCHS",
    "ACCOUNTANT",
    "SAMPLING_MECHANISM",
    "CLIPPING_MECHANISM",
    "LOSS_REDUCTION",
    "GRAD_SAMPLE_MODE",
    "CALIBRATED_NOISE_MULTIPLIERS",
]

missing_objects = [
    name
    for name in REQUIRED_SECTION_09_OBJECTS
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Section 9 is missing required objects: "
        f"{missing_objects}"
    )


# --------------------------------------------------------------------------------------------------
# 2. Validate privacy configuration
# --------------------------------------------------------------------------------------------------

if str(ACCOUNTANT).lower() != "rdp":
    raise RuntimeError(
        f"Section 9 requires RDP accountant. "
        f"Received: {ACCOUNTANT}"
    )

if str(SAMPLING_MECHANISM).lower() != "poisson":
    raise RuntimeError(
        f"Section 9 requires Poisson sampling. "
        f"Received: {SAMPLING_MECHANISM}"
    )

if str(CLIPPING_MECHANISM).lower() != "flat":
    raise RuntimeError(
        f"Section 9 requires flat clipping. "
        f"Received: {CLIPPING_MECHANISM}"
    )

if str(LOSS_REDUCTION).lower() != "mean":
    raise RuntimeError(
        f"Section 9 requires mean loss reduction. "
        f"Received: {LOSS_REDUCTION}"
    )

if str(GRAD_SAMPLE_MODE).lower() != "hooks":
    raise RuntimeError(
        f"Section 9 requires hooks grad-sample mode. "
        f"Received: {GRAD_SAMPLE_MODE}"
    )

if float(MAX_GRAD_NORM) <= 0:
    raise RuntimeError(
        "MAX_GRAD_NORM must be positive."
    )

if int(DP_BATCH_SIZE) <= 0:
    raise RuntimeError(
        "DP_BATCH_SIZE must be positive."
    )

if int(DP_EPOCHS) <= 0:
    raise RuntimeError(
        "DP_EPOCHS must be positive."
    )

print(
    "✓ Privacy configuration validated."
)


# --------------------------------------------------------------------------------------------------
# 3. Define DP-SGD discriminator wrapper
# --------------------------------------------------------------------------------------------------

def make_private_discriminator(
    critic,
    optimizer,
    data_loader,
    noise_multiplier,
    max_grad_norm,
):
    """
    Attach Opacus DP-SGD to the SPP-GAN discriminator.

    Opacus performs:
        1. Per-example gradient computation
        2. Flat gradient clipping
        3. Gaussian noise addition
        4. Poisson sampling
        5. DP optimizer update

    Formal privacy accounting is finalized in Notebook 11.
    """

    noise_multiplier = float(
        noise_multiplier
    )

    max_grad_norm = float(
        max_grad_norm
    )

    if not np.isfinite(
        noise_multiplier
    ):
        raise ValueError(
            "noise_multiplier must be finite."
        )

    if noise_multiplier <= 0:
        raise ValueError(
            "noise_multiplier must be positive."
        )

    if not np.isfinite(
        max_grad_norm
    ):
        raise ValueError(
            "max_grad_norm must be finite."
        )

    if max_grad_norm <= 0:
        raise ValueError(
            "max_grad_norm must be positive."
        )

    privacy_engine = PrivacyEngine(
        accountant="rdp"
    )

    (
        private_critic,
        private_optimizer,
        private_loader,
    ) = privacy_engine.make_private(
        module=critic,
        optimizer=optimizer,
        data_loader=data_loader,
        noise_multiplier=noise_multiplier,
        max_grad_norm=max_grad_norm,
        batch_first=True,
        loss_reduction="mean",
        poisson_sampling=True,
        clipping="flat",
        grad_sample_mode="hooks",
    )

    return (
        private_critic,
        private_optimizer,
        private_loader,
        privacy_engine,
    )


print(
    "✓ DP-SGD discriminator wrapper defined."
)


# --------------------------------------------------------------------------------------------------
# 4. Define WGAN-style discriminator loss
# --------------------------------------------------------------------------------------------------

def dp_discriminator_loss(
    private_critic,
    real_batch,
    fake_batch,
):
    """
    WGAN-style discriminator loss.

    Per-example objective:

        l_i = D(fake_i) - D(real_i)

    Batch objective:

        L_D = mean_i(l_i)

    fake_batch is detached from the generator during
    the discriminator update.
    """

    if not isinstance(
        real_batch,
        torch.Tensor,
    ):
        raise TypeError(
            "real_batch must be a torch.Tensor."
        )

    if not isinstance(
        fake_batch,
        torch.Tensor,
    ):
        raise TypeError(
            "fake_batch must be a torch.Tensor."
        )

    if real_batch.ndim != 2:
        raise ValueError(
            "real_batch must be 2D."
        )

    if fake_batch.ndim != 2:
        raise ValueError(
            "fake_batch must be 2D."
        )

    if real_batch.shape != fake_batch.shape:
        raise ValueError(
            "Real and fake batches must have identical shapes."
        )

    if not torch.isfinite(
        real_batch
    ).all():
        raise ValueError(
            "real_batch contains non-finite values."
        )

    if not torch.isfinite(
        fake_batch
    ).all():
        raise ValueError(
            "fake_batch contains non-finite values."
        )

    real_scores = (
        private_critic(
            real_batch
        )
        .reshape(-1)
    )

    fake_scores = (
        private_critic(
            fake_batch.detach()
        )
        .reshape(-1)
    )

    if real_scores.shape != fake_scores.shape:
        raise RuntimeError(
            "Real and fake score shapes do not match."
        )

    per_example_loss = (
        fake_scores
        -
        real_scores
    )

    loss = (
        per_example_loss.mean()
    )

    if loss.ndim != 0:
        raise RuntimeError(
            "Discriminator loss must be scalar."
        )

    if not torch.isfinite(
        loss
    ):
        raise RuntimeError(
            "Discriminator loss is non-finite."
        )

    return (
        loss,
        per_example_loss,
    )


print(
    "✓ WGAN-style per-example critic objective defined."
)


# --------------------------------------------------------------------------------------------------
# 5. Define one DP discriminator optimization step
# --------------------------------------------------------------------------------------------------

def dp_discriminator_step(
    private_critic,
    private_optimizer,
    real_batch,
    fake_batch,
):
    """
    Execute one production DP discriminator update.

    Opacus performs:
        per-example gradient calculation
        flat gradient clipping
        Gaussian noise addition
        optimizer update
    """

    private_optimizer.zero_grad(
        set_to_none=True
    )

    loss, per_example_loss = (
        dp_discriminator_loss(
            private_critic=private_critic,
            real_batch=real_batch,
            fake_batch=fake_batch,
        )
    )

    loss.backward()

    private_optimizer.step()

    return {
        "loss": loss.detach(),
        "per_example_loss": (
            per_example_loss.detach()
        ),
    }


print(
    "✓ DP discriminator optimization step defined."
)

print(
    "✓ Clipping, Gaussian noise, and Poisson sampling "
    "delegated to Opacus."
)


# --------------------------------------------------------------------------------------------------
# 6. Dataset-specific DP-SGD validation
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("DATASET-SPECIFIC DP-SGD VALIDATION")
print("-" * 100)


SECTION_09_RESULTS = []

SECTION_09_BATCH_SIZE = 8

SECTION_09_SEED_BASE = (
    int(MASTER_SEED) + 9000
    if "MASTER_SEED" in globals()
    else 11025
)


for dataset_index, dataset_id in enumerate(
    DATASET_IDS
):

    dataset_id = str(
        dataset_id
    )

    input_dim = int(
        CRITIC_INPUT_DIMENSIONS[
            dataset_id
        ]
    )

    if dataset_id not in (
        CALIBRATED_NOISE_MULTIPLIERS
    ):
        raise RuntimeError(
            f"No calibrated noise multiplier found for "
            f"{dataset_id}."
        )

    noise_multiplier = float(
        CALIBRATED_NOISE_MULTIPLIERS[
            dataset_id
        ]
    )

    dataset_seed = (
        SECTION_09_SEED_BASE
        +
        dataset_index
    )

    torch.manual_seed(
        dataset_seed
    )

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            dataset_seed
        )

    # ----------------------------------------------------------------------------------------------
    # 6.1 Fresh critic
    # ----------------------------------------------------------------------------------------------

    critic = SPPGANCritic(
        input_dim=input_dim
    )

    critic = critic.to(
        DEVICE
    )

    critic.train()

    # ----------------------------------------------------------------------------------------------
    # 6.2 Base optimizer
    # ----------------------------------------------------------------------------------------------

    optimizer = torch.optim.Adam(
        critic.parameters(),
        lr=2e-4,
        weight_decay=1e-6,
    )

    # ----------------------------------------------------------------------------------------------
    # 6.3 Validation dataset
    # ----------------------------------------------------------------------------------------------

    validation_data = torch.randn(
        SECTION_09_BATCH_SIZE * 8,
        input_dim,
        dtype=torch.float32,
    )

    validation_dataset = torch.utils.data.TensorDataset(
        validation_data
    )

    validation_loader = torch.utils.data.DataLoader(
        validation_dataset,
        batch_size=DP_BATCH_SIZE,
        shuffle=True,
        drop_last=False,
    )

    # ----------------------------------------------------------------------------------------------
    # 6.4 Attach Opacus
    # ----------------------------------------------------------------------------------------------

    (
        private_critic,
        private_optimizer,
        private_loader,
        privacy_engine,
    ) = make_private_discriminator(
        critic=critic,
        optimizer=optimizer,
        data_loader=validation_loader,
        noise_multiplier=noise_multiplier,
        max_grad_norm=float(
            MAX_GRAD_NORM
        ),
    )

    private_critic.train()

    # ----------------------------------------------------------------------------------------------
    # 6.5 Verify DPDataLoader
    # ----------------------------------------------------------------------------------------------

    private_loader_class = (
        private_loader.__class__.__name__
    )

    private_loader_module = (
        private_loader.__class__.__module__
    )

    is_opacus_dp_loader = bool(
        private_loader_class
        ==
        "DPDataLoader"
        and
        private_loader_module.startswith(
            "opacus."
        )
    )

    if not is_opacus_dp_loader:
        raise RuntimeError(
            f"{dataset_id}: unexpected Opacus private loader. "
            f"Class={private_loader_class}, "
            f"Module={private_loader_module}"
        )

    # ----------------------------------------------------------------------------------------------
    # 6.6 Verify Poisson sampling through Opacus loader metadata
    # ----------------------------------------------------------------------------------------------

    sampler = getattr(
        private_loader,
        "sampler",
        None,
    )

    batch_sampler = getattr(
        private_loader,
        "batch_sampler",
        None,
    )

    sampler_class = (
        sampler.__class__.__name__
        if sampler is not None
        else ""
    )

    batch_sampler_class = (
        batch_sampler.__class__.__name__
        if batch_sampler is not None
        else ""
    )

    sampler_module = (
        sampler.__class__.__module__
        if sampler is not None
        else ""
    )

    batch_sampler_module = (
        batch_sampler.__class__.__module__
        if batch_sampler is not None
        else ""
    )

    poisson_sampling_verified = bool(
        (
            "UniformWithReplacementSampler"
            in sampler_class
            or
            "UniformWithReplacementSampler"
            in batch_sampler_class
            or
            "UniformWithReplacementSampler"
            in sampler_module
            or
            "UniformWithReplacementSampler"
            in batch_sampler_module
        )
        or
        is_opacus_dp_loader
    )

    if not poisson_sampling_verified:
        raise RuntimeError(
            f"{dataset_id}: could not verify Opacus "
            "Poisson-sampling infrastructure. "
            f"Loader={private_loader_class}, "
            f"Sampler={sampler_class}, "
            f"BatchSampler={batch_sampler_class}"
        )

    # ----------------------------------------------------------------------------------------------
    # 6.7 Obtain one private batch
    # ----------------------------------------------------------------------------------------------

    private_iterator = iter(
        private_loader
    )

    real_batch = next(
        private_iterator
    )[0]

    real_batch = real_batch.to(
        DEVICE
    )

    fake_batch = torch.randn(
        real_batch.shape,
        dtype=real_batch.dtype,
        device=real_batch.device,
    )

    if real_batch.shape != fake_batch.shape:
        raise RuntimeError(
            f"{dataset_id}: real/fake shape mismatch."
        )

    # ----------------------------------------------------------------------------------------------
    # 6.8 Verify fake detachment
    # ----------------------------------------------------------------------------------------------

    generator_placeholder = torch.randn(
        real_batch.shape,
        dtype=real_batch.dtype,
        device=real_batch.device,
        requires_grad=True,
    )

    detached_fake = (
        generator_placeholder.detach()
    )

    fake_detachment_verified = bool(
        not detached_fake.requires_grad
    )

    if not fake_detachment_verified:
        raise RuntimeError(
            f"{dataset_id}: fake batch detachment failed."
        )

    # ----------------------------------------------------------------------------------------------
    # 6.9 Record parameters before update
    # ----------------------------------------------------------------------------------------------

    parameters_before = [
        parameter.detach().clone()
        for parameter
        in private_critic.parameters()
    ]

    # ----------------------------------------------------------------------------------------------
    # 6.10 Execute actual DP-SGD update
    # ----------------------------------------------------------------------------------------------

    step_result = dp_discriminator_step(
        private_critic=private_critic,
        private_optimizer=private_optimizer,
        real_batch=real_batch,
        fake_batch=fake_batch,
    )

    loss_value = float(
        step_result["loss"]
        .detach()
        .cpu()
        .item()
    )

    per_example_loss = (
        step_result["per_example_loss"]
    )

    # ----------------------------------------------------------------------------------------------
    # 6.11 Validate loss
    # ----------------------------------------------------------------------------------------------

    loss_scalar_valid = bool(
        np.isfinite(
            loss_value
        )
    )

    per_example_loss_valid = bool(
        per_example_loss.ndim == 1
        and
        per_example_loss.numel() > 0
        and
        torch.isfinite(
            per_example_loss
        ).all().item()
    )

    if not loss_scalar_valid:
        raise RuntimeError(
            f"{dataset_id}: DP discriminator loss is non-finite."
        )

    if not per_example_loss_valid:
        raise RuntimeError(
            f"{dataset_id}: per-example loss invalid."
        )

    # ----------------------------------------------------------------------------------------------
    # 6.12 Verify parameter update
    # ----------------------------------------------------------------------------------------------

    parameters_after = [
        parameter.detach().clone()
        for parameter
        in private_critic.parameters()
    ]

    parameter_changed = bool(
        any(
            not torch.equal(
                before,
                after,
            )
            for before, after
            in zip(
                parameters_before,
                parameters_after,
            )
        )
    )

    if not parameter_changed:
        raise RuntimeError(
            f"{dataset_id}: DP optimizer did not update "
            "critic parameters."
        )

    # ----------------------------------------------------------------------------------------------
    # 6.13 Verify grad_sample state
    # ----------------------------------------------------------------------------------------------

    grad_sample_observed = False
    grad_sample_valid = True
    grad_sample_parameter_count = 0

    for parameter in private_critic.parameters():

        if not hasattr(
            parameter,
            "grad_sample",
        ):
            continue

        grad_sample = (
            parameter.grad_sample
        )

        if grad_sample is None:
            continue

        grad_sample_observed = True

        if isinstance(
            grad_sample,
            list,
        ):
            entries = grad_sample
        else:
            entries = [
                grad_sample
            ]

        for entry in entries:

            if entry is None:
                continue

            grad_sample_parameter_count += 1

            if not torch.isfinite(
                entry
            ).all():
                grad_sample_valid = False

    grad_sample_observed = bool(
        grad_sample_observed
    )

    grad_sample_valid = bool(
        grad_sample_valid
    )

    if not grad_sample_observed:
        raise RuntimeError(
            f"{dataset_id}: Opacus grad_sample was not observed."
        )

    if not grad_sample_valid:
        raise RuntimeError(
            f"{dataset_id}: Opacus grad_sample contains "
            "non-finite values."
        )

    # ----------------------------------------------------------------------------------------------
    # 6.14 Verify Opacus DP optimizer
    # ----------------------------------------------------------------------------------------------

    optimizer_class_name = (
        private_optimizer.__class__.__name__
    )

    optimizer_module_name = (
        private_optimizer.__class__.__module__
    )

    optimizer_is_opacus = bool(
        (
            "opacus"
            in
            optimizer_module_name.lower()
        )
        or
        (
            "dpoptimizer"
            in
            optimizer_class_name.lower()
        )
    )

    if not optimizer_is_opacus:
        raise RuntimeError(
            f"{dataset_id}: optimizer is not an Opacus DP optimizer. "
            f"Class={optimizer_class_name}, "
            f"Module={optimizer_module_name}"
        )

    # ----------------------------------------------------------------------------------------------
    # 6.15 Verify calibrated noise multiplier
    # ----------------------------------------------------------------------------------------------

    optimizer_noise_multiplier = getattr(
        private_optimizer,
        "noise_multiplier",
        None,
    )

    if optimizer_noise_multiplier is None:
        raise RuntimeError(
            f"{dataset_id}: Opacus optimizer does not expose "
            "noise_multiplier."
        )

    optimizer_noise_multiplier = float(
        optimizer_noise_multiplier
    )

    noise_multiplier_match = bool(
        np.isclose(
            optimizer_noise_multiplier,
            noise_multiplier,
            rtol=0.0,
            atol=1e-12,
        )
    )

    if not noise_multiplier_match:
        raise RuntimeError(
            f"{dataset_id}: noise multiplier mismatch. "
            f"Expected={noise_multiplier}, "
            f"Observed={optimizer_noise_multiplier}"
        )

    # ----------------------------------------------------------------------------------------------
    # 6.16 Verify clipping threshold
    # ----------------------------------------------------------------------------------------------

    optimizer_max_grad_norm = getattr(
        private_optimizer,
        "max_grad_norm",
        None,
    )

    if optimizer_max_grad_norm is None:
        raise RuntimeError(
            f"{dataset_id}: Opacus optimizer does not expose "
            "max_grad_norm."
        )

    optimizer_max_grad_norm = float(
        optimizer_max_grad_norm
    )

    clipping_match = bool(
        np.isclose(
            optimizer_max_grad_norm,
            float(MAX_GRAD_NORM),
            rtol=0.0,
            atol=1e-12,
        )
    )

    if not clipping_match:
        raise RuntimeError(
            f"{dataset_id}: clipping threshold mismatch. "
            f"Expected={float(MAX_GRAD_NORM)}, "
            f"Observed={optimizer_max_grad_norm}"
        )

    # ----------------------------------------------------------------------------------------------
    # 6.17 Verify PrivacyEngine
    # ----------------------------------------------------------------------------------------------

    privacy_engine_valid = bool(
        isinstance(
            privacy_engine,
            PrivacyEngine,
        )
    )

    if not privacy_engine_valid:
        raise RuntimeError(
            f"{dataset_id}: PrivacyEngine validation failed."
        )

    # ----------------------------------------------------------------------------------------------
    # 6.18 Final dataset validation
    # ----------------------------------------------------------------------------------------------

    dataset_status = bool(
        all(
            [
                is_opacus_dp_loader,
                poisson_sampling_verified,
                fake_detachment_verified,
                loss_scalar_valid,
                per_example_loss_valid,
                parameter_changed,
                grad_sample_observed,
                grad_sample_valid,
                optimizer_is_opacus,
                noise_multiplier_match,
                clipping_match,
                privacy_engine_valid,
            ]
        )
    )

    if not dataset_status:
        raise RuntimeError(
            f"{dataset_id}: Section 9 validation failed."
        )

    SECTION_09_RESULTS.append(
        {
            "dataset_id": dataset_id,
            "critic_input_dimension": int(
                input_dim
            ),
            "validation_batch_size": int(
                real_batch.shape[0]
            ),
            "noise_multiplier": float(
                noise_multiplier
            ),
            "optimizer_noise_multiplier": float(
                optimizer_noise_multiplier
            ),
            "max_grad_norm": float(
                MAX_GRAD_NORM
            ),
            "optimizer_max_grad_norm": float(
                optimizer_max_grad_norm
            ),
            "loss": float(
                loss_value
            ),
            "per_example_loss_count": int(
                per_example_loss.numel()
            ),
            "fake_detachment_verified": bool(
                fake_detachment_verified
            ),
            "grad_sample_observed": bool(
                grad_sample_observed
            ),
            "grad_sample_parameter_count": int(
                grad_sample_parameter_count
            ),
            "grad_sample_finite": bool(
                grad_sample_valid
            ),
            "optimizer_class": str(
                optimizer_class_name
            ),
            "optimizer_module": str(
                optimizer_module_name
            ),
            "optimizer_is_opacus": bool(
                optimizer_is_opacus
            ),
            "private_loader_class": str(
                private_loader_class
            ),
            "private_loader_module": str(
                private_loader_module
            ),
            "sampler_class": str(
                sampler_class
            ),
            "sampler_module": str(
                sampler_module
            ),
            "poisson_sampling_verified": bool(
                poisson_sampling_verified
            ),
            "privacy_engine_valid": bool(
                privacy_engine_valid
            ),
            "parameter_update_verified": bool(
                parameter_changed
            ),
            "status": "PASS",
        }
    )

    print(
        f"✓ {dataset_id:<20} : PASS | "
        f"input={input_dim} | "
        f"σ={noise_multiplier:.6f} | "
        f"loss={loss_value:.6f} | "
        f"Opacus=YES | "
        f"DPDataLoader=YES"
    )


# --------------------------------------------------------------------------------------------------
# 7. Create validation DataFrame
# --------------------------------------------------------------------------------------------------

SECTION_09_DP_SGD_VALIDATION_DF = pd.DataFrame(
    SECTION_09_RESULTS
)

if SECTION_09_DP_SGD_VALIDATION_DF.empty:
    raise RuntimeError(
        "Section 9 validation DataFrame is empty."
    )

if len(
    SECTION_09_DP_SGD_VALIDATION_DF
) != len(
    DATASET_IDS
):
    raise RuntimeError(
        "Section 9 validation must contain one row per dataset."
    )


# --------------------------------------------------------------------------------------------------
# 8. Prepare canonical directories
# --------------------------------------------------------------------------------------------------

NB10_ROOT = (
    Path(
        "/content/drive/MyDrive/SPP_GAN_Research"
    )
    /
    "results"
    /
    "notebooks"
    /
    "notebook_10"
)

VALIDATION_ROOT = (
    NB10_ROOT
    /
    "validation"
)

METADATA_ROOT = (
    NB10_ROOT
    /
    "metadata"
)

VALIDATION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

METADATA_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------------------------------------------------------
# 9. Persist validation artifact
# --------------------------------------------------------------------------------------------------

SECTION_09_VALIDATION_PATH = (
    VALIDATION_ROOT
    /
    "dp_sgd_discriminator_validation.csv"
)

SECTION_09_DP_SGD_VALIDATION_DF.to_csv(
    SECTION_09_VALIDATION_PATH,
    index=False,
)

print(
    "\n✓ DP-SGD validation artifact persisted."
)

print(
    f"  Validation : {SECTION_09_VALIDATION_PATH}"
)


# --------------------------------------------------------------------------------------------------
# 10. Build manifest
# --------------------------------------------------------------------------------------------------

SECTION_09_MANIFEST = {
    "notebook": "10",
    "section": "09",
    "title": "Define DP-SGD Discriminator Update",
    "status": "PASS",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "algorithm": {
        "type": "DP-SGD",
        "optimizer_framework": "Opacus",
        "accountant": str(
            ACCOUNTANT
        ),
        "sampling": str(
            SAMPLING_MECHANISM
        ),
        "clipping": str(
            CLIPPING_MECHANISM
        ),
        "grad_sample_mode": str(
            GRAD_SAMPLE_MODE
        ),
        "loss_reduction": str(
            LOSS_REDUCTION
        ),
    },

    "discriminator_objective": {
        "form": (
            "L_D = mean(D(fake) - D(real))"
        ),
        "per_example_form": (
            "l_i = D(fake_i) - D(real_i)"
        ),
        "fake_batch_detached": True,
        "generator_update_not_performed": True,
    },

    "privacy_parameters": {
        "max_grad_norm": float(
            MAX_GRAD_NORM
        ),
        "batch_size_nominal": int(
            DP_BATCH_SIZE
        ),
        "epochs": int(
            DP_EPOCHS
        ),
        "noise_multipliers": {
            str(dataset_id): float(
                value
            )
            for dataset_id, value
            in CALIBRATED_NOISE_MULTIPLIERS.items()
        },
    },

    "validation": {
        "dataset_count": int(
            len(
                SECTION_09_RESULTS
            )
        ),
        "all_datasets_passed": bool(
            all(
                row["status"] == "PASS"
                for row
                in SECTION_09_RESULTS
            )
        ),
        "opacus_optimizer_verified": bool(
            all(
                row["optimizer_is_opacus"]
                for row
                in SECTION_09_RESULTS
            )
        ),
        "dp_data_loader_verified": bool(
            all(
                row["private_loader_class"]
                ==
                "DPDataLoader"
                for row
                in SECTION_09_RESULTS
            )
        ),
        "noise_multiplier_verified": bool(
            all(
                np.isclose(
                    row["optimizer_noise_multiplier"],
                    row["noise_multiplier"],
                    rtol=0.0,
                    atol=1e-12,
                )
                for row
                in SECTION_09_RESULTS
            )
        ),
        "clipping_threshold_verified": bool(
            all(
                np.isclose(
                    row["optimizer_max_grad_norm"],
                    row["max_grad_norm"],
                    rtol=0.0,
                    atol=1e-12,
                )
                for row
                in SECTION_09_RESULTS
            )
        ),
        "grad_sample_verified": bool(
            all(
                row["grad_sample_observed"]
                and
                row["grad_sample_finite"]
                for row
                in SECTION_09_RESULTS
            )
        ),
        "parameter_update_verified": bool(
            all(
                row["parameter_update_verified"]
                for row
                in SECTION_09_RESULTS
            )
        ),
        "poisson_sampling_verified": bool(
            all(
                row["poisson_sampling_verified"]
                for row
                in SECTION_09_RESULTS
            )
        ),
        "fake_detachment_verified": bool(
            all(
                row["fake_detachment_verified"]
                for row
                in SECTION_09_RESULTS
            )
        ),
    },

    "artifacts": {
        "validation_csv": str(
            SECTION_09_VALIDATION_PATH
        ),
    },

    "privacy_boundary": {
        "per_example_gradients": (
            "Notebook 10 Section 6"
        ),
        "gradient_clipping": (
            "Notebook 10 Section 7"
        ),
        "gaussian_noise": (
            "Notebook 10 Section 8"
        ),
        "dp_sgd_update": (
            "Notebook 10 Section 9"
        ),
        "privacy_accounting": (
            "Notebook 11"
        ),
        "achieved_epsilon": (
            "Notebook 11"
        ),
        "end_to_end_privacy_claim": False,
    },
}


# --------------------------------------------------------------------------------------------------
# 11. JSON serializability validation
# --------------------------------------------------------------------------------------------------

try:

    json.dumps(
        SECTION_09_MANIFEST,
        indent=2,
        sort_keys=True,
    )

except (
    TypeError,
    ValueError,
) as exc:

    raise RuntimeError(
        "Section 9 manifest failed JSON serializability validation."
    ) from exc

print(
    "✓ Section 9 manifest JSON serializability : PASS"
)


# --------------------------------------------------------------------------------------------------
# 12. Persist manifest
# --------------------------------------------------------------------------------------------------

SECTION_09_MANIFEST_PATH = (
    METADATA_ROOT
    /
    "section_09_dp_sgd_discriminator_manifest.json"
)

with open(
    SECTION_09_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        SECTION_09_MANIFEST,
        f,
        indent=2,
        sort_keys=True,
    )

print(
    f"✓ Section 9 manifest persisted: "
    f"{SECTION_09_MANIFEST_PATH}"
)


# --------------------------------------------------------------------------------------------------
# 13. Final verification
# --------------------------------------------------------------------------------------------------

VALIDATION_ALL_PASS = bool(
    (
        SECTION_09_DP_SGD_VALIDATION_DF[
            "status"
        ]
        ==
        "PASS"
    ).all()
)

ARTIFACTS_EXIST = bool(
    SECTION_09_VALIDATION_PATH.exists()
    and
    SECTION_09_MANIFEST_PATH.exists()
)

SECTION_09_PASS = bool(
    VALIDATION_ALL_PASS
    and
    ARTIFACTS_EXIST
)

if not SECTION_09_PASS:
    raise RuntimeError(
        "SECTION 9 FINAL VERIFICATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 14. Final output
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 9 FINAL VERIFICATION")
print("=" * 100)

print(
    "✓ DP-SGD discriminator wrapper             : PASS"
)

print(
    "✓ WGAN-style discriminator objective      : PASS"
)

print(
    "✓ Per-example loss computation             : PASS"
)

print(
    "✓ Opacus PrivacyEngine                    : PASS"
)

print(
    "✓ Opacus DP optimizer                     : PASS"
)

print(
    "✓ Opacus DPDataLoader                     : PASS"
)

print(
    "✓ Poisson sampling infrastructure          : PASS"
)

print(
    "✓ Per-example gradient generation          : PASS"
)

print(
    "✓ Flat clipping configuration              : PASS"
)

print(
    "✓ Gaussian noise configuration             : PASS"
)

print(
    "✓ Calibrated noise multipliers             : PASS"
)

print(
    "✓ Actual discriminator parameter update    : PASS"
)

print(
    "✓ Fake-batch detachment                    : PASS"
)

print(
    "✓ Dataset-specific validation              : PASS"
)

print(
    "✓ Validation artifact persisted            : PASS"
)

print(
    "✓ Section 9 manifest JSON serializability  : PASS"
)

print(
    "✓ Section 9 manifest persisted              : PASS"
)


# --------------------------------------------------------------------------------------------------
# 15. Privacy boundary
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("PRIVACY BOUNDARY")
print("-" * 100)

print(
    "Per-example gradients : Notebook 10 Section 6"
)

print(
    "Gradient clipping     : Notebook 10 Section 7"
)

print(
    "Gaussian noise        : Notebook 10 Section 8"
)

print(
    "DP-SGD update         : Notebook 10 Section 9"
)

print(
    "Privacy accounting    : Notebook 11"
)

print(
    "Achieved epsilon      : Notebook 11"
)

print(
    "End-to-end DP claim   : DISABLED"
)

print("\n" + "=" * 100)
print("SECTION 9 STATUS: PASS")
print("=" * 100)


9. DEFINE DP-SGD DISCRIMINATOR UPDATE
✓ Privacy configuration validated.
✓ DP-SGD discriminator wrapper defined.
✓ WGAN-style per-example critic objective defined.
✓ DP discriminator optimization step defined.
✓ Clipping, Gaussian noise, and Poisson sampling delegated to Opacus.

----------------------------------------------------------------------------------------------------
DATASET-SPECIFIC DP-SGD VALIDATION
----------------------------------------------------------------------------------------------------
✓ adult_income         : PASS | input=105 | σ=1.216221 | loss=-0.005895 | Opacus=YES | DPDataLoader=YES
✓ bank_marketing       : PASS | input=51 | σ=1.251014 | loss=0.007547 | Opacus=YES | DPDataLoader=YES


/tmp/ipykernel_5574/2714192293.py:340: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  loss.backward()


✓ diabetes_130us       : PASS | input=2329 | σ=0.953403 | loss=-0.011117 | Opacus=YES | DPDataLoader=YES

✓ DP-SGD validation artifact persisted.
  Validation : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10/validation/dp_sgd_discriminator_validation.csv
✓ Section 9 manifest JSON serializability : PASS
✓ Section 9 manifest persisted: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10/metadata/section_09_dp_sgd_discriminator_manifest.json

SECTION 9 FINAL VERIFICATION
✓ DP-SGD discriminator wrapper             : PASS
✓ WGAN-style discriminator objective      : PASS
✓ Per-example loss computation             : PASS
✓ Opacus PrivacyEngine                    : PASS
✓ Opacus DP optimizer                     : PASS
✓ Opacus DPDataLoader                     : PASS
✓ Poisson sampling infrastructure          : PASS
✓ Per-example gradient generation          : PASS
✓ Flat clipping configuration              : PASS
✓ Gaussian noise configuration          

In [13]:
# ==================================================================================================
# 10. DEFINE SAMPLING MECHANISM
# ==================================================================================================

print("\n" + "=" * 100)
print("10. DEFINE SAMPLING MECHANISM")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Validate required dependencies
# --------------------------------------------------------------------------------------------------

REQUIRED_SECTION_10_OBJECTS = [
    "DATASET_IDS",
    "MASTER_SEED",
    "TRAIN_ROWS",
    "DP_BATCH_SIZE",
    "SAMPLING_MECHANISM",
    "PrivacyEngine",
]

missing_objects = [
    name
    for name in REQUIRED_SECTION_10_OBJECTS
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Section 10 is missing required objects: "
        f"{missing_objects}"
    )


# --------------------------------------------------------------------------------------------------
# 2. Validate sampling configuration
# --------------------------------------------------------------------------------------------------

if str(
    SAMPLING_MECHANISM
).lower() != "poisson":

    raise RuntimeError(
        "Notebook 10 requires Poisson sampling. "
        f"Received: {SAMPLING_MECHANISM}"
    )

if int(
    DP_BATCH_SIZE
) <= 0:

    raise RuntimeError(
        "DP_BATCH_SIZE must be positive."
    )

if not isinstance(
    TRAIN_ROWS,
    dict,
):

    raise TypeError(
        "TRAIN_ROWS must be a dictionary."
    )

EXPECTED_DATASETS = {
    str(dataset_id)
    for dataset_id in DATASET_IDS
}

AVAILABLE_TRAIN_ROWS = {
    str(dataset_id)
    for dataset_id in TRAIN_ROWS.keys()
}

if AVAILABLE_TRAIN_ROWS != EXPECTED_DATASETS:

    raise RuntimeError(
        "TRAIN_ROWS dataset coverage does not match DATASET_IDS. "
        f"Expected={sorted(EXPECTED_DATASETS)}, "
        f"Received={sorted(AVAILABLE_TRAIN_ROWS)}"
    )

print(
    "✓ Sampling configuration validated."
)

print(
    "✓ Configured sampling mechanism : Poisson"
)

print(
    f"✓ Nominal DP batch size        : {int(DP_BATCH_SIZE)}"
)


# --------------------------------------------------------------------------------------------------
# 3. Define diagnostic Poisson/Bernoulli sampler
# --------------------------------------------------------------------------------------------------

def poisson_sample_indices(
    n_samples,
    sample_rate,
    generator=None,
):
    """
    Diagnostic Bernoulli inclusion sampler.

    Each record is independently included with probability:

        q = sample_rate

    This function is used only for mathematical diagnostics.

    Production DP training uses the Opacus DPDataLoader.
    """

    n_samples = int(
        n_samples
    )

    sample_rate = float(
        sample_rate
    )

    if n_samples <= 0:
        raise ValueError(
            "n_samples must be positive."
        )

    if not np.isfinite(
        sample_rate
    ):
        raise ValueError(
            "sample_rate must be finite."
        )

    if not (
        0.0 < sample_rate <= 1.0
    ):
        raise ValueError(
            "sample_rate must be in (0, 1]."
        )

    included = (
        torch.rand(
            n_samples,
            generator=generator,
            device="cpu",
        )
        <
        sample_rate
    )

    return torch.nonzero(
        included,
        as_tuple=False,
    ).flatten()


print(
    "✓ Diagnostic Bernoulli/Poisson sampler defined."
)


# --------------------------------------------------------------------------------------------------
# 4. Define RAM-safe diagnostic validator
# --------------------------------------------------------------------------------------------------

def validate_poisson_sampling(
    n_samples,
    sample_rate,
    repetitions=1000,
    seed=None,
):
    """
    RAM-safe diagnostic validation of Bernoulli/Poisson
    inclusion sampling.

    For:

        B ~ Binomial(N, q)

    the expected batch size is:

        E[B] = Nq

    and the variance is:

        Var(B) = Nq(1-q)
    """

    n_samples = int(
        n_samples
    )

    sample_rate = float(
        sample_rate
    )

    repetitions = int(
        repetitions
    )

    if n_samples <= 0:
        raise ValueError(
            "n_samples must be positive."
        )

    if not (
        0.0 < sample_rate <= 1.0
    ):
        raise ValueError(
            "sample_rate must be in (0, 1]."
        )

    if repetitions <= 0:
        raise ValueError(
            "repetitions must be positive."
        )

    if seed is None:
        seed = int(
            MASTER_SEED
        )

    generator = torch.Generator(
        device="cpu"
    )

    generator.manual_seed(
        int(seed)
    )

    batch_sizes = []

    for _ in range(
        repetitions
    ):

        indices = poisson_sample_indices(
            n_samples=n_samples,
            sample_rate=sample_rate,
            generator=generator,
        )

        batch_sizes.append(
            int(
                indices.numel()
            )
        )

    batch_sizes = np.asarray(
        batch_sizes,
        dtype=np.int64,
    )

    expected_batch_size = (
        n_samples
        *
        sample_rate
    )

    expected_variance = (
        n_samples
        *
        sample_rate
        *
        (
            1.0
            -
            sample_rate
        )
    )

    expected_std = float(
        np.sqrt(
            expected_variance
        )
    )

    observed_mean = float(
        batch_sizes.mean()
    )

    observed_std = float(
        batch_sizes.std(
            ddof=1
        )
    )

    return {
        "n_samples": int(
            n_samples
        ),
        "sample_rate": float(
            sample_rate
        ),
        "repetitions": int(
            repetitions
        ),
        "mean_batch_size": float(
            observed_mean
        ),
        "min_batch_size": int(
            batch_sizes.min()
        ),
        "max_batch_size": int(
            batch_sizes.max()
        ),
        "observed_std_batch_size": float(
            observed_std
        ),
        "expected_batch_size": float(
            expected_batch_size
        ),
        "expected_variance": float(
            expected_variance
        ),
        "expected_std_batch_size": float(
            expected_std
        ),
        "seed": int(
            seed
        ),
    }


print(
    "✓ Diagnostic Poisson validator defined."
)


# --------------------------------------------------------------------------------------------------
# 5. Mathematical validation of diagnostic sampler
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("POISSON SAMPLING MATHEMATICAL VALIDATION")
print("-" * 100)


DIAGNOSTIC_N = 10000
DIAGNOSTIC_Q = 0.10
DIAGNOSTIC_REPETITIONS = 1000

DIAGNOSTIC_RESULT = (
    validate_poisson_sampling(
        n_samples=DIAGNOSTIC_N,
        sample_rate=DIAGNOSTIC_Q,
        repetitions=DIAGNOSTIC_REPETITIONS,
        seed=int(
            MASTER_SEED
        ),
    )
)


# --------------------------------------------------------------------------------------------------
# 5.1 Expected batch-size validation
# --------------------------------------------------------------------------------------------------

EXPECTED_BATCH_SIZE = float(
    DIAGNOSTIC_RESULT[
        "expected_batch_size"
    ]
)

OBSERVED_MEAN_BATCH_SIZE = float(
    DIAGNOSTIC_RESULT[
        "mean_batch_size"
    ]
)

EXPECTED_STD_BATCH_SIZE = float(
    DIAGNOSTIC_RESULT[
        "expected_std_batch_size"
    ]
)

OBSERVED_STD_BATCH_SIZE = float(
    DIAGNOSTIC_RESULT[
        "observed_std_batch_size"
    ]
)

MEAN_TOLERANCE = float(
    max(
        1.0,
        5.0
        *
        EXPECTED_STD_BATCH_SIZE
        /
        np.sqrt(
            DIAGNOSTIC_REPETITIONS
        ),
    )
)

MEAN_VALID = bool(
    abs(
        OBSERVED_MEAN_BATCH_SIZE
        -
        EXPECTED_BATCH_SIZE
    )
    <=
    MEAN_TOLERANCE
)

if not MEAN_VALID:

    raise RuntimeError(
        "Poisson empirical mean batch size is outside "
        "the validation tolerance. "
        f"Expected={EXPECTED_BATCH_SIZE:.6f}, "
        f"Observed={OBSERVED_MEAN_BATCH_SIZE:.6f}, "
        f"Tolerance={MEAN_TOLERANCE:.6f}"
    )

print(
    f"✓ Expected batch size E[B]=Nq : PASS "
    f"(expected={EXPECTED_BATCH_SIZE:.3f}, "
    f"observed={OBSERVED_MEAN_BATCH_SIZE:.3f})"
)


# --------------------------------------------------------------------------------------------------
# 5.2 Batch-size variability validation
# --------------------------------------------------------------------------------------------------

if EXPECTED_STD_BATCH_SIZE > 0:

    STD_RATIO = (
        OBSERVED_STD_BATCH_SIZE
        /
        EXPECTED_STD_BATCH_SIZE
    )

else:

    STD_RATIO = 1.0


STD_VALID = bool(
    0.90
    <=
    STD_RATIO
    <=
    1.10
)

if not STD_VALID:

    raise RuntimeError(
        "Poisson batch-size standard deviation is outside "
        "the validation interval. "
        f"Expected={EXPECTED_STD_BATCH_SIZE:.6f}, "
        f"Observed={OBSERVED_STD_BATCH_SIZE:.6f}"
    )

print(
    f"✓ Poisson batch-size variability : PASS "
    f"(expected SD={EXPECTED_STD_BATCH_SIZE:.3f}, "
    f"observed SD={OBSERVED_STD_BATCH_SIZE:.3f})"
)


# --------------------------------------------------------------------------------------------------
# 5.3 Batch-size range validation
# --------------------------------------------------------------------------------------------------

MIN_BATCH_SIZE = int(
    DIAGNOSTIC_RESULT[
        "min_batch_size"
    ]
)

MAX_BATCH_SIZE = int(
    DIAGNOSTIC_RESULT[
        "max_batch_size"
    ]
)

BATCH_RANGE_VALID = bool(
    0
    <=
    MIN_BATCH_SIZE
    <=
    MAX_BATCH_SIZE
    <=
    DIAGNOSTIC_N
)

if not BATCH_RANGE_VALID:

    raise RuntimeError(
        "Observed Poisson batch-size range is invalid."
    )

print(
    f"✓ Batch-size bounds [0,N] : PASS "
    f"(min={MIN_BATCH_SIZE}, max={MAX_BATCH_SIZE})"
)


# --------------------------------------------------------------------------------------------------
# 5.4 Independent-sampling validation
# --------------------------------------------------------------------------------------------------

GENERATOR_A = torch.Generator(
    device="cpu"
)

GENERATOR_B = torch.Generator(
    device="cpu"
)

GENERATOR_A.manual_seed(
    int(MASTER_SEED) + 101
)

GENERATOR_B.manual_seed(
    int(MASTER_SEED) + 202
)

SAMPLE_A = poisson_sample_indices(
    n_samples=DIAGNOSTIC_N,
    sample_rate=DIAGNOSTIC_Q,
    generator=GENERATOR_A,
)

SAMPLE_B = poisson_sample_indices(
    n_samples=DIAGNOSTIC_N,
    sample_rate=DIAGNOSTIC_Q,
    generator=GENERATOR_B,
)

INDEPENDENT_SAMPLING_VALID = bool(
    not torch.equal(
        SAMPLE_A,
        SAMPLE_B,
    )
)

if not INDEPENDENT_SAMPLING_VALID:

    raise RuntimeError(
        "Independent Poisson sampling draws were identical."
    )

print(
    "✓ Independent Poisson sampling draws : PASS"
)


# --------------------------------------------------------------------------------------------------
# 5.5 Zero-inclusion possibility validation
# --------------------------------------------------------------------------------------------------

SMALL_N = 10
SMALL_Q = 0.05
ZERO_TEST_REPETITIONS = 5000

ZERO_TEST_GENERATOR = torch.Generator(
    device="cpu"
)

ZERO_TEST_GENERATOR.manual_seed(
    int(MASTER_SEED) + 303
)

zero_batch_observed = False

for _ in range(
    ZERO_TEST_REPETITIONS
):

    zero_indices = poisson_sample_indices(
        n_samples=SMALL_N,
        sample_rate=SMALL_Q,
        generator=ZERO_TEST_GENERATOR,
    )

    if zero_indices.numel() == 0:

        zero_batch_observed = True
        break


ZERO_INCLUSION_BEHAVIOR_VALID = bool(
    zero_batch_observed
)

if not ZERO_INCLUSION_BEHAVIOR_VALID:

    raise RuntimeError(
        "Diagnostic Poisson sampler did not demonstrate "
        "zero-inclusion behavior where it is probabilistically possible."
    )

print(
    "✓ Zero-inclusion behavior possible : PASS"
)


# --------------------------------------------------------------------------------------------------
# 5.6 Input validation
# --------------------------------------------------------------------------------------------------

INPUT_VALIDATION_PASS = True

try:

    poisson_sample_indices(
        n_samples=0,
        sample_rate=0.1,
    )

    INPUT_VALIDATION_PASS = False

except ValueError:
    pass


try:

    poisson_sample_indices(
        n_samples=100,
        sample_rate=0.0,
    )

    INPUT_VALIDATION_PASS = False

except ValueError:
    pass


try:

    poisson_sample_indices(
        n_samples=100,
        sample_rate=1.1,
    )

    INPUT_VALIDATION_PASS = False

except ValueError:
    pass


INPUT_VALIDATION_PASS = bool(
    INPUT_VALIDATION_PASS
)

if not INPUT_VALIDATION_PASS:

    raise RuntimeError(
        "Diagnostic Poisson sampler input validation failed."
    )

print(
    "✓ Sampler input validation : PASS"
)


# --------------------------------------------------------------------------------------------------
# 6. Dataset-specific sampling validation
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("DATASET-SPECIFIC SAMPLING VALIDATION")
print("-" * 100)


SECTION_10_RESULTS = []

DATASET_REPETITIONS = 500

for dataset_index, dataset_id in enumerate(
    DATASET_IDS
):

    dataset_id = str(
        dataset_id
    )

    n_train = int(
        TRAIN_ROWS[
            dataset_id
        ]
    )

    if n_train <= 0:

        raise RuntimeError(
            f"{dataset_id}: invalid training-row count."
        )

    sample_rate = float(
        min(
            1.0,
            int(DP_BATCH_SIZE)
            /
            n_train,
        )
    )

    dataset_seed = (
        int(MASTER_SEED)
        +
        10000
        +
        dataset_index
    )

    result = validate_poisson_sampling(
        n_samples=n_train,
        sample_rate=sample_rate,
        repetitions=DATASET_REPETITIONS,
        seed=dataset_seed,
    )

    expected_batch_size = float(
        result[
            "expected_batch_size"
        ]
    )

    observed_mean = float(
        result[
            "mean_batch_size"
        ]
    )

    expected_std = float(
        result[
            "expected_std_batch_size"
        ]
    )

    observed_std = float(
        result[
            "observed_std_batch_size"
        ]
    )

    if expected_std > 0:

        std_ratio = (
            observed_std
            /
            expected_std
        )

    else:

        std_ratio = 1.0

    dataset_mean_tolerance = float(
        max(
            1.0,
            5.0
            *
            expected_std
            /
            np.sqrt(
                DATASET_REPETITIONS
            ),
        )
    )

    dataset_mean_valid = bool(
        abs(
            observed_mean
            -
            expected_batch_size
        )
        <=
        dataset_mean_tolerance
    )

    dataset_std_valid = bool(
        0.90
        <=
        std_ratio
        <=
        1.10
    )

    dataset_range_valid = bool(
        0
        <=
        result["min_batch_size"]
        <=
        result["max_batch_size"]
        <=
        n_train
    )

    dataset_status = bool(
        all(
            [
                dataset_mean_valid,
                dataset_std_valid,
                dataset_range_valid,
            ]
        )
    )

    if not dataset_status:

        raise RuntimeError(
            f"{dataset_id}: dataset-specific Poisson "
            "sampling validation failed."
        )

    SECTION_10_RESULTS.append(
        {
            "dataset_id": dataset_id,
            "n_train": int(
                n_train
            ),
            "nominal_batch_size": int(
                DP_BATCH_SIZE
            ),
            "poisson_sample_rate": float(
                sample_rate
            ),
            "expected_batch_size": float(
                expected_batch_size
            ),
            "observed_mean_batch_size": float(
                observed_mean
            ),
            "expected_batch_std": float(
                expected_std
            ),
            "observed_batch_std": float(
                observed_std
            ),
            "std_ratio": float(
                std_ratio
            ),
            "min_batch_size": int(
                result["min_batch_size"]
            ),
            "max_batch_size": int(
                result["max_batch_size"]
            ),
            "repetitions": int(
                DATASET_REPETITIONS
            ),
            "mean_validation": bool(
                dataset_mean_valid
            ),
            "std_validation": bool(
                dataset_std_valid
            ),
            "range_validation": bool(
                dataset_range_valid
            ),
            "status": "PASS",
        }
    )

    print(
        f"✓ {dataset_id:<20} : PASS | "
        f"N={n_train} | "
        f"q={sample_rate:.8f} | "
        f"E[B]={expected_batch_size:.3f} | "
        f"mean={observed_mean:.3f}"
    )


# --------------------------------------------------------------------------------------------------
# 7. Validate production Opacus DPDataLoader
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("PRODUCTION OPACUS DPDataLoader VALIDATION")
print("-" * 100)


OPACUS_LOADER_RESULTS = []


for dataset_index, dataset_id in enumerate(
    DATASET_IDS
):

    dataset_id = str(
        dataset_id
    )

    n_train = int(
        TRAIN_ROWS[
            dataset_id
        ]
    )

    input_dimension = (
        int(
            CRITIC_INPUT_DIMENSIONS[
                dataset_id
            ]
        )
        if "CRITIC_INPUT_DIMENSIONS" in globals()
        else 1
    )

    torch.manual_seed(
        int(MASTER_SEED)
        +
        11000
        +
        dataset_index
    )

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            int(MASTER_SEED)
            +
            11000
            +
            dataset_index
        )

    # ----------------------------------------------------------------------------------------------
    # 7.1 RAM-safe validation dataset
    # ----------------------------------------------------------------------------------------------

    loader_validation_rows = min(
        256,
        max(
            32,
            int(DP_BATCH_SIZE) * 2,
        ),
    )

    validation_tensor = torch.randn(
        loader_validation_rows,
        input_dimension,
        dtype=torch.float32,
    )

    validation_dataset = (
        torch.utils.data.TensorDataset(
            validation_tensor
        )
    )

    validation_loader = (
        torch.utils.data.DataLoader(
            validation_dataset,
            batch_size=int(
                DP_BATCH_SIZE
            ),
            shuffle=True,
            drop_last=False,
        )
    )

    # ----------------------------------------------------------------------------------------------
    # 7.2 Fresh critic and optimizer
    # ----------------------------------------------------------------------------------------------

    if "SPPGANCritic" not in globals():

        raise RuntimeError(
            "SPPGANCritic is required for production "
            "DPDataLoader validation."
        )

    critic = SPPGANCritic(
        input_dim=input_dimension
    ).to(
        DEVICE
    )

    optimizer = torch.optim.Adam(
        critic.parameters(),
        lr=2e-4,
        weight_decay=1e-6,
    )

    # ----------------------------------------------------------------------------------------------
    # 7.3 Dataset-specific calibrated sigma
    # ----------------------------------------------------------------------------------------------

    if "CALIBRATED_NOISE_MULTIPLIERS" not in globals():

        raise RuntimeError(
            "CALIBRATED_NOISE_MULTIPLIERS is required."
        )

    noise_multiplier = float(
        CALIBRATED_NOISE_MULTIPLIERS[
            dataset_id
        ]
    )

    # ----------------------------------------------------------------------------------------------
    # 7.4 Attach Opacus
    # ----------------------------------------------------------------------------------------------

    privacy_engine = PrivacyEngine(
        accountant="rdp"
    )

    (
        private_critic,
        private_optimizer,
        private_loader,
    ) = privacy_engine.make_private(
        module=critic,
        optimizer=optimizer,
        data_loader=validation_loader,
        noise_multiplier=noise_multiplier,
        max_grad_norm=float(
            MAX_GRAD_NORM
        ),
        batch_first=True,
        loss_reduction="mean",
        poisson_sampling=True,
        clipping="flat",
        grad_sample_mode="hooks",
    )

    # ----------------------------------------------------------------------------------------------
    # 7.5 Validate private loader identity
    # ----------------------------------------------------------------------------------------------

    loader_class = (
        private_loader.__class__.__name__
    )

    loader_module = (
        private_loader.__class__.__module__
    )

    dp_loader_valid = bool(
        loader_class
        ==
        "DPDataLoader"
        and
        loader_module.startswith(
            "opacus."
        )
    )

    if not dp_loader_valid:

        raise RuntimeError(
            f"{dataset_id}: expected Opacus DPDataLoader. "
            f"Class={loader_class}, "
            f"Module={loader_module}"
        )

    # ----------------------------------------------------------------------------------------------
    # 7.6 Inspect sampler
    # ----------------------------------------------------------------------------------------------

    sampler = getattr(
        private_loader,
        "sampler",
        None,
    )

    batch_sampler = getattr(
        private_loader,
        "batch_sampler",
        None,
    )

    sampler_class = (
        sampler.__class__.__name__
        if sampler is not None
        else ""
    )

    sampler_module = (
        sampler.__class__.__module__
        if sampler is not None
        else ""
    )

    batch_sampler_class = (
        batch_sampler.__class__.__name__
        if batch_sampler is not None
        else ""
    )

    batch_sampler_module = (
        batch_sampler.__class__.__module__
        if batch_sampler is not None
        else ""
    )

    # ----------------------------------------------------------------------------------------------
    # 7.7 Validate Opacus sampling infrastructure
    # ----------------------------------------------------------------------------------------------

    sampler_text = (
        f"{sampler_class} "
        f"{sampler_module} "
        f"{batch_sampler_class} "
        f"{batch_sampler_module}"
    ).lower()

    sampler_infrastructure_valid = bool(
        (
            "uniformwithreplacementsampler"
            in sampler_text
        )
        or
        (
            "dpdataloader"
            in loader_class.lower()
        )
    )

    if not sampler_infrastructure_valid:

        raise RuntimeError(
            f"{dataset_id}: Opacus sampling infrastructure "
            "could not be validated."
        )

    # ----------------------------------------------------------------------------------------------
    # 7.8 Validate loader length and batch generation
    # ----------------------------------------------------------------------------------------------

    loader_length = int(
        len(
            private_loader
        )
    )

    loader_length_valid = bool(
        loader_length > 0
    )

    if not loader_length_valid:

        raise RuntimeError(
            f"{dataset_id}: DPDataLoader has zero batches."
        )

    private_iterator = iter(
        private_loader
    )

    first_batch = next(
        private_iterator
    )

    if not isinstance(
        first_batch,
        (tuple, list),
    ):

        raise RuntimeError(
            f"{dataset_id}: DPDataLoader output is not tuple/list."
        )

    first_batch_tensor = first_batch[0]

    observed_private_batch_size = int(
        first_batch_tensor.shape[0]
    )

    observed_private_batch_dimension = int(
        first_batch_tensor.shape[1]
    )

    batch_dimension_valid = bool(
        observed_private_batch_dimension
        ==
        input_dimension
    )

    batch_size_valid = bool(
        observed_private_batch_size > 0
        and
        observed_private_batch_size
        <=
        loader_validation_rows
    )

    # ----------------------------------------------------------------------------------------------
    # 7.9 Validate loader batch-size variability
    # ----------------------------------------------------------------------------------------------

    observed_batch_sizes = [
        observed_private_batch_size
    ]

    for _ in range(
        min(
            19,
            max(
                0,
                loader_length - 1,
            ),
        )
    ):

        try:

            next_batch = next(
                private_iterator
            )[0]

            observed_batch_sizes.append(
                int(
                    next_batch.shape[0]
                )
            )

        except StopIteration:

            break

    observed_batch_sizes = np.asarray(
        observed_batch_sizes,
        dtype=np.int64,
    )

    private_batch_sizes_valid = bool(
        (
            observed_batch_sizes > 0
        ).all()
        and
        (
            observed_batch_sizes
            <=
            loader_validation_rows
        ).all()
    )

    # ----------------------------------------------------------------------------------------------
    # 7.10 Verify private optimizer configuration
    # ----------------------------------------------------------------------------------------------

    optimizer_noise_multiplier = float(
        getattr(
            private_optimizer,
            "noise_multiplier",
        )
    )

    optimizer_max_grad_norm = float(
        getattr(
            private_optimizer,
            "max_grad_norm",
        )
    )

    optimizer_noise_match = bool(
        np.isclose(
            optimizer_noise_multiplier,
            noise_multiplier,
            rtol=0.0,
            atol=1e-12,
        )
    )

    optimizer_clip_match = bool(
        np.isclose(
            optimizer_max_grad_norm,
            float(MAX_GRAD_NORM),
            rtol=0.0,
            atol=1e-12,
        )
    )

    # ----------------------------------------------------------------------------------------------
    # 7.11 Final production loader validation
    # ----------------------------------------------------------------------------------------------

    production_loader_status = bool(
        all(
            [
                dp_loader_valid,
                sampler_infrastructure_valid,
                loader_length_valid,
                batch_dimension_valid,
                batch_size_valid,
                private_batch_sizes_valid,
                optimizer_noise_match,
                optimizer_clip_match,
            ]
        )
    )

    if not production_loader_status:

        raise RuntimeError(
            f"{dataset_id}: production Opacus DPDataLoader "
            "validation failed."
        )

    OPACUS_LOADER_RESULTS.append(
        {
            "dataset_id": dataset_id,
            "train_rows_reference": int(
                n_train
            ),
            "validation_rows": int(
                loader_validation_rows
            ),
            "nominal_batch_size": int(
                DP_BATCH_SIZE
            ),
            "noise_multiplier": float(
                noise_multiplier
            ),
            "optimizer_noise_multiplier": float(
                optimizer_noise_multiplier
            ),
            "max_grad_norm": float(
                MAX_GRAD_NORM
            ),
            "optimizer_max_grad_norm": float(
                optimizer_max_grad_norm
            ),
            "loader_class": str(
                loader_class
            ),
            "loader_module": str(
                loader_module
            ),
            "sampler_class": str(
                sampler_class
            ),
            "sampler_module": str(
                sampler_module
            ),
            "batch_sampler_class": str(
                batch_sampler_class
            ),
            "batch_sampler_module": str(
                batch_sampler_module
            ),
            "dp_loader_valid": bool(
                dp_loader_valid
            ),
            "sampler_infrastructure_valid": bool(
                sampler_infrastructure_valid
            ),
            "loader_length": int(
                loader_length
            ),
            "observed_first_batch_size": int(
                observed_private_batch_size
            ),
            "observed_batch_dimension": int(
                observed_private_batch_dimension
            ),
            "batch_dimension_valid": bool(
                batch_dimension_valid
            ),
            "batch_size_valid": bool(
                batch_size_valid
            ),
            "private_batch_sizes_valid": bool(
                private_batch_sizes_valid
            ),
            "optimizer_noise_match": bool(
                optimizer_noise_match
            ),
            "optimizer_clip_match": bool(
                optimizer_clip_match
            ),
            "status": "PASS",
        }
    )

    print(
        f"✓ {dataset_id:<20} : PASS | "
        f"DPDataLoader=YES | "
        f"q={DP_BATCH_SIZE / n_train:.8f} | "
        f"first_batch={observed_private_batch_size}"
    )


# --------------------------------------------------------------------------------------------------
# 8. Build validation DataFrames
# --------------------------------------------------------------------------------------------------

SECTION_10_DIAGNOSTIC_VALIDATION_DF = pd.DataFrame(
    [
        {
            "validation": "expected_batch_size",
            "status": (
                "PASS"
                if MEAN_VALID
                else "FAIL"
            ),
            "expected": float(
                EXPECTED_BATCH_SIZE
            ),
            "observed": float(
                OBSERVED_MEAN_BATCH_SIZE
            ),
        },
        {
            "validation": "batch_size_variability",
            "status": (
                "PASS"
                if STD_VALID
                else "FAIL"
            ),
            "expected": float(
                EXPECTED_STD_BATCH_SIZE
            ),
            "observed": float(
                OBSERVED_STD_BATCH_SIZE
            ),
        },
        {
            "validation": "batch_size_bounds",
            "status": (
                "PASS"
                if BATCH_RANGE_VALID
                else "FAIL"
            ),
            "expected": "0 <= B <= N",
            "observed": (
                f"{MIN_BATCH_SIZE} <= B <= "
                f"{MAX_BATCH_SIZE}"
            ),
        },
        {
            "validation": "independent_sampling",
            "status": (
                "PASS"
                if INDEPENDENT_SAMPLING_VALID
                else "FAIL"
            ),
            "expected": True,
            "observed": bool(
                INDEPENDENT_SAMPLING_VALID
            ),
        },
        {
            "validation": "zero_inclusion_behavior",
            "status": (
                "PASS"
                if ZERO_INCLUSION_BEHAVIOR_VALID
                else "FAIL"
            ),
            "expected": True,
            "observed": bool(
                ZERO_INCLUSION_BEHAVIOR_VALID
            ),
        },
        {
            "validation": "input_validation",
            "status": (
                "PASS"
                if INPUT_VALIDATION_PASS
                else "FAIL"
            ),
            "expected": True,
            "observed": bool(
                INPUT_VALIDATION_PASS
            ),
        },
    ]
)

SECTION_10_DATASET_VALIDATION_DF = pd.DataFrame(
    SECTION_10_RESULTS
)

SECTION_10_OPACUS_VALIDATION_DF = pd.DataFrame(
    OPACUS_LOADER_RESULTS
)


# --------------------------------------------------------------------------------------------------
# 9. Prepare canonical directories
# --------------------------------------------------------------------------------------------------

NB10_ROOT = (
    Path(
        "/content/drive/MyDrive/SPP_GAN_Research"
    )
    /
    "results"
    /
    "notebooks"
    /
    "notebook_10"
)

VALIDATION_ROOT = (
    NB10_ROOT
    /
    "validation"
)

METADATA_ROOT = (
    NB10_ROOT
    /
    "metadata"
)

VALIDATION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

METADATA_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------------------------------------------------------
# 10. Persist validation artifacts
# --------------------------------------------------------------------------------------------------

SECTION_10_DIAGNOSTIC_PATH = (
    VALIDATION_ROOT
    /
    "poisson_sampling_diagnostic_validation.csv"
)

SECTION_10_DATASET_PATH = (
    VALIDATION_ROOT
    /
    "poisson_sampling_dataset_validation.csv"
)

SECTION_10_OPACUS_PATH = (
    VALIDATION_ROOT
    /
    "opacus_dp_dataloader_validation.csv"
)

SECTION_10_DIAGNOSTIC_VALIDATION_DF.to_csv(
    SECTION_10_DIAGNOSTIC_PATH,
    index=False,
)

SECTION_10_DATASET_VALIDATION_DF.to_csv(
    SECTION_10_DATASET_PATH,
    index=False,
)

SECTION_10_OPACUS_VALIDATION_DF.to_csv(
    SECTION_10_OPACUS_PATH,
    index=False,
)

print(
    "\n✓ Sampling validation artifacts persisted."
)

print(
    f"  Diagnostic : {SECTION_10_DIAGNOSTIC_PATH}"
)

print(
    f"  Dataset    : {SECTION_10_DATASET_PATH}"
)

print(
    f"  Opacus     : {SECTION_10_OPACUS_PATH}"
)


# --------------------------------------------------------------------------------------------------
# 11. Build Section 10 manifest
# --------------------------------------------------------------------------------------------------

SECTION_10_MANIFEST = {
    "notebook": "10",
    "section": "10",
    "title": "Define Sampling Mechanism",
    "status": "PASS",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "sampling": {
        "production_mechanism": (
            "Opacus DPDataLoader"
        ),
        "privacy_sampling_model": (
            "Poisson sampling / independent Bernoulli inclusion"
        ),
        "diagnostic_sampler": (
            "poisson_sample_indices"
        ),
        "diagnostic_sampler_is_production": False,
        "nominal_batch_size": int(
            DP_BATCH_SIZE
        ),
    },

    "mathematical_validation": {
        "distribution": (
            "Binomial(N,q)"
        ),
        "expected_batch_size": (
            "N*q"
        ),
        "variance": (
            "N*q*(1-q)"
        ),
        "expected_batch_size_validation": bool(
            MEAN_VALID
        ),
        "batch_size_variability_validation": bool(
            STD_VALID
        ),
        "batch_size_bounds_validation": bool(
            BATCH_RANGE_VALID
        ),
        "independent_sampling_validation": bool(
            INDEPENDENT_SAMPLING_VALID
        ),
        "zero_inclusion_validation": bool(
            ZERO_INCLUSION_BEHAVIOR_VALID
        ),
        "input_validation": bool(
            INPUT_VALIDATION_PASS
        ),
    },

    "dataset_validation": {
        "dataset_count": int(
            len(
                SECTION_10_RESULTS
            )
        ),
        "all_datasets_passed": bool(
            all(
                row["status"] == "PASS"
                for row
                in SECTION_10_RESULTS
            )
        ),
    },

    "opacus_validation": {
        "dataset_count": int(
            len(
                OPACUS_LOADER_RESULTS
            )
        ),
        "all_datasets_passed": bool(
            all(
                row["status"] == "PASS"
                for row
                in OPACUS_LOADER_RESULTS
            )
        ),
        "dp_dataloader_verified": bool(
            all(
                row["dp_loader_valid"]
                for row
                in OPACUS_LOADER_RESULTS
            )
        ),
        "sampling_infrastructure_verified": bool(
            all(
                row["sampler_infrastructure_valid"]
                for row
                in OPACUS_LOADER_RESULTS
            )
        ),
        "optimizer_noise_verified": bool(
            all(
                row["optimizer_noise_match"]
                for row
                in OPACUS_LOADER_RESULTS
            )
        ),
        "optimizer_clipping_verified": bool(
            all(
                row["optimizer_clip_match"]
                for row
                in OPACUS_LOADER_RESULTS
            )
        ),
    },

    "artifacts": {
        "diagnostic_validation": str(
            SECTION_10_DIAGNOSTIC_PATH
        ),
        "dataset_validation": str(
            SECTION_10_DATASET_PATH
        ),
        "opacus_validation": str(
            SECTION_10_OPACUS_PATH
        ),
    },

    "privacy_boundary": {
        "diagnostic_sampler": (
            "mathematical validation only"
        ),
        "production_sampling": (
            "Opacus DPDataLoader"
        ),
        "dp_sgd_update": (
            "Notebook 10 Section 9"
        ),
        "privacy_accounting": (
            "Notebook 11"
        ),
        "achieved_epsilon": (
            "Notebook 11"
        ),
        "end_to_end_privacy_claim": False,
    },
}


# --------------------------------------------------------------------------------------------------
# 12. Validate manifest serializability
# --------------------------------------------------------------------------------------------------

try:

    json.dumps(
        SECTION_10_MANIFEST,
        indent=2,
        sort_keys=True,
    )

except (
    TypeError,
    ValueError,
) as exc:

    raise RuntimeError(
        "Section 10 manifest failed JSON serializability validation."
    ) from exc

print(
    "✓ Section 10 manifest JSON serializability : PASS"
)


# --------------------------------------------------------------------------------------------------
# 13. Persist manifest
# --------------------------------------------------------------------------------------------------

SECTION_10_MANIFEST_PATH = (
    METADATA_ROOT
    /
    "section_10_sampling_mechanism_manifest.json"
)

with open(
    SECTION_10_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        SECTION_10_MANIFEST,
        f,
        indent=2,
        sort_keys=True,
    )

print(
    f"✓ Section 10 manifest persisted: "
    f"{SECTION_10_MANIFEST_PATH}"
)


# --------------------------------------------------------------------------------------------------
# 14. Final verification
# --------------------------------------------------------------------------------------------------

DIAGNOSTIC_ALL_PASS = bool(
    (
        SECTION_10_DIAGNOSTIC_VALIDATION_DF[
            "status"
        ]
        ==
        "PASS"
    ).all()
)

DATASET_ALL_PASS = bool(
    (
        SECTION_10_DATASET_VALIDATION_DF[
            "status"
        ]
        ==
        "PASS"
    ).all()
)

OPACUS_ALL_PASS = bool(
    (
        SECTION_10_OPACUS_VALIDATION_DF[
            "status"
        ]
        ==
        "PASS"
    ).all()
)

ARTIFACTS_EXIST = bool(
    SECTION_10_DIAGNOSTIC_PATH.exists()
    and
    SECTION_10_DATASET_PATH.exists()
    and
    SECTION_10_OPACUS_PATH.exists()
    and
    SECTION_10_MANIFEST_PATH.exists()
)

SECTION_10_PASS = bool(
    DIAGNOSTIC_ALL_PASS
    and
    DATASET_ALL_PASS
    and
    OPACUS_ALL_PASS
    and
    ARTIFACTS_EXIST
)

if not SECTION_10_PASS:

    raise RuntimeError(
        "SECTION 10 FINAL VERIFICATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 15. Final output
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 10 FINAL VERIFICATION")
print("=" * 100)

print(
    "✓ Diagnostic Poisson sampler                 : PASS"
)

print(
    "✓ Expected batch-size validation             : PASS"
)

print(
    "✓ Batch-size variability validation          : PASS"
)

print(
    "✓ Batch-size bounds validation               : PASS"
)

print(
    "✓ Independent sampling validation            : PASS"
)

print(
    "✓ Zero-inclusion behavior validation         : PASS"
)

print(
    "✓ Sampler input validation                   : PASS"
)

print(
    "✓ Dataset-specific sampling validation       : PASS"
)

print(
    "✓ Opacus DPDataLoader                        : PASS"
)

print(
    "✓ Opacus sampling infrastructure             : PASS"
)

print(
    "✓ Opacus optimizer noise configuration      : PASS"
)

print(
    "✓ Opacus clipping configuration              : PASS"
)

print(
    "✓ Validation artifacts persisted             : PASS"
)

print(
    "✓ Section 10 manifest JSON serializability   : PASS"
)

print(
    "✓ Section 10 manifest persisted              : PASS"
)


# --------------------------------------------------------------------------------------------------
# 16. Privacy boundary
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("PRIVACY BOUNDARY")
print("-" * 100)

print(
    "Diagnostic sampler   : mathematical validation only"
)

print(
    "Production sampling  : Opacus DPDataLoader"
)

print(
    "DP-SGD update        : Notebook 10 Section 9"
)

print(
    "Privacy accounting   : Notebook 11"
)

print(
    "Achieved epsilon     : Notebook 11"
)

print(
    "End-to-end DP claim  : DISABLED"
)

print("\n" + "=" * 100)
print("SECTION 10 STATUS: PASS")
print("=" * 100)


10. DEFINE SAMPLING MECHANISM
✓ Sampling configuration validated.
✓ Configured sampling mechanism : Poisson
✓ Nominal DP batch size        : 128
✓ Diagnostic Bernoulli/Poisson sampler defined.
✓ Diagnostic Poisson validator defined.

----------------------------------------------------------------------------------------------------
POISSON SAMPLING MATHEMATICAL VALIDATION
----------------------------------------------------------------------------------------------------
✓ Expected batch size E[B]=Nq : PASS (expected=1000.000, observed=1001.046)
✓ Poisson batch-size variability : PASS (expected SD=30.000, observed SD=28.751)
✓ Batch-size bounds [0,N] : PASS (min=908, max=1082)
✓ Independent Poisson sampling draws : PASS
✓ Zero-inclusion behavior possible : PASS
✓ Sampler input validation : PASS

----------------------------------------------------------------------------------------------------
DATASET-SPECIFIC SAMPLING VALIDATION
-----------------------------------------------------

In [14]:
# ==================================================================================================
# 11. TEST GRADIENT PRIVATIZATION
# ==================================================================================================

print("\n" + "=" * 100)
print("11. TEST GRADIENT PRIVATIZATION")
print("=" * 100)

GRADIENT_TEST_ROWS = []

TEST_BATCH_SIZE = 8

for row in ARCHITECTURE_SUMMARY_DF.itertuples():

    dataset_id = row.dataset

    transformed_dimension = int(
        row.transformed_dimension
    )

    torch.manual_seed(
        MASTER_SEED
    )

    critic = SPPGANCritic(
        transformed_dimension
    ).to(
        DEVICE
    )

    dp_critic = (
        wrap_critic_for_per_sample_gradients(
            critic
        )
    )

    dp_critic.train()

    real_batch = torch.randn(
        TEST_BATCH_SIZE,
        transformed_dimension,
        device=DEVICE,
    )

    fake_batch = torch.randn(
        TEST_BATCH_SIZE,
        transformed_dimension,
        device=DEVICE,
    )

    dp_critic.zero_grad(
        set_to_none=True
    )

    loss, per_example_loss = (
        dp_discriminator_loss(
            private_critic=dp_critic,
            real_batch=real_batch,
            fake_batch=fake_batch,
        )
    )

    if per_example_loss.shape[0] != (
        TEST_BATCH_SIZE
    ):
        raise RuntimeError(
            f"Invalid per-example loss size for {dataset_id}."
        )

    loss.backward()

    gradients = (
        get_per_example_gradients(
            dp_critic
        )
    )

    norms = (
        per_example_gradient_norms(
            gradients
        )
    )

    clipped_gradients = (
        clip_per_example_gradients(
            gradients,
            MAX_GRAD_NORM,
        )
    )

    clipped_norms = (
        per_example_gradient_norms(
            clipped_gradients
        )
    )

    shape_pass = all(
        gradient.shape[0]
        == TEST_BATCH_SIZE
        for gradient in gradients.values()
    )

    finite_pass = all(
        torch.isfinite(
            gradient
        ).all().item()
        for gradient in gradients.values()
    )

    clipping_pass = (
        torch.isfinite(
            clipped_norms
        ).all().item()
        and
        float(
            clipped_norms.max()
            .detach()
            .cpu()
        )
        <= (
            MAX_GRAD_NORM
            + 1e-5
        )
    )

    nonzero_pass = (
        float(
            norms.max()
            .detach()
            .cpu()
        )
        > 0
    )

    status = (
        "PASS"
        if all([
            shape_pass,
            finite_pass,
            clipping_pass,
            nonzero_pass,
        ])
        else "FAIL"
    )

    GRADIENT_TEST_ROWS.append({
        "dataset": dataset_id,
        "transformed_dimension": transformed_dimension,
        "batch_size": TEST_BATCH_SIZE,
        "gradient_parameter_count": len(
            gradients
        ),
        "max_raw_gradient_norm": float(
            norms.max()
            .detach()
            .cpu()
        ),
        "max_clipped_gradient_norm": float(
            clipped_norms.max()
            .detach()
            .cpu()
        ),
        "shape_validation": (
            "PASS"
            if shape_pass
            else "FAIL"
        ),
        "finite_validation": (
            "PASS"
            if finite_pass
            else "FAIL"
        ),
        "clipping_validation": (
            "PASS"
            if clipping_pass
            else "FAIL"
        ),
        "nonzero_validation": (
            "PASS"
            if nonzero_pass
            else "FAIL"
        ),
        "status": status,
    })

    del (
        dp_critic,
        critic,
        real_batch,
        fake_batch,
        gradients,
        clipped_gradients,
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

GRADIENT_TEST_DF = pd.DataFrame(
    GRADIENT_TEST_ROWS
)

if not (
    GRADIENT_TEST_DF[
        "status"
    ]
    .eq("PASS")
    .all()
):
    raise RuntimeError(
        "Gradient privatization validation failed."
    )

GRADIENT_TEST_PATH = (
    DIRS["validation"] /
    "dp_gradient_privatization_tests.csv"
)

GRADIENT_TEST_DF.to_csv(
    GRADIENT_TEST_PATH,
    index=False,
)

display(
    GRADIENT_TEST_DF
)

print(
    f"✓ Gradient privatization tests saved:\n"
    f"  {GRADIENT_TEST_PATH}"
)

print(
    "SECTION 11 STATUS: PASS"
)


11. TEST GRADIENT PRIVATIZATION


/tmp/ipykernel_5574/770324283.py:70: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  loss.backward()


,dataset,transformed_dimension,batch_size,gradient_parameter_count,max_raw_gradient_norm,max_clipped_gradient_norm,shape_validation,finite_validation,clipping_validation,nonzero_validation,status
0,adult_income,105,8,6,6.049580,1.0,PASS,PASS,PASS,PASS,PASS
1,bank_marketing,51,8,6,5.771035,1.0,PASS,PASS,PASS,PASS,PASS
2,diabetes_130us,2329,8,6,13.850853,1.0,PASS,PASS,PASS,PASS,PASS


✓ Gradient privatization tests saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10/validation/dp_gradient_privatization_tests.csv
SECTION 11 STATUS: PASS


In [15]:
# ==================================================================================================
# 12. TEST NOISE INJECTION
# ==================================================================================================

print("\n" + "=" * 100)
print("12. TEST NOISE INJECTION")
print("=" * 100)

NOISE_TEST_ROWS = []

NOISE_TEST_SIZE = 8192

noise_generator = torch.Generator(
    device=DEVICE.type
)

noise_generator.manual_seed(
    MASTER_SEED
)

for row in PRIVACY_PARAMETER_DF.itertuples():

    reference = torch.zeros(
        NOISE_TEST_SIZE,
        device=DEVICE,
        dtype=torch.float32,
    )

    noise = gaussian_noise(
        reference_tensor=reference,
        noise_multiplier=row.noise_multiplier,
        max_grad_norm=MAX_GRAD_NORM,
        generator=noise_generator,
    )

    expected_std = (
        row.noise_multiplier
        *
        MAX_GRAD_NORM
    )

    observed_std = float(
        noise.std(
            unbiased=True
        )
        .detach()
        .cpu()
    )

    finite_pass = bool(
        torch.isfinite(
            noise
        ).all().item()
    )

    nonzero_fraction = float(
        (
            noise != 0
        )
        .float()
        .mean()
        .detach()
        .cpu()
    )

    nonzero_pass = (
        nonzero_fraction > 0
    )

    relative_error = (
        abs(
            observed_std
            -
            expected_std
        )
        /
        expected_std
    )

    scale_pass = (
        relative_error < 0.15
    )

    status = (
        "PASS"
        if all([
            finite_pass,
            nonzero_pass,
            scale_pass,
        ])
        else "FAIL"
    )

    NOISE_TEST_ROWS.append({
        "dataset": row.dataset,
        "noise_multiplier": row.noise_multiplier,
        "expected_std": expected_std,
        "observed_std": observed_std,
        "relative_std_error": relative_error,
        "finite": finite_pass,
        "nonzero": nonzero_pass,
        "scale_validation": scale_pass,
        "status": status,
    })

    del (
        reference,
        noise,
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

NOISE_TEST_DF = pd.DataFrame(
    NOISE_TEST_ROWS
)

if not (
    NOISE_TEST_DF[
        "status"
    ]
    .eq("PASS")
    .all()
):
    raise RuntimeError(
        "Gaussian noise validation failed."
    )

NOISE_TEST_PATH = (
    DIRS["validation"] /
    "dp_gaussian_noise_tests.csv"
)

NOISE_TEST_DF.to_csv(
    NOISE_TEST_PATH,
    index=False,
)

display(
    NOISE_TEST_DF
)

print(
    f"✓ Noise tests saved:\n"
    f"  {NOISE_TEST_PATH}"
)

print("SECTION 12 STATUS: PASS")


12. TEST NOISE INJECTION


,dataset,noise_multiplier,expected_std,observed_std,relative_std_error,finite,nonzero,scale_validation,status
0,adult_income,1.216221,1.216221,1.211632,0.003773,True,True,True,PASS
1,bank_marketing,1.251014,1.251014,1.234061,0.013551,True,True,True,PASS
2,diabetes_130us,0.953403,0.953403,0.954594,0.001249,True,True,True,PASS


✓ Noise tests saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10/validation/dp_gaussian_noise_tests.csv
SECTION 12 STATUS: PASS


In [16]:
# ==================================================================================================
# 13. VALIDATE PRIVACY MECHANISM
# ==================================================================================================

print("\n" + "=" * 100)
print("13. VALIDATE PRIVACY MECHANISM")
print("=" * 100)

PRIVACY_MECHANISM_ROWS = []

for row in ARCHITECTURE_SUMMARY_DF.itertuples():

    dataset_id = row.dataset

    transformed_dimension = int(
        row.transformed_dimension
    )

    # ----------------------------------------------------------------------------------------------
    # Critic
    # ----------------------------------------------------------------------------------------------

    critic = SPPGANCritic(
        transformed_dimension
    ).to(
        DEVICE
    )

    critic.train()

    validation_errors = (
        ModuleValidator.validate(
            critic,
            strict=False,
        )
    )

    critic_compatible = (
        len(validation_errors) == 0
    )

    # ----------------------------------------------------------------------------------------------
    # Dummy training data
    # ----------------------------------------------------------------------------------------------

    dummy_rows = 32

    dummy_x = torch.randn(
        dummy_rows,
        transformed_dimension,
    )

    dummy_dataset = (
        torch.utils.data.TensorDataset(
            dummy_x
        )
    )

    dummy_loader = (
        torch.utils.data.DataLoader(
            dummy_dataset,
            batch_size=8,
            shuffle=False,
            num_workers=0,
            pin_memory=False,
        )
    )

    optimizer = torch.optim.Adam(
        critic.parameters(),
        lr=2e-4,
    )

    # ----------------------------------------------------------------------------------------------
    # PrivacyEngine compatibility
    # ----------------------------------------------------------------------------------------------

    privacy_engine = PrivacyEngine(
        accountant="rdp"
    )

    try:

        engine_compatible = bool(
            privacy_engine.is_compatible(
                module=critic,
                optimizer=optimizer,
                data_loader=dummy_loader,
            )
        )

    except Exception:

        engine_compatible = False

    # ----------------------------------------------------------------------------------------------
    # Poisson diagnostic
    # ----------------------------------------------------------------------------------------------

    sample_rate = (
        DP_BATCH_SIZE
        /
        TRAIN_ROWS[dataset_id]
    )

    poisson_test = (
        validate_poisson_sampling(
            n_samples=TRAIN_ROWS[dataset_id],
            sample_rate=sample_rate,
            repetitions=50,
        )
    )

    poisson_pass = (
        poisson_test[
            "mean_batch_size"
        ] > 0
        and
        poisson_test[
            "mean_batch_size"
        ] < TRAIN_ROWS[dataset_id]
        and
        poisson_test[
            "max_batch_size"
        ] > 0
    )

    # ----------------------------------------------------------------------------------------------
    # Actual Opacus wrapping test
    # ----------------------------------------------------------------------------------------------

    try:

        (
            private_critic,
            private_optimizer,
            private_loader,
        ) = privacy_engine.make_private(
            module=critic,
            optimizer=optimizer,
            data_loader=dummy_loader,
            noise_multiplier=float(
                PRIVACY_PARAMETER_DF.loc[
                    PRIVACY_PARAMETER_DF[
                        "dataset"
                    ]
                    == dataset_id,
                    "noise_multiplier",
                ].iloc[0]
            ),
            max_grad_norm=MAX_GRAD_NORM,
            batch_first=True,
            loss_reduction="mean",
            poisson_sampling=True,
            clipping="flat",
            grad_sample_mode="hooks",
            wrap_model=True,
        )

        wrapping_pass = (
            private_critic is not None
            and
            private_optimizer is not None
            and
            private_loader is not None
        )

    except Exception as exc:

        wrapping_pass = False

        print(
            f"⚠ DP wrapping failed for {dataset_id}: "
            f"{exc}"
        )

    status = (
        "PASS"
        if all([
            critic_compatible,
            engine_compatible,
            poisson_pass,
            wrapping_pass,
        ])
        else "FAIL"
    )

    PRIVACY_MECHANISM_ROWS.append({
        "dataset": dataset_id,
        "transformed_dimension": transformed_dimension,
        "critic_compatible": critic_compatible,
        "privacy_engine_compatible": engine_compatible,
        "poisson_sampling": poisson_pass,
        "dp_wrapping": wrapping_pass,
        "status": status,
    })

    del (
        critic,
        optimizer,
        dummy_loader,
        dummy_dataset,
        dummy_x,
        privacy_engine,
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

PRIVACY_MECHANISM_DF = pd.DataFrame(
    PRIVACY_MECHANISM_ROWS
)

if not (
    PRIVACY_MECHANISM_DF[
        "status"
    ]
    .eq("PASS")
    .all()
):
    raise RuntimeError(
        "Privacy mechanism validation failed."
    )

PRIVACY_MECHANISM_PATH = (
    DIRS["validation"] /
    "dp_privacy_mechanism_validation.csv"
)

PRIVACY_MECHANISM_DF.to_csv(
    PRIVACY_MECHANISM_PATH,
    index=False,
)

display(
    PRIVACY_MECHANISM_DF
)

print(
    f"✓ Privacy mechanism validation saved:\n"
    f"  {PRIVACY_MECHANISM_PATH}"
)

print("SECTION 13 STATUS: PASS")


13. VALIDATE PRIVACY MECHANISM


/usr/local/lib/python3.13/dist-packages/opacus/privacy_engine.py:98: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(


,dataset,transformed_dimension,critic_compatible,privacy_engine_compatible,poisson_sampling,dp_wrapping,status
0,adult_income,105,True,True,True,True,PASS
1,bank_marketing,51,True,True,True,True,PASS
2,diabetes_130us,2329,True,True,True,True,PASS


✓ Privacy mechanism validation saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10/validation/dp_privacy_mechanism_validation.csv
SECTION 13 STATUS: PASS


In [18]:
# ==================================================================================================
# 14. RECORD PRIVACY METADATA
# ==================================================================================================

print("\n" + "=" * 100)
print("14. RECORD PRIVACY METADATA")
print("=" * 100)

# -----------------------------------------------------------------------------------------------
# 1. Validate canonical privacy parameter source
# -----------------------------------------------------------------------------------------------

REQUIRED_PRIVACY_COLUMNS = {
    "dataset",
    "n_train",
    "target_epsilon",
    "delta",
    "batch_size_nominal",
    "poisson_sample_rate",
    "epochs",
    "nominal_steps_per_epoch",
    "nominal_total_steps",
    "max_grad_norm",
    "noise_multiplier",
    "accountant",
    "sampling",
    "clipping",
    "loss_reduction",
    "achieved_epsilon_status",
}

missing_privacy_columns = (
    REQUIRED_PRIVACY_COLUMNS
    -
    set(PRIVACY_PARAMETER_DF.columns)
)

if missing_privacy_columns:
    raise RuntimeError(
        "PRIVACY_PARAMETER_DF is missing required columns: "
        f"{sorted(missing_privacy_columns)}"
    )

if not set(DATASET_IDS).issubset(
    set(PRIVACY_PARAMETER_DF["dataset"])
):
    raise RuntimeError(
        "PRIVACY_PARAMETER_DF does not contain all canonical datasets."
    )

if len(PRIVACY_PARAMETER_DF) != len(DATASET_IDS):
    raise RuntimeError(
        "Expected exactly one privacy-parameter row per dataset."
    )

# -----------------------------------------------------------------------------------------------
# 2. Build privacy metadata from canonical Section 5 parameters
# -----------------------------------------------------------------------------------------------

PRIVACY_METADATA_ROWS = []

for row in PRIVACY_PARAMETER_DF.itertuples(index=False):

    dataset_id = str(row.dataset)

    if dataset_id not in DATASET_IDS:
        raise RuntimeError(
            f"Unexpected dataset in privacy metadata: {dataset_id}"
        )

    PRIVACY_METADATA_ROWS.append({

        # ---------------------------------------------------------------------------------------
        # Dataset and privacy parameters
        # ---------------------------------------------------------------------------------------

        "dataset":
            dataset_id,

        "n_train":
            int(row.n_train),

        "target_epsilon":
            float(row.target_epsilon),

        "delta":
            float(row.delta),

        # Preserve publication-facing names while sourcing directly
        # from the canonical Section 5 fields.
        "batch_size":
            int(row.batch_size_nominal),

        "sample_rate":
            float(row.poisson_sample_rate),

        "epochs":
            int(row.epochs),

        "nominal_steps_per_epoch":
            float(row.nominal_steps_per_epoch),

        "nominal_total_steps":
            float(row.nominal_total_steps),

        "max_grad_norm":
            float(row.max_grad_norm),

        "noise_multiplier":
            float(row.noise_multiplier),

        # ---------------------------------------------------------------------------------------
        # Mechanism configuration
        # ---------------------------------------------------------------------------------------

        "accountant":
            str(row.accountant),

        "sampling":
            str(row.sampling),

        "clipping":
            str(row.clipping),

        "loss_reduction":
            str(row.loss_reduction),

        "protected_component":
            "SPP-GAN discriminator",

        "per_example_gradients":
            True,

        # ---------------------------------------------------------------------------------------
        # Privacy boundary
        # ---------------------------------------------------------------------------------------

        "generator_private":
            False,

        "statistical_guidance_private":
            False,

        "preprocessing_private":
            False,

        # ---------------------------------------------------------------------------------------
        # Achieved privacy accounting
        # ---------------------------------------------------------------------------------------

        "achieved_epsilon":
            None,

        "achieved_epsilon_status":
            "DEFERRED_TO_NOTEBOOK_11",

        # ---------------------------------------------------------------------------------------
        # Downstream execution status
        # ---------------------------------------------------------------------------------------

        "training_status":
            "DEFERRED_TO_NOTEBOOK_12",

        "synthetic_generation_status":
            "DEFERRED_TO_NOTEBOOK_13",

        # ---------------------------------------------------------------------------------------
        # Formal end-to-end privacy claim
        # ---------------------------------------------------------------------------------------

        "end_to_end_privacy_claim":
            False,

        "status":
            "PASS",
    })

# -----------------------------------------------------------------------------------------------
# 3. Create metadata dataframe
# -----------------------------------------------------------------------------------------------

PRIVACY_METADATA_DF = pd.DataFrame(
    PRIVACY_METADATA_ROWS
)

# -----------------------------------------------------------------------------------------------
# 4. Validate metadata
# -----------------------------------------------------------------------------------------------

EXPECTED_METADATA_DATASETS = set(DATASET_IDS)

if set(
    PRIVACY_METADATA_DF["dataset"]
) != EXPECTED_METADATA_DATASETS:

    raise RuntimeError(
        "Privacy metadata dataset coverage does not match DATASET_IDS."
    )

if len(PRIVACY_METADATA_DF) != len(DATASET_IDS):
    raise RuntimeError(
        "Privacy metadata must contain exactly one row per dataset."
    )

numeric_metadata_columns = [
    "n_train",
    "target_epsilon",
    "delta",
    "batch_size",
    "sample_rate",
    "epochs",
    "nominal_steps_per_epoch",
    "nominal_total_steps",
    "max_grad_norm",
    "noise_multiplier",
]

for column in numeric_metadata_columns:

    values = pd.to_numeric(
        PRIVACY_METADATA_DF[column],
        errors="coerce",
    )

    if not np.isfinite(
        values.to_numpy(dtype=float)
    ).all():

        raise RuntimeError(
            f"Non-finite privacy metadata detected in column: {column}"
        )

# -----------------------------------------------------------------------------------------------
# 5. Validate privacy boundary
# -----------------------------------------------------------------------------------------------

if not PRIVACY_METADATA_DF[
    "generator_private"
].eq(False).all():

    raise RuntimeError(
        "Generator privacy boundary is inconsistent."
    )

if not PRIVACY_METADATA_DF[
    "statistical_guidance_private"
].eq(False).all():

    raise RuntimeError(
        "Statistical-guidance privacy boundary is inconsistent."
    )

if not PRIVACY_METADATA_DF[
    "preprocessing_private"
].eq(False).all():

    raise RuntimeError(
        "Preprocessing privacy boundary is inconsistent."
    )

if not PRIVACY_METADATA_DF[
    "end_to_end_privacy_claim"
].eq(False).all():

    raise RuntimeError(
        "End-to-end privacy claim must remain disabled in Notebook 10."
    )

if not PRIVACY_METADATA_DF[
    "achieved_epsilon_status"
].eq(
    "DEFERRED_TO_NOTEBOOK_11"
).all():

    raise RuntimeError(
        "Achieved epsilon must remain deferred to Notebook 11."
    )

# -----------------------------------------------------------------------------------------------
# 6. Persist metadata
# -----------------------------------------------------------------------------------------------

PRIVACY_METADATA_PATH = (
    DIRS["metadata"]
    /
    "sppgan_privacy_metadata.csv"
)

PRIVACY_METADATA_DF.to_csv(
    PRIVACY_METADATA_PATH,
    index=False,
)

# -----------------------------------------------------------------------------------------------
# 7. Display
# -----------------------------------------------------------------------------------------------

display(
    PRIVACY_METADATA_DF
)

print(
    f"✓ Privacy metadata saved:\n"
    f"  {PRIVACY_METADATA_PATH}"
)

print(
    f"✓ Dataset coverage validated: "
    f"{len(PRIVACY_METADATA_DF)} datasets"
)

print(
    "✓ Achieved epsilon correctly deferred to Notebook 11."
)

print(
    "✓ End-to-end privacy claim remains disabled."
)

print(
    "SECTION 14 STATUS: PASS"
)


14. RECORD PRIVACY METADATA


,dataset,n_train,target_epsilon,delta,batch_size,sample_rate,epochs,nominal_steps_per_epoch,nominal_total_steps,max_grad_norm,...,per_example_gradients,generator_private,statistical_guidance_private,preprocessing_private,achieved_epsilon,achieved_epsilon_status,training_status,synthetic_generation_status,end_to_end_privacy_claim,status
0,adult_income,34189,5.0,0.00001,128,0.003744,300,267.101562,80130.46875,1.0,...,True,False,False,False,None,DEFERRED_TO_NOTEBOOK_11,DEFERRED_TO_NOTEBOOK_12,DEFERRED_TO_NOTEBOOK_13,False,PASS
1,bank_marketing,31647,5.0,0.00001,128,0.004045,300,247.242188,74172.65625,1.0,...,True,False,False,False,None,DEFERRED_TO_NOTEBOOK_11,DEFERRED_TO_NOTEBOOK_12,DEFERRED_TO_NOTEBOOK_13,False,PASS
2,diabetes_130us,71236,5.0,0.00001,128,0.001797,300,556.531250,166959.37500,1.0,...,True,False,False,False,None,DEFERRED_TO_NOTEBOOK_11,DEFERRED_TO_NOTEBOOK_12,DEFERRED_TO_NOTEBOOK_13,False,PASS


✓ Privacy metadata saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10/metadata/sppgan_privacy_metadata.csv
✓ Dataset coverage validated: 3 datasets
✓ Achieved epsilon correctly deferred to Notebook 11.
✓ End-to-end privacy claim remains disabled.
SECTION 14 STATUS: PASS


In [19]:
# ==================================================================================================
# 15. SAVE DP MODULE
# ==================================================================================================

print("\n" + "=" * 100)
print("15. SAVE DP MODULE")
print("=" * 100)

DP_MODULE_PATH = (
    DIRS["models"] /
    "sppgan_differential_privacy.py"
)

DP_MODULE_SOURCE = r'''
# ==================================================================================================
# SPP-GAN DIFFERENTIAL PRIVACY MODULE
# ==================================================================================================

import torch
import torch.nn as nn

from opacus import PrivacyEngine
from opacus.grad_sample import GradSampleModule


class SPPGANCritic(nn.Module):

    def __init__(self, input_dim):

        super().__init__()

        self.input_dim = int(
            input_dim
        )

        self.network = nn.Sequential(
            nn.Linear(
                self.input_dim,
                256,
            ),
            nn.LeakyReLU(
                0.2
            ),
            nn.Linear(
                256,
                256,
            ),
            nn.LeakyReLU(
                0.2
            ),
            nn.Linear(
                256,
                1,
            ),
        )

    def forward(self, x):

        return self.network(x)


def wrap_critic_for_per_sample_gradients(
    critic,
):

    return GradSampleModule(
        critic,
        batch_first=True,
        loss_reduction="mean",
        strict=True,
        force_functorch=False,
    )


def dp_discriminator_loss(
    private_critic,
    real_batch,
    fake_batch,
):

    if real_batch.shape != fake_batch.shape:

        raise ValueError(
            "Real and fake batches must have identical shapes."
        )

    real_scores = (
        private_critic(
            real_batch
        )
        .reshape(-1)
    )

    fake_scores = (
        private_critic(
            fake_batch.detach()
        )
        .reshape(-1)
    )

    per_example_loss = (
        fake_scores
        -
        real_scores
    )

    return (
        per_example_loss.mean(),
        per_example_loss,
    )


def make_private_discriminator(
    critic,
    optimizer,
    data_loader,
    noise_multiplier,
    max_grad_norm,
):

    privacy_engine = PrivacyEngine(
        accountant="rdp"
    )

    (
        private_critic,
        private_optimizer,
        private_loader,
    ) = privacy_engine.make_private(
        module=critic,
        optimizer=optimizer,
        data_loader=data_loader,
        noise_multiplier=float(
            noise_multiplier
        ),
        max_grad_norm=float(
            max_grad_norm
        ),
        batch_first=True,
        loss_reduction="mean",
        poisson_sampling=True,
        clipping="flat",
        grad_sample_mode="hooks",
        wrap_model=True,
    )

    return (
        private_critic,
        private_optimizer,
        private_loader,
        privacy_engine,
    )


def dp_discriminator_step(
    private_critic,
    private_optimizer,
    real_batch,
    fake_batch,
):

    private_optimizer.zero_grad(
        set_to_none=True
    )

    loss, per_example_loss = (
        dp_discriminator_loss(
            private_critic,
            real_batch,
            fake_batch,
        )
    )

    loss.backward()

    private_optimizer.step()

    return {
        "loss": loss.detach(),
        "per_example_loss":
            per_example_loss.detach(),
    }
'''

with open(
    DP_MODULE_PATH,
    "w",
    encoding="utf-8",
) as f:

    f.write(
        DP_MODULE_SOURCE
    )

# --------------------------------------------------------------------------------------------------
# Syntax validation
# --------------------------------------------------------------------------------------------------

with open(
    DP_MODULE_PATH,
    "r",
    encoding="utf-8",
) as f:

    source = f.read()

compile(
    source,
    str(DP_MODULE_PATH),
    "exec",
)

print(
    f"✓ DP module saved:\n"
    f"  {DP_MODULE_PATH}"
)

print(
    "✓ Python syntax validation: PASS"
)

print("SECTION 15 STATUS: PASS")


15. SAVE DP MODULE
✓ DP module saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10/models/sppgan_differential_privacy.py
✓ Python syntax validation: PASS
SECTION 15 STATUS: PASS


In [20]:
# ==================================================================================================
# 16. SAVE PRIVACY CONFIGURATION
# ==================================================================================================

print("\n" + "=" * 100)
print("16. SAVE PRIVACY CONFIGURATION")
print("=" * 100)

PERSISTED_PRIVACY_CONFIGURATION = {

    "configuration_version":
        "1.0",

    "notebook":
        NOTEBOOK_ID,

    "name":
        NOTEBOOK_NAME,

    "framework":
        FRAMEWORK_NAME,

    "privacy_definition": {

        "mechanism":
            "DP-SGD",

        "protected_component":
            "SPP-GAN discriminator",

        "gradient_representation":
            "per-example discriminator gradients",

        "clipping":
            "flat L2",

        "noise":
            "Gaussian",

        "sampling":
            "Poisson",

        "accountant":
            "RDP",

        "generator_private":
            False,

        "statistical_guidance_private":
            False,

        "preprocessing_private":
            False,

        "end_to_end_privacy_claim":
            False,
    },

    "parameters": {

        "target_epsilon":
            TARGET_EPSILON,

        "delta_rule":
            "min(1e-5, 1/N_train)",

        "max_grad_norm":
            MAX_GRAD_NORM,

        "batch_size":
            DP_BATCH_SIZE,

        "epochs":
            DP_EPOCHS,

        "accountant":
            ACCOUNTANT,

        "sampling":
            SAMPLING_MECHANISM,

        "clipping":
            CLIPPING_MECHANISM,

        "loss_reduction":
            LOSS_REDUCTION,

        "grad_sample_mode":
            GRAD_SAMPLE_MODE,
    },

    "dataset_parameters":
        PRIVACY_PARAMETER_DF.to_dict(
            orient="records"
        ),

    "source_dependencies": {

        "notebook_08_architecture":
            str(
                NB08_ROOT
            ),

        "notebook_09_statistical_guidance":
            str(
                NB09_ROOT
            ),
    },

    "downstream": {

        "notebook_11":
            "Formal RDP privacy accounting",

        "notebook_12":
            "SPP-GAN DP training",

        "notebook_13":
            "Synthetic data generation",
    },

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}

PRIVACY_CONFIGURATION_PATH = (
    DIRS["configuration"] /
    "sppgan_privacy_configuration.json"
)

with open(
    PRIVACY_CONFIGURATION_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        PERSISTED_PRIVACY_CONFIGURATION,
        f,
        indent=2,
        ensure_ascii=False,
    )

# --------------------------------------------------------------------------------------------------
# Reload
# --------------------------------------------------------------------------------------------------

with open(
    PRIVACY_CONFIGURATION_PATH,
    "r",
    encoding="utf-8",
) as f:

    RELOADED_PRIVACY_CONFIGURATION = json.load(
        f
    )

REQUIRED_PRIVACY_CONFIGURATION_KEYS = {
    "configuration_version",
    "notebook",
    "name",
    "framework",
    "privacy_definition",
    "parameters",
    "dataset_parameters",
    "source_dependencies",
    "downstream",
    "created_utc",
}

missing_keys = (
    REQUIRED_PRIVACY_CONFIGURATION_KEYS
    -
    set(
        RELOADED_PRIVACY_CONFIGURATION.keys()
    )
)

if missing_keys:
    raise RuntimeError(
        "Persisted privacy configuration is missing:\n"
        f"{sorted(missing_keys)}"
    )

print(
    f"✓ Privacy configuration saved:\n"
    f"  {PRIVACY_CONFIGURATION_PATH}"
)

print(
    "✓ Configuration reload validation: PASS"
)

print("SECTION 16 STATUS: PASS")


16. SAVE PRIVACY CONFIGURATION
✓ Privacy configuration saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10/configuration/sppgan_privacy_configuration.json
✓ Configuration reload validation: PASS
SECTION 16 STATUS: PASS


In [21]:
# ==================================================================================================
# 17. SAVE PRIVACY MANIFEST
# ==================================================================================================

print("\n" + "=" * 100)
print("17. SAVE PRIVACY MANIFEST")
print("=" * 100)


def sha256_file(
    path,
):
    """
    RAM-safe SHA-256 calculation.
    """

    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


PRIVACY_ARTIFACTS = [
    DP_MODULE_PATH,
    PRIVACY_CONFIGURATION_PATH,
    PRIVACY_METADATA_PATH,
    GRADIENT_TEST_PATH,
    NOISE_TEST_PATH,
    PRIVACY_MECHANISM_PATH,
]

PRIVACY_ARTIFACT_REGISTRY = []

for artifact_path in PRIVACY_ARTIFACTS:

    artifact_path = Path(
        artifact_path
    )

    if not artifact_path.exists():

        raise FileNotFoundError(
            f"Required artifact missing:\n"
            f"{artifact_path}"
        )

    PRIVACY_ARTIFACT_REGISTRY.append({

        "artifact":
            artifact_path.name,

        "absolute_path":
            str(
                artifact_path
            ),

        "exists":
            True,

        "size_bytes":
            artifact_path.stat().st_size,

        "sha256":
            sha256_file(
                artifact_path
            ),

    })

PRIVACY_ARTIFACT_REGISTRY_DF = pd.DataFrame(
    PRIVACY_ARTIFACT_REGISTRY
)

PRIVACY_ARTIFACT_REGISTRY_PATH = (
    DIRS["metadata"] /
    "sppgan_privacy_artifact_registry.csv"
)

PRIVACY_ARTIFACT_REGISTRY_DF.to_csv(
    PRIVACY_ARTIFACT_REGISTRY_PATH,
    index=False,
)

display(
    PRIVACY_ARTIFACT_REGISTRY_DF
)

print(
    f"✓ Privacy artifact registry saved:\n"
    f"  {PRIVACY_ARTIFACT_REGISTRY_PATH}"
)

print(
    f"✓ Registered artifacts : "
    f"{len(PRIVACY_ARTIFACT_REGISTRY_DF)}"
)

print("SECTION 17 STATUS: PASS")


17. SAVE PRIVACY MANIFEST


,artifact,absolute_path,exists,size_bytes,sha256
0,sppgan_differential_privacy.py,/content/drive/MyDrive/SPP_GAN_Research/result...,True,3154,aa488ff70f911a2b9f23a112e8d5a70ec98ad92de19951...
1,sppgan_privacy_configuration.json,/content/drive/MyDrive/SPP_GAN_Research/result...,True,4122,bad9208a21f1729d494b3c5d1019be442ff0476edb364c...
2,sppgan_privacy_metadata.csv,/content/drive/MyDrive/SPP_GAN_Research/result...,True,1184,a2bb67937ff1b95f04328db367f039e0e0c17ac040cbef...
3,dp_gradient_privatization_tests.csv,/content/drive/MyDrive/SPP_GAN_Research/result...,True,419,897952a59f4c835f44f9f5aa077f3a97757f882978aac0...
4,dp_gaussian_noise_tests.csv,/content/drive/MyDrive/SPP_GAN_Research/result...,True,448,cdea78396a07ed972f28a94dd938634eb304fec627bdc9...
5,dp_privacy_mechanism_validation.csv,/content/drive/MyDrive/SPP_GAN_Research/result...,True,240,b8afc6ffe384971ff09e705fa4a0b19cdb4e4e1f76e38b...


✓ Privacy artifact registry saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10/metadata/sppgan_privacy_artifact_registry.csv
✓ Registered artifacts : 6
SECTION 17 STATUS: PASS


In [22]:
# ==================================================================================================
# 18. FINAL VERIFICATION
# ==================================================================================================

print("\n" + "=" * 100)
print("18. FINAL VERIFICATION")
print("=" * 100)

FINAL_CHECKS = []

def add_final_check(
    name,
    condition,
):
    FINAL_CHECKS.append({
        "check": name,
        "status": (
            "PASS"
            if bool(condition)
            else "FAIL"
        ),
    })


# --------------------------------------------------------------------------------------------------
# Root / dependencies
# --------------------------------------------------------------------------------------------------

add_final_check(
    "Project root exists",
    PROJECT_ROOT.exists(),
)

add_final_check(
    "Notebook 08 architecture loaded",
    len(
        ARCHITECTURE_SUMMARY_DF
    ) == len(DATASET_IDS),
)

add_final_check(
    "Notebook 09 configuration loaded",
    NB09_CONFIGURATION_PATH.exists(),
)

# --------------------------------------------------------------------------------------------------
# Privacy parameters
# --------------------------------------------------------------------------------------------------

add_final_check(
    "Target epsilon positive",
    TARGET_EPSILON > 0,
)

add_final_check(
    "Maximum gradient norm positive",
    MAX_GRAD_NORM > 0,
)

add_final_check(
    "Poisson sampling selected",
    SAMPLING_MECHANISM == "poisson",
)

add_final_check(
    "Flat clipping selected",
    CLIPPING_MECHANISM == "flat",
)

add_final_check(
    "RDP accountant selected",
    ACCOUNTANT == "rdp",
)

# --------------------------------------------------------------------------------------------------
# Validation results
# --------------------------------------------------------------------------------------------------

add_final_check(
    "Gradient privatization tests pass",
    GRADIENT_TEST_DF[
        "status"
    ].eq("PASS").all(),
)

add_final_check(
    "Gaussian noise tests pass",
    NOISE_TEST_DF[
        "status"
    ].eq("PASS").all(),
)

add_final_check(
    "Privacy mechanism tests pass",
    PRIVACY_MECHANISM_DF[
        "status"
    ].eq("PASS").all(),
)

# --------------------------------------------------------------------------------------------------
# Artifacts
# --------------------------------------------------------------------------------------------------

add_final_check(
    "DP module exists",
    DP_MODULE_PATH.exists(),
)

add_final_check(
    "Privacy configuration exists",
    PRIVACY_CONFIGURATION_PATH.exists(),
)

add_final_check(
    "Privacy metadata exists",
    PRIVACY_METADATA_PATH.exists(),
)

add_final_check(
    "Artifact registry exists",
    PRIVACY_ARTIFACT_REGISTRY_PATH.exists(),
)

FINAL_CHECKS_DF = pd.DataFrame(
    FINAL_CHECKS
)

if not (
    FINAL_CHECKS_DF[
        "status"
    ]
    .eq("PASS")
    .all()
):

    display(
        FINAL_CHECKS_DF
    )

    raise RuntimeError(
        "Notebook 10 final verification failed."
    )

FINAL_VERIFICATION_PATH = (
    DIRS["validation"] /
    "sppgan_privacy_final_verification.csv"
)

FINAL_CHECKS_DF.to_csv(
    FINAL_VERIFICATION_PATH,
    index=False,
)

display(
    FINAL_CHECKS_DF
)

print(
    f"✓ Final verification saved:\n"
    f"  {FINAL_VERIFICATION_PATH}"
)

print(
    "✓ All final checks: PASS"
)

print("SECTION 18 STATUS: PASS")


18. FINAL VERIFICATION


,check,status
0,Project root exists,PASS
1,Notebook 08 architecture loaded,PASS
2,Notebook 09 configuration loaded,PASS
3,Target epsilon positive,PASS
4,Maximum gradient norm positive,PASS
5,Poisson sampling selected,PASS
6,Flat clipping selected,PASS
7,RDP accountant selected,PASS
8,Gradient privatization tests pass,PASS
9,Gaussian noise tests pass,PASS


✓ Final verification saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10/validation/sppgan_privacy_final_verification.csv
✓ All final checks: PASS
SECTION 18 STATUS: PASS


In [23]:
# ==================================================================================================
# 19. COMPLETION SUMMARY
# ==================================================================================================

print("\n" + "=" * 100)
print("19. COMPLETION SUMMARY")
print("=" * 100)

COMPLETION_MANIFEST = {

    "notebook":
        NOTEBOOK_ID,

    "name":
        NOTEBOOK_NAME,

    "framework":
        FRAMEWORK_NAME,

    "status":
        "PASS",

    "project_root":
        str(
            PROJECT_ROOT
        ),

    "datasets_registered":
        len(
            DATASET_IDS
        ),

    "datasets":
        DATASET_IDS,

    "privacy_mechanism": {

        "mechanism":
            "DP-SGD",

        "protected_component":
            "SPP-GAN discriminator",

        "per_example_gradients":
            "PASS",

        "flat_gradient_clipping":
            "PASS",

        "gaussian_noise":
            "PASS",

        "poisson_sampling":
            "PASS",

        "accountant":
            "RDP",
    },

    "parameters": {

        "target_epsilon":
            TARGET_EPSILON,

        "delta_rule":
            "min(1e-5, 1/N_train)",

        "max_grad_norm":
            MAX_GRAD_NORM,

        "batch_size":
            DP_BATCH_SIZE,

        "epochs":
            DP_EPOCHS,
    },

    "validation": {

        "architecture":
            "PASS",

        "privacy_parameters":
            "PASS",

        "gradient_privatization":
            "PASS",

        "noise_injection":
            "PASS",

        "privacy_mechanism":
            "PASS",

        "final_verification":
            "PASS",
    },

    "training_performed":
        False,

    "privacy_accounting_performed":
        False,

    "achieved_epsilon_reported":
        False,

    "synthetic_generation_performed":
        False,

    "generator_private":
        False,

    "statistical_guidance_private":
        False,

    "preprocessing_private":
        False,

    "end_to_end_privacy_claim":
        False,

    "artifacts": {

        "dp_module":
            str(
                DP_MODULE_PATH
            ),

        "privacy_configuration":
            str(
                PRIVACY_CONFIGURATION_PATH
            ),

        "privacy_metadata":
            str(
                PRIVACY_METADATA_PATH
            ),

        "gradient_tests":
            str(
                GRADIENT_TEST_PATH
            ),

        "noise_tests":
            str(
                NOISE_TEST_PATH
            ),

        "mechanism_validation":
            str(
                PRIVACY_MECHANISM_PATH
            ),

        "artifact_registry":
            str(
                PRIVACY_ARTIFACT_REGISTRY_PATH
            ),

        "final_verification":
            str(
                FINAL_VERIFICATION_PATH
            ),
    },

    "source_dependencies": {

        "notebook_08":
            str(
                NB08_ROOT
            ),

        "notebook_09":
            str(
                NB09_ROOT
            ),
    },

    "downstream": {

        "notebook_11":
            "SPP-GAN Privacy Accounting",

        "notebook_12":
            "SPP-GAN DP Training",

        "notebook_13":
            "SPP-GAN Synthetic Generation",
    },

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}

COMPLETION_PATH = (
    DIRS["validation"] /
    "sppgan_notebook_10_completion.json"
)

with open(
    COMPLETION_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        COMPLETION_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False,
    )

# --------------------------------------------------------------------------------------------------
# Final display
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("NOTEBOOK 10 — FINAL STATUS")
print("=" * 100)

print(
    f"Framework                       : "
    f"{FRAMEWORK_NAME}"
)

print(
    f"Datasets                        : "
    f"{len(DATASET_IDS)}"
)

print(
    f"Privacy mechanism               : DP-SGD"
)

print(
    f"Protected component             : "
    f"SPP-GAN discriminator"
)

print(
    f"Per-example gradients           : PASS"
)

print(
    f"Flat gradient clipping          : PASS"
)

print(
    f"Gaussian noise                  : PASS"
)

print(
    f"Poisson sampling                : PASS"
)

print(
    f"RDP accountant                  : CONFIGURED"
)

print(
    f"Target epsilon                  : "
    f"{TARGET_EPSILON}"
)

print(
    f"Maximum gradient norm           : "
    f"{MAX_GRAD_NORM}"
)

print(
    f"DP batch size                   : "
    f"{DP_BATCH_SIZE}"
)

print(
    f"DP epochs                       : "
    f"{DP_EPOCHS}"
)

print(
    f"Training                        : "
    f"NOT PERFORMED"
)

print(
    f"Privacy accounting              : "
    f"DEFERRED TO NOTEBOOK 11"
)

print(
    f"Achieved epsilon                : "
    f"NOT CLAIMED"
)

print(
    f"Synthetic generation            : "
    f"NOT PERFORMED"
)

print(
    f"End-to-end privacy claim        : "
    f"NOT ESTABLISHED"
)

print(
    f"Overall status                  : "
    f"PASS"
)

print()
print("Artifacts:")

print(
    f"  DP module       : "
    f"{DP_MODULE_PATH}"
)

print(
    f"  Configuration   : "
    f"{PRIVACY_CONFIGURATION_PATH}"
)

print(
    f"  Metadata        : "
    f"{PRIVACY_METADATA_PATH}"
)

print(
    f"  Manifest        : "
    f"{PRIVACY_ARTIFACT_REGISTRY_PATH}"
)

print(
    f"  Verification    : "
    f"{FINAL_VERIFICATION_PATH}"
)

print(
    f"  Completion      : "
    f"{COMPLETION_PATH}"
)

print()
print("Next:")

print(
    "  Notebook 11 — SPP-GAN Privacy Accounting"
)

print("=" * 100)

print(
    "\n✓ NOTEBOOK 10 COMPLETED SUCCESSFULLY."
)


19. COMPLETION SUMMARY

NOTEBOOK 10 — FINAL STATUS
Framework                       : SPP-GAN
Datasets                        : 3
Privacy mechanism               : DP-SGD
Protected component             : SPP-GAN discriminator
Per-example gradients           : PASS
Flat gradient clipping          : PASS
Gaussian noise                  : PASS
Poisson sampling                : PASS
RDP accountant                  : CONFIGURED
Target epsilon                  : 5.0
Maximum gradient norm           : 1.0
DP batch size                   : 128
DP epochs                       : 300
Training                        : NOT PERFORMED
Privacy accounting              : DEFERRED TO NOTEBOOK 11
Achieved epsilon                : NOT CLAIMED
Synthetic generation            : NOT PERFORMED
End-to-end privacy claim        : NOT ESTABLISHED
Overall status                  : PASS

Artifacts:
  DP module       : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10/models/sppgan_differential_pr